In [ ]:
import ocdkit.measure
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# ------------------------------------------------------------
# Global config
# ------------------------------------------------------------

dtype = torch.float32
device = torch.device("cpu")

# ------------------------------------------------------------
# sRGB <-> linear utilities
# ------------------------------------------------------------

def srgb_to_linear(srgb: torch.Tensor) -> torch.Tensor:
    """sRGB in [0,1] -> linear sRGB."""
    srgb = srgb.clamp(0.0, 1.0)
    threshold = 0.04045
    low = srgb <= threshold
    high = ~low
    out = torch.zeros_like(srgb, dtype=dtype)
    out[low] = srgb[low] / 12.92
    out[high] = ((srgb[high] + 0.055) / 1.055) ** 2.4
    return out

def linear_to_srgb(linear: torch.Tensor) -> torch.Tensor:
    """linear sRGB -> sRGB (no clipping)."""
    threshold = 0.0031308
    low = linear <= threshold
    high = ~low
    out = torch.zeros_like(linear, dtype=dtype)
    out[low] = 12.92 * linear[low]
    out[high] = 1.055 * (linear[high].clamp(min=0.0)) ** (1.0 / 2.4) - 0.055
    return out

# ------------------------------------------------------------
# OKLab conversion matrices (Björn Ottosson)
# ------------------------------------------------------------

M1 = torch.tensor([
    [0.4122214708, 0.5363325363, 0.0514459929],
    [0.2119034982, 0.6806995451, 0.1073969566],
    [0.0883024619, 0.2817188376, 0.6299787005],
], dtype=dtype, device=device)

M2 = torch.tensor([
    [ 0.2104542553,  0.7936177850, -0.0040720468],
    [ 1.9779984951, -2.4285922050,  0.4505937099],
    [ 0.0259040371,  0.7827717662, -0.8086757660],
], dtype=dtype, device=device)

M1_inv = torch.tensor([
    [ 4.0767416621, -3.3077115913,  0.2309699292],
    [-1.2684380046,  2.6097574011, -0.3413193965],
    [-0.0041960863, -0.7034186147,  1.7076147010],
], dtype=dtype, device=device)

M2_inv = torch.tensor([
    [1.0000000000,  0.3963377774,  0.2158037573],
    [1.0000000000, -0.1055613458, -0.0638541728],
    [1.0000000000, -0.0894841775, -1.2914855480],
], dtype=dtype, device=device)

def linear_to_oklab(rgb_lin: torch.Tensor) -> torch.Tensor:
    """linear sRGB -> OKLab."""
    shape = rgb_lin.shape
    x = rgb_lin.reshape(-1, 3)
    lms = x @ M1.T
    lms = torch.clamp(lms, min=0.0)
    lms_cbrt = torch.sign(lms) * torch.abs(lms) ** (1.0 / 3.0)
    lab = lms_cbrt @ M2.T
    return lab.reshape(shape)

def oklab_to_linear(lab: torch.Tensor) -> torch.Tensor:
    """OKLab -> linear sRGB."""
    shape = lab.shape
    x = lab.reshape(-1, 3)
    lms_cbrt = x @ M2_inv.T
    lms = lms_cbrt ** 3
    rgb_lin = lms @ M1_inv.T
    return rgb_lin.reshape(shape)

# ------------------------------------------------------------
# Parametric cyclic path in OKLab with hue warp
# ------------------------------------------------------------

class CyclicOKLabPath(torch.nn.Module):
    """
    L(θ) and C(θ) are low-order Fourier series.
    Hue in the ab-plane is θ + φ(θ), where φ(θ) is another Fourier series.
    """
    def __init__(self, n_harmonics: int = 2):
        super().__init__()
        self.n_harm = n_harmonics

        # Base levels
        self.L0 = torch.nn.Parameter(torch.tensor(0.65, dtype=dtype))
        self.C0 = torch.nn.Parameter(torch.tensor(0.10, dtype=dtype))

        # Fourier coefficients for L and C
        self.L_cos = torch.nn.Parameter(torch.zeros(n_harmonics, dtype=dtype))
        self.L_sin = torch.nn.Parameter(torch.zeros(n_harmonics, dtype=dtype))

        self.C_cos = torch.nn.Parameter(torch.zeros(n_harmonics, dtype=dtype))
        self.C_sin = torch.nn.Parameter(torch.zeros(n_harmonics, dtype=dtype))

        # Fourier coefficients for hue warp φ(θ)
        self.phi_cos = torch.nn.Parameter(torch.zeros(n_harmonics, dtype=dtype))
        self.phi_sin = torch.nn.Parameter(torch.zeros(n_harmonics, dtype=dtype))

    def forward(self, theta: torch.Tensor):
        """
        theta: [N] angles in [0, 2π)
        returns OKLab, L(θ), C(θ), φ(θ)
        """
        L = self.L0.clone()
        C = self.C0.clone()
        phi = torch.zeros_like(theta)

        for k in range(1, self.n_harm + 1):
            ck = float(k)
            cosk = torch.cos(ck * theta)
            sink = torch.sin(ck * theta)
            L = L + self.L_cos[k-1] * cosk + self.L_sin[k-1] * sink
            C = C + self.C_cos[k-1] * cosk + self.C_sin[k-1] * sink
            phi = phi + self.phi_cos[k-1] * cosk + self.phi_sin[k-1] * sink

        L = torch.clamp(L, 0.0, 1.0)
        C = torch.clamp(C, 0.0, 1.0)
        phi = torch.clamp(phi, -0.7, 0.7)  # ≈ ±40°

        angle = theta + phi
        a = C * torch.cos(angle)
        b = C * torch.sin(angle)
        lab = torch.stack([L, a, b], dim=-1)
        return lab, L, C, phi

# ------------------------------------------------------------
# Loss components
# ------------------------------------------------------------

# def compute_losses(path: CyclicOKLabPath, n_samples: int = 128):
#     """Compute all loss terms on a ring of n_samples points."""
#     if n_samples % 2 == 1:
#         n_samples += 1
#     theta = torch.linspace(0.0, 2.0 * math.pi, n_samples + 1, dtype=dtype, device=device)[:-1]
#     lab_raw, L, C, phi = path(theta)

#     # Gamut penalty in linear sRGB
#     rgb_lin = oklab_to_linear(lab_raw)
#     below = torch.clamp(-rgb_lin, min=0.0)
#     above = torch.clamp(rgb_lin - 1.0, min=0.0)
#     L_gamut = torch.mean(below ** 2 + above ** 2)

#     # Clip to displayable sRGB and re-evaluate OKLab
#     rgb = linear_to_srgb(rgb_lin)
#     rgb_clipped = rgb.clamp(0.0, 1.0)
#     rgb_lin_clipped = srgb_to_linear(rgb_clipped)
#     lab_clip = linear_to_oklab(rgb_lin_clipped)

#     # Uniformity of step size along ring
#     lab_roll = torch.roll(lab_clip, shifts=-1, dims=0)
#     deltas = torch.linalg.norm(lab_roll - lab_clip, dim=-1)
#     mean_delta = torch.mean(deltas)
#     L_uniform = torch.var(deltas / (mean_delta + 1e-8))

#     # Saturation after clipping
#     C_mag = torch.linalg.norm(lab_clip[:, 1:], dim=-1)
#     L_saturation = -torch.mean(C_mag)

#     # Complement balance
#     half = n_samples // 2
#     lab_shift = torch.roll(lab_clip, shifts=half, dims=0)
#     L_comp_L = torch.mean((lab_clip[:, 0] + lab_shift[:, 0] - 1.2) ** 2)
#     C_mag_shift = torch.linalg.norm(lab_shift[:, 1:], dim=-1)
#     L_comp_C = torch.mean((C_mag - C_mag_shift) ** 2)

#     # Smoothness for L, C, φ (second differences)
#     def second_diff(x):
#         return torch.roll(x, -1, 0) - 2 * x + torch.roll(x, 1, 0)

#     L_smooth = torch.mean(
#         second_diff(L) ** 2
#         + second_diff(C) ** 2
#         + 0.5 * second_diff(phi) ** 2
#     )

#     # Penalize large warp magnitude
#     L_phi_mag = torch.mean(phi ** 2)

#     losses = {
#         "L_gamut": L_gamut,
#         "L_uniform": L_uniform,
#         "L_saturation": L_saturation,
#         "L_comp_L": L_comp_L,
#         "L_comp_C": L_comp_C,
#         "L_smooth": L_smooth,
#         "L_phi_mag": L_phi_mag,
#         "rgb": rgb_clipped,
#         "lab": lab_clip,
#         "theta": theta,
#         "L_vals": L,
#         "C_vals": C,
#         "phi_vals": phi,
#     }
#     return losses
def compute_losses(path: CyclicOKLabPath, n_samples: int = 128):
    if n_samples % 2 == 1:
        n_samples += 1
    theta = torch.linspace(0.0, 2.0 * math.pi, n_samples + 1, dtype=dtype, device=device)[:-1]
    lab_raw, L, C, phi = path(theta)

    rgb_lin = oklab_to_linear(lab_raw)
    below = torch.clamp(-rgb_lin, min=0.0)
    above = torch.clamp(rgb_lin - 1.0, min=0.0)
    L_gamut = torch.mean(below ** 2 + above ** 2)

    rgb = linear_to_srgb(rgb_lin)
    rgb_clipped = rgb.clamp(0.0, 1.0)
    rgb_lin_clipped = srgb_to_linear(rgb_clipped)
    lab_clip = linear_to_oklab(rgb_lin_clipped)

    lab_roll = torch.roll(lab_clip, shifts=-1, dims=0)
    deltas = torch.linalg.norm(lab_roll - lab_clip, dim=-1)
    mean_delta = torch.mean(deltas)
    L_uniform = torch.var(deltas / (mean_delta + 1e-8))

    C_mag = torch.linalg.norm(lab_clip[:, 1:], dim=-1)
    L_saturation = -torch.mean(C_mag)

    half = n_samples // 2
    lab_shift = torch.roll(lab_clip, shifts=half, dims=0)
    L_comp_L = torch.mean((lab_clip[:, 0] + lab_shift[:, 0] - 1.2) ** 2)
    C_mag_shift = torch.linalg.norm(lab_shift[:, 1:], dim=-1)
    L_comp_C = torch.mean((C_mag - C_mag_shift) ** 2)

    def second_diff(x):
        return torch.roll(x, -1, 0) - 2 * x + torch.roll(x, 1, 0)

    L_smooth = torch.mean(
        second_diff(L) ** 2
        + second_diff(C) ** 2
        + 0.5 * second_diff(phi) ** 2
    )

    L_phi_mag = torch.mean(phi ** 2)

    losses = {
        "L_gamut": L_gamut,
        "L_uniform": L_uniform,
        "L_saturation": L_saturation,
        "L_comp_L": L_comp_L,
        "L_comp_C": L_comp_C,
        "L_smooth": L_smooth,
        "L_phi_mag": L_phi_mag,
        "rgb": rgb_clipped,
        "lab": lab_clip,
        "theta": theta,
        "L_vals": L,
        "C_vals": C,
        "phi_vals": phi,
    }
    return losses

def total_loss(losses,
               w_gamut=8.0,
               w_uniform=1.0,
               w_saturation=0.5,
               w_comp=0.3,
               w_smooth=0.05,
               w_phi_mag=0.02):
    return (
        w_gamut * losses["L_gamut"]
        + w_uniform * losses["L_uniform"]
        + w_saturation * losses["L_saturation"]
        + w_comp * (losses["L_comp_L"] + 0.5 * losses["L_comp_C"])
        + w_smooth * losses["L_smooth"]
        + w_phi_mag * losses["L_phi_mag"]
    )
# ------------------------------------------------------------
# Optimization driver
# ------------------------------------------------------------

def optimize_cyclic_path(
    steps: int = 600,
    n_harmonics: int = 2,
    lr: float = 0.02,
    n_samples_opt: int = 128,
):
    """
    Optimize Fourier coefficients in CyclicOKLabPath.

    steps:         maximum number of iterations
    n_harmonics:   number of Fourier harmonics in L, C, φ
    lr:            learning rate
    n_samples_opt: number of samples on ring during optimization
    """
    path = CyclicOKLabPath(n_harmonics=n_harmonics).to(device)
    opt = torch.optim.Adam(path.parameters(), lr=lr)

    last_loss = None
    for step in range(steps):
        opt.zero_grad()
        losses = compute_losses(path, n_samples=n_samples_opt)
        loss = total_loss(losses)
        loss.backward()
        opt.step()

        if (step + 1) % 200 == 0:
            print(
                f"step {step+1:4d}: "
                f"loss={loss.item():.4f} "
                f"gamut={losses['L_gamut'].item():.4e} "
                f"uniform={losses['L_uniform'].item():.4e} "
                f"sat={-losses['L_saturation'].item():.4f}"
            )

        # Very simple early stop
        if last_loss is not None and abs(loss.item() - last_loss) < 1e-5:
            print(f"Early stop at step {step+1}, loss change < 1e-5")
            break
        last_loss = loss.item()

    # Recompute on a dense ring for visualization
    with torch.no_grad():
        final_losses = compute_losses(path, n_samples=512)
    return path, final_losses

# ------------------------------------------------------------
# Visualization: OKLab path vs sRGB gamut, and hue disk
# ------------------------------------------------------------

def sample_srgb_gamut_grid(n: int = 17):
    """Sample a coarse sRGB cube and map to OKLab for plotting."""
    vals = torch.linspace(0.0, 1.0, n, dtype=dtype, device=device)
    r, g, b = torch.meshgrid(vals, vals, vals, indexing="ij")
    rgb = torch.stack([r, g, b], dim=-1).reshape(-1, 3)
    rgb_lin = srgb_to_linear(rgb)
    lab = linear_to_oklab(rgb_lin)
    return rgb.cpu().numpy(), lab.cpu().numpy()

def plot_oklab_path_and_gamut(lab_path: torch.Tensor, rgb_path: torch.Tensor):
    """Plot optimized path against sRGB gamut in OKLab."""
    lab_np = lab_path.cpu().numpy()
    rgb_np = rgb_path.cpu().numpy()
    rgb_gamut, lab_gamut = sample_srgb_gamut_grid(n=17)

    fig = plt.figure(figsize=(12, 5))

    # 3D OKLab
    ax3d = fig.add_subplot(1, 2, 1, projection="3d")
    ax3d.scatter(
        lab_gamut[:, 0], lab_gamut[:, 1], lab_gamut[:, 2],
        s=1, alpha=0.1
    )
    ax3d.plot(
        lab_np[:, 0], lab_np[:, 1], lab_np[:, 2],
        linewidth=3
    )
    ax3d.set_xlabel("L")
    ax3d.set_ylabel("a")
    ax3d.set_zlabel("b")
    ax3d.set_title("OKLab sRGB gamut + optimized cyclic path")

    # ab-plane slice near mean L
    ax2 = fig.add_subplot(1, 2, 2)
    L_mean = lab_np[:, 0].mean()
    mask = np.abs(lab_gamut[:, 0] - L_mean) < 0.05
    ax2.scatter(
        lab_gamut[mask, 1], lab_gamut[mask, 2],
        s=3, alpha=0.3
    )
    ax2.scatter(
        lab_np[:, 1], lab_np[:, 2],
        c=rgb_np, s=20, edgecolors="none"
    )
    ax2.set_xlabel("a")
    ax2.set_ylabel("b")
    ax2.set_aspect("equal", "box")
    ax2.set_title(f"ab-plane near L={L_mean:.2f}")

    plt.tight_layout()

def make_hue_disk(rgb_ring: torch.Tensor, img_res: int = 512):
    """Build a hue disk image from a ring of sRGB colors."""
    rgb_np = rgb_ring.cpu().numpy()
    N = rgb_np.shape[0]

    y, x = np.ogrid[-1:1:complex(0, img_res), -1:1:complex(0, img_res)]
    r = np.sqrt(x * x + y * y)
    theta = np.arctan2(y, x)
    theta = (theta + 2 * np.pi) % (2 * np.pi)

    img = np.ones((img_res, img_res, 3), dtype=np.float64)
    inside = r <= 1.0
    idx = (theta[inside] / (2 * np.pi) * N).astype(int)
    idx = np.clip(idx, 0, N - 1)
    img[inside] = rgb_np[idx]
    return img

def plot_results(path, losses):
    """Plot hue disk and parameter curves."""
    lab = losses["lab"]
    rgb = losses["rgb"]
    theta = losses["theta"]

    plot_oklab_path_and_gamut(lab, rgb)

    disk = make_hue_disk(rgb, img_res=512)

    fig, ax = plt.subplots(1, 3, figsize=(14, 4))
    ax[0].imshow(disk, origin="lower")
    ax[0].axis("off")
    ax[0].set_title("Optimized cyclic colormap (hue disk)")

    L_vals = losses["L_vals"].cpu().numpy()
    C_vals = losses["C_vals"].cpu().numpy()
    phi_vals = losses["phi_vals"].cpu().numpy()
    th = theta.cpu().numpy()

    ax[1].plot(th, L_vals, label="L(θ)")
    ax[1].plot(th, C_vals, label="C(θ)")
    ax[1].set_xlabel("θ (radians)")
    ax[1].legend()
    ax[1].set_title("L and C vs θ")

    ax[2].plot(th, phi_vals)
    ax[2].set_xlabel("θ (radians)")
    ax[2].set_title("Hue warp φ(θ)")
    plt.tight_layout()

def build_listed_cmap(rgb_ring: torch.Tensor, name: str = "cyclic_oklab_optimized"):
    """Return a Matplotlib ListedColormap from the optimized ring."""
    rgb_np = rgb_ring.clamp(0.0, 1.0).cpu().numpy()
    return ListedColormap(rgb_np, name=name)

# ------------------------------------------------------------
# Main
# ------------------------------------------------------------
from matplotlib.colors import hsv_to_rgb
import omnirefactor
from omnirefactor.plot import imshow
from omnirefactor import io
import fastremap
from pathlib import Path
import os
import torch

# ------------------------------------------------------------
# Helper: resample a ring (sRGB) to n_hues by interpolation
# ------------------------------------------------------------

def resample_ring_srgb(rgb_srgb_full: np.ndarray, n_hues: int) -> np.ndarray:
    """
    rgb_srgb_full: [N,3] sRGB ring samples (e.g. losses['rgb'])
    returns [n_hues,3] sRGB ring by arcwise interpolation.
    """
    rgb = np.asarray(rgb_srgb_full, dtype=float)
    N = rgb.shape[0]
    t_full = np.linspace(0.0, 1.0, N, endpoint=False)
    t_target = np.linspace(0.0, 1.0, n_hues, endpoint=False)

    idx0 = np.floor(t_target * N).astype(int) % N
    idx1 = (idx0 + 1) % N
    frac = (t_target * N) - np.floor(t_target * N)

    ring = (1 - frac[:, None]) * rgb[idx0] + frac[:, None] * rgb[idx1]
    return ring


# ------------------------------------------------------------
# Helper: map a linear RGB ring to hue disk and flow images
# ------------------------------------------------------------

def make_flow_images_for_ring(rgb_ring_lin: np.ndarray,
                              center_lin: np.ndarray,
                              flows: torch.Tensor):
    """
    rgb_ring_lin: [H,3] linear sRGB ring (n_hues = H)
    center_lin:   [3] linear sRGB center color
    flows:        [2,H,W] Omnipose flow field tensor (u,v)
    returns:
        disk_rgb (Hc,Wc,3)
        flow_rgb (H,W,3)
        flow_white (H,W,3)
        flow_black (H,W,3)
    """
    n_hues = rgb_ring_lin.shape[0]

    # 1) hue disk
    disk = build_hue_disk_from_ring(rgb_ring_lin, center_lin, size=512)

    # 2) flow -> hue/radius mapping
    u = flows[0].cpu().numpy()
    v = flows[1].cpu().numpy()
    Hf, Wf = u.shape

    angle = np.mod(np.arctan2(v, u), 2*np.pi)
    mag = np.sqrt(u*u + v*v)
    mag /= (mag.max() + 1e-8)

    hue_f = angle/(2*np.pi)*n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)

    ring0 = rgb_ring_lin[idx0]
    ring1 = rgb_ring_lin[idx1]
    ring_interp = (1 - t[..., None]) * ring0 + t[..., None] * ring1

    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow), 0, 1)

    alpha = mag[..., None]
    white = np.ones_like(flow_rgb)
    black = np.zeros_like(flow_rgb)
    flow_white = alpha * flow_rgb + (1 - alpha) * white
    flow_black = alpha * flow_rgb + (1 - alpha) * black

    return disk, flow_rgb, flow_white, flow_black


# ------------------------------------------------------------
# Main visualization: optimized OKLab vs sinebow vs HSV on Omnipose flows
# ------------------------------------------------------------

def visualize_oklab_vs_sinebow_hsv_on_omnipose(
    path,
    losses,
    n_hues: int = 72,
    grid_n: int = 32,
    min_chroma: float = 0.03,
    w_J: float = 0.6,
    w_C: float = 0.4,
    lambda_smooth: float = 0.25,
    size: int = 512,
    device_str: str = "cpu",
):
    """
    Compare three cyclic colormaps on the same Omnipose E. coli flow field:

      Row 1: OKLab-optimized colormap from `path`/`losses`
      Row 2: sinebow
      Row 3: HSV

      Columns: [hue disk, flow chroma, flow alpha on white, flow alpha on black]
    """

    dev = torch.device(device_str)

    # --------------------------------------------------------
    # 1) Build three linear-RGB rings: OKLab-optimized, sinebow, HSV
    # --------------------------------------------------------

    # Optimized OKLab ring (sRGB) from losses["rgb"] (torch)
    rgb_opt_full = losses["rgb"].detach().cpu().numpy()  # [N,3] sRGB
    rgb_opt_srgb = resample_ring_srgb(rgb_opt_full, n_hues)  # [n_hues,3]
    rgb_opt_lin = srgb_to_linear_np(rgb_opt_srgb)

    # sinebow ring in sRGB
    t = np.linspace(0, 1, n_hues, endpoint=False)
    sine_r = 0.5 + 0.5 * np.sin(2*np.pi*(t + 0/3))
    sine_g = 0.5 + 0.5 * np.sin(2*np.pi*(t + 1/3))
    sine_b = 0.5 + 0.5 * np.sin(2*np.pi*(t + 2/3))
    rgb_sine_srgb = np.stack([sine_r, sine_g, sine_b], axis=1)
    rgb_sine_lin = srgb_to_linear_np(rgb_sine_srgb)

    # HSV ring in sRGB
    hsv_vals = np.stack([t, np.ones_like(t), np.ones_like(t)], axis=1)
    rgb_hsv_srgb = hsv_to_rgb(hsv_vals)
    rgb_hsv_lin = srgb_to_linear_np(rgb_hsv_srgb)

    # common center (neutral gray)
    center_srgb = np.array([0.5, 0.5, 0.5], dtype=float)
    center_lin = srgb_to_linear_np(center_srgb)

    # --------------------------------------------------------
    # 2) Load Omnipose E. coli flows once
    # --------------------------------------------------------

    omnidir = Path(omnirefactor.__file__).parent.parent.parent
    basedir = os.path.join(omnidir, "docs", "_static")
    name = "ecoli"
    ext = ".tif"

    image = io.imread(os.path.join(basedir, name + ext))
    masks = io.imread(os.path.join(basedir, name + "_labels" + ext))
    slc = ocdkit.measure.crop_bbox(masks > 0, pad=0)[0]
    masks = fastremap.renumber(masks[slc])[0]
    image = image[slc]

    flows_all = omnirefactor.core.masks_to_flows(masks, device="cpu")[4].to(dev)  # [2,H,W]
    flows = flows_all  # [2,H,W]

    # --------------------------------------------------------
    # 3) Build images for each ring
    # --------------------------------------------------------

    disk_opt, flow_opt, flow_opt_white, flow_opt_black = make_flow_images_for_ring(
        rgb_opt_lin, center_lin, flows
    )
    disk_sine, flow_sine, flow_sine_white, flow_sine_black = make_flow_images_for_ring(
        rgb_sine_lin, center_lin, flows
    )
    disk_hsv, flow_hsv, flow_hsv_white, flow_hsv_black = make_flow_images_for_ring(
        rgb_hsv_lin, center_lin, flows
    )

    # --------------------------------------------------------
    # 4) Plot 3×4 grid
    # --------------------------------------------------------

    fig, axes = plt.subplots(3, 4, figsize=(16, 10))
    row_titles = ["OKLab-optimized", "sinebow", "HSV"]

    # Row 0: OKLab-optimized
    axes[0,0].imshow(disk_opt);           axes[0,0].set_title("Hue disk");                  axes[0,0].axis("off")
    axes[0,1].imshow(flow_opt);           axes[0,1].set_title("Flow mag → chroma");         axes[0,1].axis("off")
    axes[0,2].imshow(flow_opt_white);     axes[0,2].set_title("Flow mag → alpha (white)");  axes[0,2].axis("off")
    axes[0,3].imshow(flow_opt_black);     axes[0,3].set_title("Flow mag → alpha (black)");  axes[0,3].axis("off")

    # Row 1: sinebow
    axes[1,0].imshow(disk_sine);          axes[1,0].axis("off")
    axes[1,1].imshow(flow_sine);          axes[1,1].axis("off")
    axes[1,2].imshow(flow_sine_white);    axes[1,2].axis("off")
    axes[1,3].imshow(flow_sine_black);    axes[1,3].axis("off")

    # Row 2: HSV
    axes[2,0].imshow(disk_hsv);           axes[2,0].axis("off")
    axes[2,1].imshow(flow_hsv);           axes[2,1].axis("off")
    axes[2,2].imshow(flow_hsv_white);     axes[2,2].axis("off")
    axes[2,3].imshow(flow_hsv_black);     axes[2,3].axis("off")

    for i, title in enumerate(row_titles):
        axes[i,0].set_ylabel(title, fontsize=12)

    plt.tight_layout()
    plt.show()
    
path, losses = optimize_cyclic_path(
    steps=600,
    n_harmonics=2,
    lr=0.02,
    n_samples_opt=512,
)
print("Final scalar losses:")
for k, v in losses.items():
    if isinstance(v, torch.Tensor) and v.dim() == 0:
        print(f"  {k}: {v.item():.6f}")

plot_results(path, losses)
plt.show()
visualize_oklab_vs_sinebow_hsv_on_omnipose(
    path,
    losses,
    n_hues=72,
    device_str="cpu",
)
    

In [ ]:
import numpy as np
import torch
import itertools
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML
from matplotlib.colors import hsv_to_rgb

# ============================================================
# Extract optimized curve from `losses`
# ============================================================

lab_opt = losses["lab"].detach().cpu().numpy()   # [N,3]
rgb_opt = losses["rgb"].detach().cpu().numpy()   # [N,3]

# (these come from `compute_losses` inside optimize_cyclic_path)


# ============================================================
# helper: warped OKLab surfaces for each sRGB cube face
# ============================================================

def make_face_oklab(face: str, n: int = 96):
    t = np.linspace(0.0, 1.0, n, dtype=np.float32)
    u, v = np.meshgrid(t, t, indexing="ij")
    srgb = np.zeros((n, n, 3), dtype=np.float32)

    if   face == "R0": srgb[...,0]=0.0; srgb[...,1]=u; srgb[...,2]=v
    elif face == "R1": srgb[...,0]=1.0; srgb[...,1]=u; srgb[...,2]=v
    elif face == "G0": srgb[...,1]=0.0; srgb[...,0]=u; srgb[...,2]=v
    elif face == "G1": srgb[...,1]=1.0; srgb[...,0]=u; srgb[...,2]=v
    elif face == "B0": srgb[...,2]=0.0; srgb[...,0]=u; srgb[...,1]=v
    elif face == "B1": srgb[...,2]=1.0; srgb[...,0]=u; srgb[...,1]=v
    else:
        raise ValueError("bad face name")

    srgb_t = torch.from_numpy(srgb.reshape(-1,3)).to(device)
    rgb_lin = srgb_to_linear(srgb_t)
    lab = linear_to_oklab(rgb_lin).cpu().numpy().reshape(n,n,3)

    return lab[...,0], lab[...,1], lab[...,2]   # L, a, b grids


faces = ["R0","R1","G0","G1","B0","B1"]
face_surfaces = {f: make_face_oklab(f, n=96) for f in faces}


# ============================================================
# Recompute sinebow + HSV for completeness
# ============================================================

def sinebow(n=512):
    t = np.linspace(0, 1, n, endpoint=False)
    r = 0.5 + 0.5*np.sin(2*np.pi*(t + 0/3))
    g = 0.5 + 0.5*np.sin(2*np.pi*(t + 1/3))
    b = 0.5 + 0.5*np.sin(2*np.pi*(t + 2/3))
    return np.stack([r,g,b],axis=1)

n_ring = 512

sine_rgb = sinebow(n_ring).astype(np.float32)
sine_lab = linear_to_oklab(srgb_to_linear(torch.from_numpy(sine_rgb).to(device))).cpu().numpy()

h = np.linspace(0,1,n_ring,endpoint=False)
hsv = np.stack([h, np.ones_like(h), np.ones_like(h)], axis=1)
hsv_rgb = hsv_to_rgb(hsv).astype(np.float32)
hsv_lab = linear_to_oklab(srgb_to_linear(torch.from_numpy(hsv_rgb).to(device))).cpu().numpy()


# ============================================================
# Build the figure
# ============================================================

fig = go.Figure()


# ------------------------------------------------------------
# Draw warped cube faces (flat gray, no shading)
# ------------------------------------------------------------

for f in faces:
    Lg, ag, bg = face_surfaces[f]
    fig.add_trace(go.Surface(
        x=Lg, y=ag, z=bg,
        surfacecolor=np.zeros_like(Lg),    # constant color
        colorscale=[[0,"rgb(120,120,120)"],[1,"rgb(120,120,120)"]],
        showscale=False,
        opacity=0.15,
        hoverinfo="skip",
        lighting=dict(ambient=1.0, diffuse=0.0, specular=0.0),
        name=f,
        showlegend=False,
    ))


# ------------------------------------------------------------
# Add sRGB cube edges in neutral mid-gray
# ------------------------------------------------------------

verts = np.array(list(itertools.product([0.0,1.0],[0.0,1.0],[0.0,1.0])), dtype=np.float32)
edges = [(i,j) for i in range(8) for j in range(i+1,8)
         if np.sum(np.abs(verts[i]-verts[j])) == 1.0]

t_edge = np.linspace(0.0,1.0,64,dtype=np.float32)
for i,j in edges:
    p0 = verts[i]; p1 = verts[j]
    srgb_line = p0 + (p1-p0)[None,:] * t_edge[:,None]
    lab_edge = linear_to_oklab(srgb_to_linear(torch.from_numpy(srgb_line).float().to(device))).cpu().numpy()

    fig.add_trace(go.Scatter3d(
        x=lab_edge[:,0], y=lab_edge[:,1], z=lab_edge[:,2],
        mode="lines",
        line=dict(width=3, color="rgb(128,128,128)"),
        hoverinfo="skip",
        showlegend=False
    ))


# ------------------------------------------------------------
# Plot optimized cyclic path (the one from your optimization)
# ------------------------------------------------------------

fig.add_trace(go.Scatter3d(
    x=lab_opt[:,0], y=lab_opt[:,1], z=lab_opt[:,2],
    mode="lines+markers",
    line=dict(width=6, color="#ffffff"),
    marker=dict(size=4, color=rgb_opt),
    name="optimized",
))


# ------------------------------------------------------------
# Plot sinebow + HSV
# ------------------------------------------------------------

fig.add_trace(go.Scatter3d(
    x=sine_lab[:,0], y=sine_lab[:,1], z=sine_lab[:,2],
    mode="lines+markers",
    line=dict(width=4, color="#aaaaaa"),
    marker=dict(size=4, color=sine_rgb),
    name="sinebow",
))

fig.add_trace(go.Scatter3d(
    x=hsv_lab[:,0], y=hsv_lab[:,1], z=hsv_lab[:,2],
    mode="lines+markers",
    line=dict(width=4, color="#cccccc"),
    marker=dict(size=4, color=hsv_rgb),
    name="HSV",
))


# ------------------------------------------------------------
# Aesthetics: gray text everywhere
# ------------------------------------------------------------

gray = "rgb(200,200,200)"

axis_style = dict(
    showbackground=False,
    gridcolor="rgba(200,200,200,0.25)",
    zerolinecolor="rgba(200,200,200,0.35)",
    tickfont=dict(color=gray),
)

fig.update_layout(
    width=1000, height=780,
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    scene=dict(
        bgcolor="rgba(0,0,0,0)",
        xaxis=dict(axis_style, title=dict(text="L", font=dict(color=gray))),
        yaxis=dict(axis_style, title=dict(text="a", font=dict(color=gray))),
        zaxis=dict(axis_style, title=dict(text="b", font=dict(color=gray))),
        aspectmode="data",
    ),
    font=dict(color=gray),
    legend=dict(
        font=dict(color=gray, size=14),
        bgcolor="rgba(0,0,0,0)"
    ),
    title=dict(
        text="OKLab: warped sRGB cube (neutral) + edges + optimized/sinebow/HSV",
        font=dict(color=gray, size=20),
    )
)

HTML(pio.to_html(fig, include_plotlyjs="cdn", full_html=False))

In [ ]:
import math
import numpy as np
import torch
import itertools
import matplotlib.pyplot as plt
from matplotlib.colors import hsv_to_rgb

import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display

import omnirefactor
from omnirefactor.plot import imshow
from omnirefactor import io
import fastremap
from pathlib import Path
import os

# ============================================================
# Global config
# ============================================================

dtype = torch.float32
device = torch.device("cpu")

# ============================================================
# Torch: sRGB <-> linear, OKLab
# ============================================================

def srgb_to_linear(srgb: torch.Tensor) -> torch.Tensor:
    srgb = srgb.to(dtype=dtype).clamp(0.0, 1.0)
    threshold = 0.04045
    low = srgb <= threshold
    high = ~low
    out = torch.zeros_like(srgb)
    out[low] = srgb[low] / 12.92
    out[high] = ((srgb[high] + 0.055) / 1.055) ** 2.4
    return out

def linear_to_srgb(linear: torch.Tensor) -> torch.Tensor:
    linear = linear.to(dtype=dtype)
    threshold = 0.0031308
    low = linear <= threshold
    high = ~low
    out = torch.zeros_like(linear)
    out[low] = 12.92 * linear[low]
    out[high] = 1.055 * (linear[high].clamp(min=0.0)) ** (1.0 / 2.4) - 0.055
    return out

M1 = torch.tensor([
    [0.4122214708, 0.5363325363, 0.0514459929],
    [0.2119034982, 0.6806995451, 0.1073969566],
    [0.0883024619, 0.2817188376, 0.6299787005],
], dtype=dtype, device=device)

M2 = torch.tensor([
    [ 0.2104542553,  0.7936177850, -0.0040720468],
    [ 1.9779984951, -2.4285922050,  0.4505937099],
    [ 0.0259040371,  0.7827717662, -0.8086757660],
], dtype=dtype, device=device)

M1_inv = torch.tensor([
    [ 4.0767416621, -3.3077115913,  0.2309699292],
    [-1.2684380046,  2.6097574011, -0.3413193965],
    [-0.0041960863, -0.7034186147,  1.7076147010],
], dtype=dtype, device=device)

M2_inv = torch.tensor([
    [1.0000000000,  0.3963377774,  0.2158037573],
    [1.0000000000, -0.1055613458, -0.0638541728],
    [1.0000000000, -0.0894841775, -1.2914855480],
], dtype=dtype, device=device)

def linear_to_oklab(rgb_lin: torch.Tensor) -> torch.Tensor:
    shape = rgb_lin.shape
    x = rgb_lin.reshape(-1, 3)
    lms = x @ M1.T
    lms = torch.clamp(lms, min=0.0)
    lms_cbrt = torch.sign(lms) * torch.abs(lms) ** (1.0 / 3.0)
    lab = lms_cbrt @ M2.T
    return lab.reshape(shape)

def oklab_to_linear(lab: torch.Tensor) -> torch.Tensor:
    shape = lab.shape
    x = lab.reshape(-1, 3)
    lms_cbrt = x @ M2_inv.T
    lms = lms_cbrt ** 3
    rgb_lin = lms @ M1_inv.T
    return rgb_lin.reshape(shape)

# ============================================================
# NumPy sRGB <-> linear (for disks / flows)
# ============================================================

def srgb_to_linear_np(rgb):
    rgb = np.asarray(rgb, dtype=float)
    a = 0.055
    return np.where(rgb <= 0.04045, rgb / 12.92,
                    ((rgb + a) / (1 + a))**2.4)

def linear_to_srgb_np(rgb):
    rgb = np.asarray(rgb, dtype=float)
    a = 0.055
    return np.where(
        rgb <= 0.0031308,
        12.92 * rgb,
        (1 + a) * np.power(np.clip(rgb, 0, None), 1 / 2.4) - a,
    )

# ============================================================
# 1) Largest OKLab circle in plane spanned by RGB primaries
# ============================================================

def build_oklab_plane_circle_ring(n_samples: int = 512,
                                  margin: float = 1e-4) -> np.ndarray:
    prim_srgb = torch.tensor([
        [1.0, 0.0, 0.0],
        [0.0, 1.0, 0.0],
        [0.0, 0.0, 1.0],
    ], dtype=dtype, device=device)

    prim_lin = srgb_to_linear(prim_srgb)
    prim_lab = linear_to_oklab(prim_lin)

    center = prim_lab.mean(dim=0)
    v1 = prim_lab[0] - center
    v2 = prim_lab[1] - center

    e1 = v1 / (v1.norm() + 1e-8)
    v2_orth = v2 - torch.dot(v2, e1) * e1
    e2 = v2_orth / (v2_orth.norm() + 1e-8)

    theta = torch.linspace(0.0, 2.0 * math.pi, n_samples, dtype=dtype, device=device)

    def is_safe(scale: float) -> bool:
        cos_t = torch.cos(theta)
        sin_t = torch.sin(theta)
        offset = scale * (cos_t[:, None] * e1[None, :] + sin_t[:, None] * e2[None, :])
        lab = center[None, :] + offset
        rgb_lin = oklab_to_linear(lab)
        return bool(((rgb_lin >= -margin) & (rgb_lin <= 1.0 + margin)).all())

    s_low, s_high = 0.0, 1.0
    for _ in range(40):
        mid = 0.5 * (s_low + s_high)
        if is_safe(mid):
            s_low = mid
        else:
            s_high = mid
    s_opt = s_low

    cos_t = torch.cos(theta)
    sin_t = torch.sin(theta)
    offset = s_opt * (cos_t[:, None] * e1[None, :] + sin_t[:, None] * e2[None, :])
    lab_circle = center[None, :] + offset
    rgb_lin_circle = oklab_to_linear(lab_circle)
    rgb_srgb_circle = linear_to_srgb(rgb_lin_circle).clamp(0.0, 1.0).cpu().numpy()
    return rgb_srgb_circle  # [n_samples,3]

# ============================================================
# 2) RGB-primary-plane boundary ring (R-G-B triangle)
# ============================================================

def build_primary_plane_boundary_ring(n_samples: int = 72) -> np.ndarray:
    """
    Cyclic ring along the triangle R->G->B->R in sRGB.
    Assumes n_samples is divisible by 3 (e.g., 72).
    """
    assert n_samples % 3 == 0, "n_samples must be divisible by 3"
    n_edge = n_samples // 3

    # R->G: (1,0,0) -> (0,1,0)
    t_rg = np.linspace(0, 1, n_edge, endpoint=False)
    rg = np.stack([1 - t_rg, t_rg, np.zeros_like(t_rg)], axis=1)

    # G->B: (0,1,0) -> (0,0,1)
    t_gb = np.linspace(0, 1, n_edge, endpoint=False)
    gb = np.stack([np.zeros_like(t_gb), 1 - t_gb, t_gb], axis=1)

    # B->R: (0,0,1) -> (1,0,0)
    t_br = np.linspace(0, 1, n_edge, endpoint=False)
    br = np.stack([t_br, np.zeros_like(t_br), 1 - t_br], axis=1)

    ring = np.concatenate([rg, gb, br], axis=0)  # [n_samples,3]
    return ring

# ============================================================
# 3) Build hue disks and flow images from a ring
# ============================================================

def build_hue_disk_from_ring(rgb_ring_lin: np.ndarray,
                             center_lin: np.ndarray,
                             size: int = 512) -> np.ndarray:
    n_hues = rgb_ring_lin.shape[0]
    N = size
    y, x = np.mgrid[0:N, 0:N]
    x = (x + 0.5) / N * 2 - 1
    y = (y + 0.5) / N * 2 - 1
    r = np.sqrt(x * x + y * y)
    theta = np.mod(np.arctan2(y, x), 2 * np.pi)

    hue_f = theta / (2 * np.pi) * n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)

    ring0 = rgb_ring_lin[idx0]
    ring1 = rgb_ring_lin[idx1]
    ring_interp = (1 - t[..., None]) * ring0 + t[..., None] * ring1

    r_exp = r[..., None]
    rgb_lin = (1 - r_exp) * center_lin + r_exp * ring_interp
    rgb_lin[r > 1] = 1.0

    return np.clip(linear_to_srgb_np(rgb_lin), 0, 1)


def make_flow_images_for_ring(rgb_ring_lin: np.ndarray,
                              center_lin: np.ndarray,
                              flows: torch.Tensor):
    n_hues = rgb_ring_lin.shape[0]

    disk = build_hue_disk_from_ring(rgb_ring_lin, center_lin, size=512)

    u = flows[0].cpu().numpy()
    v = flows[1].cpu().numpy()
    Hf, Wf = u.shape

    angle = np.mod(np.arctan2(v, u), 2 * np.pi)
    mag = np.sqrt(u * u + v * v)
    mag /= (mag.max() + 1e-8)

    hue_f = angle / (2 * np.pi) * n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)

    ring0 = rgb_ring_lin[idx0]
    ring1 = rgb_ring_lin[idx1]
    ring_interp = (1 - t[..., None]) * ring0 + t[..., None] * ring1

    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow), 0, 1)

    alpha = mag[..., None]
    white = np.ones_like(flow_rgb)
    black = np.zeros_like(flow_rgb)
    flow_white = alpha * flow_rgb + (1 - alpha) * white
    flow_black = alpha * flow_rgb + (1 - alpha) * black

    return disk, flow_rgb, flow_white, flow_black

# ============================================================
# 4) Warped OKLab cube faces for trajectory plot
# ============================================================

def make_face_oklab(face: str, n: int = 96):
    t = np.linspace(0.0, 1.0, n, dtype=np.float32)
    u, v = np.meshgrid(t, t, indexing="ij")
    srgb = np.zeros((n, n, 3), dtype=np.float32)

    if   face == "R0": srgb[...,0]=0.0; srgb[...,1]=u; srgb[...,2]=v
    elif face == "R1": srgb[...,0]=1.0; srgb[...,1]=u; srgb[...,2]=v
    elif face == "G0": srgb[...,1]=0.0; srgb[...,0]=u; srgb[...,2]=v
    elif face == "G1": srgb[...,1]=1.0; srgb[...,0]=u; srgb[...,2]=v
    elif face == "B0": srgb[...,2]=0.0; srgb[...,0]=u; srgb[...,1]=v
    elif face == "B1": srgb[...,2]=1.0; srgb[...,0]=u; srgb[...,1]=v
    else:
        raise ValueError("bad face name")

    srgb_t = torch.from_numpy(srgb.reshape(-1,3)).to(device=device, dtype=dtype)
    rgb_lin = srgb_to_linear(srgb_t)
    lab = linear_to_oklab(rgb_lin).cpu().numpy().reshape(n,n,3)
    return lab[...,0], lab[...,1], lab[...,2]

faces = ["R0","R1","G0","G1","B0","B1"]
face_surfaces = {f: make_face_oklab(f, n=96) for f in faces}

# ============================================================
# 5) Trajectory plot with 4 curves (ellipse, primary triangle, sinebow, HSV)
# ============================================================

def plot_oklab_trajectory_4(
    rgb_ellipse_srgb: np.ndarray,
    rgb_primary_srgb: np.ndarray,
    rgb_sine_srgb: np.ndarray,
    rgb_hsv_srgb: np.ndarray,
):
    def srgb_ring_to_lab(srgb_ring):
        rgb_lin = srgb_to_linear(torch.from_numpy(srgb_ring).to(device=device, dtype=dtype))
        lab = linear_to_oklab(rgb_lin).cpu().numpy()
        return lab

    lab_ellipse = srgb_ring_to_lab(rgb_ellipse_srgb)
    lab_primary = srgb_ring_to_lab(rgb_primary_srgb)
    lab_sine    = srgb_ring_to_lab(rgb_sine_srgb)
    lab_hsv     = srgb_ring_to_lab(rgb_hsv_srgb)

    fig = go.Figure()

    # warped cube faces
    for f in faces:
        Lg, ag, bg = face_surfaces[f]
        fig.add_trace(go.Surface(
            x=Lg, y=ag, z=bg,
            surfacecolor=np.zeros_like(Lg),
            colorscale=[[0,"rgb(120,120,120)"],[1,"rgb(120,120,120)"]],
            showscale=False,
            opacity=0.15,
            hoverinfo="skip",
            lighting=dict(ambient=1.0, diffuse=0.0, specular=0.0),
            showlegend=False,
        ))

    # cube edges
    verts = np.array(list(itertools.product([0.0,1.0],[0.0,1.0],[0.0,1.0])), dtype=np.float32)
    edges = [(i,j) for i in range(8) for j in range(i+1,8)
             if np.sum(np.abs(verts[i]-verts[j])) == 1.0]
    t_edge = np.linspace(0.0,1.0,64,dtype=np.float32)
    for i,j in edges:
        p0 = verts[i]; p1 = verts[j]
        srgb_line = p0 + (p1-p0)[None,:] * t_edge[:,None]
        lab_edge = linear_to_oklab(
            srgb_to_linear(torch.from_numpy(srgb_line).to(device=device, dtype=dtype))
        ).cpu().numpy()
        fig.add_trace(go.Scatter3d(
            x=lab_edge[:,0], y=lab_edge[:,1], z=lab_edge[:,2],
            mode="lines",
            line=dict(width=3, color="rgb(128,128,128)"),
            hoverinfo="skip",
            showlegend=False,
        ))

    # ellipse-based path
    fig.add_trace(go.Scatter3d(
        x=lab_ellipse[:,0], y=lab_ellipse[:,1], z=lab_ellipse[:,2],
        mode="lines+markers",
        line=dict(width=6, color="#ffffff"),
        marker=dict(size=4, color=rgb_ellipse_srgb),
        name="OKLab-plane circle",
    ))

    # primary triangle path
    fig.add_trace(go.Scatter3d(
        x=lab_primary[:,0], y=lab_primary[:,1], z=lab_primary[:,2],
        mode="lines+markers",
        line=dict(width=5, color="#ff00ff"),
        marker=dict(size=4, color=rgb_primary_srgb),
        name="RGB-primary triangle",
    ))

    # sinebow
    fig.add_trace(go.Scatter3d(
        x=lab_sine[:,0], y=lab_sine[:,1], z=lab_sine[:,2],
        mode="lines+markers",
        line=dict(width=4, color="#aaaaaa"),
        marker=dict(size=4, color=rgb_sine_srgb),
        name="sinebow",
    ))

    # HSV
    fig.add_trace(go.Scatter3d(
        x=lab_hsv[:,0], y=lab_hsv[:,1], z=lab_hsv[:,2],
        mode="lines+markers",
        line=dict(width=4, color="#cccccc"),
        marker=dict(size=4, color=rgb_hsv_srgb),
        name="HSV",
    ))

    gray = "rgb(200,200,200)"
    axis_style = dict(
        showbackground=False,
        gridcolor="rgba(200,200,200,0.25)",
        zerolinecolor="rgba(200,200,200,0.35)",
        tickfont=dict(color=gray),
    )

    fig.update_layout(
        width=1000, height=780,
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        scene=dict(
            bgcolor="rgba(0,0,0,0)",
            xaxis=dict(axis_style, title=dict(text="L", font=dict(color=gray))),
            yaxis=dict(axis_style, title=dict(text="a", font=dict(color=gray))),
            zaxis=dict(axis_style, title=dict(text="b", font=dict(color=gray))),
            aspectmode="data",
        ),
        font=dict(color=gray),
        legend=dict(
            font=dict(color=gray, size=14),
            bgcolor="rgba(0,0,0,0)"
        ),
        title=dict(
            text="OKLab: warped sRGB cube + ellipse/primary/sinebow/HSV",
            font=dict(color=gray, size=20),
        )
    )

    return HTML(pio.to_html(fig, include_plotlyjs="cdn", full_html=False))

def build_oklab_plane_triangle_ring(n_samples: int = 72) -> np.ndarray:
    """
    Triangle in OKLab RGB-primary plane with vertices exactly at
    OKLab(R), OKLab(G), OKLab(B), and straight edges in OKLab.

    n_samples must be divisible by 3: one segment per edge.
    Returns sRGB ring: [n_samples, 3].
    """
    assert n_samples % 3 == 0, "n_samples must be divisible by 3"
    n_edge = n_samples // 3

    # Primaries in sRGB
    prim_srgb = torch.tensor(
        [
            [1.0, 0.0, 0.0],  # R
            [0.0, 1.0, 0.0],  # G
            [0.0, 0.0, 1.0],  # B
        ],
        dtype=dtype,
        device=device,
    )
    prim_lin = srgb_to_linear(prim_srgb)
    prim_lab = linear_to_oklab(prim_lin)  # [3,3]

    L_R, a_R, b_R = prim_lab[0]
    L_G, a_G, b_G = prim_lab[1]
    L_B, a_B, b_B = prim_lab[2]

    # Parameter t in [0,1] including endpoint, then drop last to avoid duplicates
    t_edge = torch.linspace(0.0, 1.0, n_edge + 1, dtype=dtype, device=device)[:-1]

    # R -> G
    L_rg = (1.0 - t_edge) * L_R + t_edge * L_G
    a_rg = (1.0 - t_edge) * a_R + t_edge * a_G
    b_rg = (1.0 - t_edge) * b_R + t_edge * b_G

    # G -> B
    L_gb = (1.0 - t_edge) * L_G + t_edge * L_B
    a_gb = (1.0 - t_edge) * a_G + t_edge * a_B
    b_gb = (1.0 - t_edge) * b_G + t_edge * b_B

    # B -> R
    L_br = (1.0 - t_edge) * L_B + t_edge * L_R
    a_br = (1.0 - t_edge) * a_B + t_edge * a_R
    b_br = (1.0 - t_edge) * b_B + t_edge * b_R

    # Concatenate segments
    L_all = torch.cat([L_rg, L_gb, L_br], dim=0)
    a_all = torch.cat([a_rg, a_gb, a_br], dim=0)
    b_all = torch.cat([b_rg, b_gb, b_br], dim=0)

    # Explicitly close the loop by appending the first point
    L_all = torch.cat([L_all, L_all[:1]], dim=0)
    a_all = torch.cat([a_all, a_all[:1]], dim=0)
    b_all = torch.cat([b_all, b_all[:1]], dim=0)

    lab_ring = torch.stack([L_all, a_all, b_all], dim=-1)  # [n_samples+1,3]

    # OKLab -> linear sRGB -> sRGB
    rgb_lin_ring = oklab_to_linear(lab_ring)
    rgb_srgb_ring = linear_to_srgb(rgb_lin_ring).clamp(0.0, 1.0).cpu().numpy()
    return rgb_srgb_ring


def compute_rgb_primary_oklab_plane():
    prim_srgb = torch.tensor(
        [
            [1.0, 0.0, 0.0],  # R
            [0.0, 1.0, 0.0],  # G
            [0.0, 0.0, 1.0],  # B
        ],
        dtype=dtype,
        device=device,
    )
    prim_lin = srgb_to_linear(prim_srgb)
    prim_lab = linear_to_oklab(prim_lin)  # [3,3]

    c = prim_lab.mean(dim=0)  # center in OKLab
    v1 = prim_lab[0] - c
    v2 = prim_lab[1] - c

    e1 = v1 / (v1.norm() + 1e-8)
    v2_orth = v2 - torch.dot(v2, e1) * e1
    e2 = v2_orth / (v2_orth.norm() + 1e-8)

    return c.cpu().numpy(), e1.cpu().numpy(), e2.cpu().numpy()


def slice_oklab_gamut_in_plane(c, e1, e2, nsamp=200, plane_tol=0.01):
    """
    Sample the sRGB cube, map to OKLab, and project onto the RGB-primary OKLab plane.
    Keep only points whose perpendicular distance to the plane is < plane_tol.
    Return 2D coords in (u,v) relative to (e1,e2).
    """
    vals = np.linspace(0.0, 1.0, nsamp)
    rr, gg, bb = np.meshgrid(vals, vals, vals, indexing="ij")
    rgb = np.stack([rr, gg, bb], axis=-1).reshape(-1, 3)  # [N,3]

    rgb_t = torch.from_numpy(rgb).to(device=device, dtype=dtype)
    rgb_lin = srgb_to_linear(rgb_t)
    lab = linear_to_oklab(rgb_lin).cpu().numpy()  # [N,3]

    delta = lab - c[None, :]
    u = delta @ e1
    v = delta @ e2

    # Perpendicular component to plane
    w = delta - (u[:, None] * e1[None, :] + v[:, None] * e2[None, :])
    dist = np.linalg.norm(w, axis=1)

    mask = dist < plane_tol
    uv = np.stack([u[mask], v[mask]], axis=1)  # [M,2]

    # Convex hull in uv-plane
    hull = ConvexHull(uv)
    poly = uv[hull.vertices]  # polygon vertices in order
    return poly
    
def max_inscribed_ellipse(poly):
    """
    Maximum-area ellipse inscribed in a convex polygon (2D).
    poly: [K,2] vertices in order.
    Returns center c_uv (2,), A (2x2) such that ellipse = { c + A x : ||x|| <= 1 }.
    """
    K = poly.shape[0]
    c = cp.Variable((2,1))
    A = cp.Variable((2,2), PSD=True)

    constraints = []
    for i in range(K):
        p = poly[i][:, None]
        p1 = poly[(i+1) % K][:, None]
        edge = (p1 - p).T  # (1,2)
        # outward normal (arbitrary orientation; sign doesn’t matter as long as consistent)
        normal = np.array([[edge[0,1], -edge[0,0]]])
        normal = normal / (np.linalg.norm(normal) + 1e-12)
        n = normal.T  # (2,1)

        # For all x s.t. ||x|| <= 1, normal·(c+Ax - p) <= 0
        # => ||A^T n|| <= (n^T (p - c))
        constraints.append(cp.norm(A.T @ n, 2) + n.T @ c <= n.T @ p)

    problem = cp.Problem(cp.Maximize(cp.log_det(A)), constraints)
    problem.solve(solver=cp.SCS, verbose=False)

    return np.asarray(c.value).reshape(2), np.asarray(A.value)

def build_oklab_plane_free_ellipse_ring(
    n_samples: int = 512,
    plane_nsamp: int = 200,
    plane_tol: float = 0.01,
    clip_eps: float = 1e-6,
) -> np.ndarray:
    """
    Largest ellipse (by Löwner/John) in the RGB-primary OKLab plane,
    then minimally shrunk around its own center in OKLab so that every
    sample maps to linear sRGB in [0,1] within clip_eps.

    Returns sRGB ring [n_samples,3].
    """
    # 1) plane basis
    c_lab, e1, e2 = compute_rgb_primary_oklab_plane()

    # 2) 2D slice polygon in uv-plane
    poly = slice_oklab_gamut_in_plane(c_lab, e1, e2, nsamp=plane_nsamp, plane_tol=plane_tol)

    # 3) maximal inscribed ellipse in uv-plane
    center_uv, A = max_inscribed_ellipse(poly)

    # 4) sample ellipse in uv
    theta = np.linspace(0.0, 2.0 * np.pi, n_samples, endpoint=False)
    unit = np.stack([np.cos(theta), np.sin(theta)], axis=1)  # [N,2]
    uv = center_uv[None, :] + unit @ A.T                     # [N,2]

    # 5) map uv -> OKLab
    lab_np = (
        c_lab[None, :]
        + uv[:, 0:1] * e1[None, :]
        + uv[:, 1:2] * e2[None, :]
    )  # [N,3]

    lab_t = torch.from_numpy(lab_np).to(device=device, dtype=dtype)

    # center of ellipse in OKLab (could be different from c_lab)
    center_t = lab_t.mean(dim=0)
    offsets = lab_t - center_t[None, :]

    def max_violation(scale: float) -> float:
        lab_scaled = center_t[None, :] + scale * offsets
        rgb_lin = oklab_to_linear(lab_scaled)
        below = -rgb_lin.min().item()
        above = (rgb_lin - 1.0).max().item()
        return max(below, above, 0.0)

    # 6) find largest scale <=1 with violation <= clip_eps
    s_lo, s_hi = 0.0, 1.0
    # if already safe at scale=1, keep it
    if max_violation(1.0) <= clip_eps:
        s_opt = 1.0
    else:
        for _ in range(40):
            mid = 0.5 * (s_lo + s_hi)
            if max_violation(mid) <= clip_eps:
                s_lo = mid
            else:
                s_hi = mid
        s_opt = s_lo

    lab_safe = center_t[None, :] + s_opt * offsets
    rgb_lin_safe = oklab_to_linear(lab_safe)
    rgb_srgb_safe = linear_to_srgb(rgb_lin_safe).clamp(0.0, 1.0).cpu().numpy()
    return rgb_srgb_safe
    
# ============================================================
# 6) Main: 4 colormaps on Omnipose flows + trajectory
# ============================================================

def visualize_ellipse_primary_sinebow_hsv_on_omnipose(
    n_hues: int = 72,
    device_str: str = "cpu",
):
    dev = torch.device(device_str)

    # 1) Rings in sRGB
    # rgb_ellipse_srgb = build_oklab_plane_circle_ring(n_samples=n_hues)
    rgb_ellipse_srgb = build_oklab_plane_free_ellipse_ring(n_samples=n_hues)

    rgb_primary_srgb = build_oklab_plane_triangle_ring(n_samples=n_hues)
    t = np.linspace(0, 1, n_hues, endpoint=False)

    # sinebow
    sine_r = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3))
    sine_g = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3))
    sine_b = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_srgb = np.stack([sine_r, sine_g, sine_b], axis=1)

    # HSV
    hsv_vals = np.stack([t, np.ones_like(t), np.ones_like(t)], axis=1)
    rgb_hsv_srgb = hsv_to_rgb(hsv_vals)

    # 2) Trajectory plot in OKLab
    traj_html = plot_oklab_trajectory_4(
        rgb_ellipse_srgb,
        rgb_primary_srgb,
        rgb_sine_srgb,
        rgb_hsv_srgb,
    )
    display(traj_html)

    # 3) Convert rings to linear for disks/flows
    rgb_ellipse_lin = srgb_to_linear_np(rgb_ellipse_srgb)
    rgb_primary_lin = srgb_to_linear_np(rgb_primary_srgb)
    rgb_sine_lin    = srgb_to_linear_np(rgb_sine_srgb)
    rgb_hsv_lin     = srgb_to_linear_np(rgb_hsv_srgb)

    center_srgb = np.array([0.5, 0.5, 0.5], dtype=float)
    center_lin = srgb_to_linear_np(center_srgb)

    # 4) Load Omnipose E. coli flows
    omnidir = Path(omnirefactor.__file__).parent.parent.parent
    basedir = os.path.join(omnidir, "docs", "_static")
    name = "ecoli"
    ext = ".tif"

    image = io.imread(os.path.join(basedir, name + ext))
    masks = io.imread(os.path.join(basedir, name + "_labels" + ext))
    slc = ocdkit.measure.crop_bbox(masks > 0, pad=0)[0]
    masks = fastremap.renumber(masks[slc])[0]
    image = image[slc]

    flows_all = omnirefactor.core.masks_to_flows(masks, device="cpu")[4].to(dev)
    flows = flows_all

    # 5) Build images for each ring
    disk_ell, flow_ell, flow_ell_white, flow_ell_black = make_flow_images_for_ring(
        rgb_ellipse_lin, center_lin, flows
    )
    disk_pri, flow_pri, flow_pri_white, flow_pri_black = make_flow_images_for_ring(
        rgb_primary_lin, center_lin, flows
    )
    disk_sine, flow_sine, flow_sine_white, flow_sine_black = make_flow_images_for_ring(
        rgb_sine_lin, center_lin, flows
    )
    disk_hsv, flow_hsv, flow_hsv_white, flow_hsv_black = make_flow_images_for_ring(
        rgb_hsv_lin, center_lin, flows
    )

    # 6) Plot 4×4 grid with row labels
    fig, axes = plt.subplots(4, 4, figsize=(16, 13))
    row_titles = [
        "OKLab-plane circle",
        "RGB-primary triangle",
        "sinebow",
        "HSV",
    ]

    # Row 0: ellipse
    axes[0,0].imshow(disk_ell);           axes[0,0].set_title("Hue disk");                  axes[0,0].axis("off")
    axes[0,1].imshow(flow_ell);           axes[0,1].set_title("Flow mag \u2192 chroma");    axes[0,1].axis("off")
    axes[0,2].imshow(flow_ell_white);     axes[0,2].set_title("Flow mag \u2192 alpha (white)");  axes[0,2].axis("off")
    axes[0,3].imshow(flow_ell_black);     axes[0,3].set_title("Flow mag \u2192 alpha (black)");  axes[0,3].axis("off")

    # Row 1: primary triangle
    axes[1,0].imshow(disk_pri);           axes[1,0].axis("off")
    axes[1,1].imshow(flow_pri);           axes[1,1].axis("off")
    axes[1,2].imshow(flow_pri_white);     axes[1,2].axis("off")
    axes[1,3].imshow(flow_pri_black);     axes[1,3].axis("off")

    # Row 2: sinebow
    axes[2,0].imshow(disk_sine);          axes[2,0].axis("off")
    axes[2,1].imshow(flow_sine);          axes[2,1].axis("off")
    axes[2,2].imshow(flow_sine_white);    axes[2,2].axis("off")
    axes[2,3].imshow(flow_sine_black);    axes[2,3].axis("off")

    # Row 3: HSV
    axes[3,0].imshow(disk_hsv);           axes[3,0].axis("off")
    axes[3,1].imshow(flow_hsv);           axes[3,1].axis("off")
    axes[3,2].imshow(flow_hsv_white);     axes[3,2].axis("off")
    axes[3,3].imshow(flow_hsv_black);     axes[3,3].axis("off")

    for i, title in enumerate(row_titles):
        axes[i,0].set_ylabel(title, fontsize=12)

    plt.tight_layout()
    plt.show()

# ============================================================
# Run everything
# ============================================================

if __name__ == "__main__":
    visualize_ellipse_primary_sinebow_hsv_on_omnipose(
        n_hues=72,  # must be divisible by 3 for the RGB-primary triangle ring
        device_str="cpu",
    )

In [ ]:
make a ;argest ellipse in rgb plane - hard to do

the hot spots appear any time the path makes a sharp corner, mayeb have a minimum radius? defined by delta E curvature? 

an intrprolation in colorpace between the corners of the rgb cube - fourier, done

In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import hsv_to_rgb

import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display

import omnirefactor
from omnirefactor.plot import imshow
from omnirefactor import io
import fastremap
from pathlib import Path
import os
import itertools

# ============================================================
# Global config
# ============================================================

dtype = torch.float32
device = torch.device("cpu")

# ============================================================
# Torch: sRGB <-> linear, OKLab
# ============================================================

def srgb_to_linear(srgb: torch.Tensor) -> torch.Tensor:
    srgb = srgb.to(dtype=dtype).clamp(0.0, 1.0)
    threshold = 0.04045
    low = srgb <= threshold
    high = ~low
    out = torch.zeros_like(srgb)
    out[low] = srgb[low] / 12.92
    out[high] = ((srgb[high] + 0.055) / 1.055) ** 2.4
    return out

def linear_to_srgb(linear: torch.Tensor) -> torch.Tensor:
    linear = linear.to(dtype=dtype)
    threshold = 0.0031308
    low = linear <= threshold
    high = ~low
    out = torch.zeros_like(linear)
    out[low] = 12.92 * linear[low]
    out[high] = 1.055 * (linear[high].clamp(min=0.0)) ** (1.0 / 2.4) - 0.055
    return out

# OKLab matrices
M1 = torch.tensor([
    [0.4122214708, 0.5363325363, 0.0514459929],
    [0.2119034982, 0.6806995451, 0.1073969566],
    [0.0883024619, 0.2817188376, 0.6299787005],
], dtype=dtype, device=device)

M2 = torch.tensor([
    [ 0.2104542553,  0.7936177850, -0.0040720468],
    [ 1.9779984951, -2.4285922050,  0.4505937099],
    [ 0.0259040371,  0.7827717662, -0.8086757660],
], dtype=dtype, device=device)

M1_inv = torch.tensor([
    [ 4.0767416621, -3.3077115913,  0.2309699292],
    [-1.2684380046,  2.6097574011, -0.3413193965],
    [-0.0041960863, -0.7034186147,  1.7076147010],
], dtype=dtype, device=device)

M2_inv = torch.tensor([
    [1.0000000000,  0.3963377774,  0.2158037573],
    [1.0000000000, -0.1055613458, -0.0638541728],
    [1.0000000000, -0.0894841775, -1.2914855480],
], dtype=dtype, device=device)

def linear_to_oklab(rgb_lin: torch.Tensor) -> torch.Tensor:
    shape = rgb_lin.shape
    x = rgb_lin.reshape(-1, 3)
    lms = x @ M1.T
    lms = torch.clamp(lms, min=0.0)
    lms_cbrt = torch.sign(lms) * torch.abs(lms) ** (1.0 / 3.0)
    lab = lms_cbrt @ M2.T
    return lab.reshape(shape)

def oklab_to_linear(lab: torch.Tensor) -> torch.Tensor:
    shape = lab.shape
    x = lab.reshape(-1, 3)
    lms_cbrt = x @ M2_inv.T
    lms = lms_cbrt ** 3
    rgb_lin = lms @ M1_inv.T
    return rgb_lin.reshape(shape)

# ============================================================
# NumPy sRGB <-> linear (for disks / flows)
# ============================================================

def srgb_to_linear_np(rgb):
    rgb = np.asarray(rgb, dtype=float)
    a = 0.055
    return np.where(rgb <= 0.04045, rgb / 12.92,
                    ((rgb + a) / (1 + a))**2.4)

def linear_to_srgb_np(rgb):
    rgb = np.asarray(rgb, dtype=float)
    a = 0.055
    return np.where(
        rgb <= 0.0031308,
        12.92 * rgb,
        (1 + a) * np.power(np.clip(rgb, 0, None), 1 / 2.4) - a,
    )

# ============================================================
# HSV hexagon path in sRGB (RGB + CMY corners)
# ============================================================

def build_hsv_hex_path_srgb(n_samples: int = 512) -> np.ndarray:
    """
    Classic HSV path: R -> Y -> G -> C -> B -> M -> R.
    Returns [n_samples,3] sRGB.
    """
    t = np.linspace(0, 1, n_samples, endpoint=False)
    hsv_vals = np.stack([t, np.ones_like(t), np.ones_like(t)], axis=1)
    rgb = hsv_to_rgb(hsv_vals)
    return rgb

# ============================================================
# Fourier smoothing in OKLab around HSV hex path
# ============================================================

def fit_fourier_oklab_from_srgb(rgb_srgb: np.ndarray,
                                n_harmonics: int,
                                n_samples: int = None) -> (np.ndarray, np.ndarray):
    """
    Take an sRGB ring, convert to OKLab, fit truncated Fourier series
    for L(θ), a(θ), b(θ) with n_harmonics, then re-evaluate to get
    a smooth OKLab ring and its sRGB version.

    Returns:
        lab_smooth: [N,3] OKLab ring
        rgb_smooth: [N,3] sRGB ring
    """
    if n_samples is None:
        n_samples = rgb_srgb.shape[0]

    # Samples along θ
    N = rgb_srgb.shape[0]
    theta = np.linspace(0.0, 2.0 * np.pi, N, endpoint=False)

    # Convert to OKLab
    rgb_t = torch.from_numpy(rgb_srgb).to(device=device, dtype=dtype)
    lab_t = linear_to_oklab(srgb_to_linear(rgb_t))
    lab_np = lab_t.cpu().numpy()  # [N,3]

    # Design matrix: 1, cos(kθ), sin(kθ) for k=1..n_harmonics
    cols = [np.ones_like(theta)]
    for k in range(1, n_harmonics + 1):
        cols.append(np.cos(k * theta))
        cols.append(np.sin(k * theta))
    X = np.stack(cols, axis=1)  # [N, 1+2M]

    # Fit each component separately
    coeffs = []
    for c_idx in range(3):
        y = lab_np[:, c_idx]
        beta, *_ = np.linalg.lstsq(X, y, rcond=None)
        coeffs.append(beta)
    coeffs = np.stack(coeffs, axis=0)  # [3, K]

    # Re-evaluate on n_samples
    theta_out = np.linspace(0.0, 2.0 * np.pi, n_samples, endpoint=False)
    cols_out = [np.ones_like(theta_out)]
    for k in range(1, n_harmonics + 1):
        cols_out.append(np.cos(k * theta_out))
        cols_out.append(np.sin(k * theta_out))
    X_out = np.stack(cols_out, axis=1)  # [N_out, 1+2M]

    lab_smooth = (coeffs @ X_out.T).T  # [N_out,3]

    # Safe scaling in OKLab around center to guarantee in-gamut
    lab_t2 = torch.from_numpy(lab_smooth).to(device=device, dtype=dtype)
    center = lab_t2.mean(dim=0)
    offsets = lab_t2 - center[None, :]

    def is_safe(scale: float) -> bool:
        lab_scaled = center[None, :] + scale * offsets
        rgb_lin = oklab_to_linear(lab_scaled)
        return bool(((rgb_lin >= -1e-6) & (rgb_lin <= 1.0 + 1e-6)).all())

    s_lo, s_hi = 0.0, 1.0
    if not is_safe(1.0):
        for _ in range(40):
            mid = 0.5 * (s_lo + s_hi)
            if is_safe(float(mid)):
                s_lo = mid
            else:
                s_hi = mid
        s_opt = s_lo
    else:
        s_opt = 1.0

    lab_safe = center[None, :] + s_opt * offsets
    rgb_lin_safe = oklab_to_linear(lab_safe)
    rgb_srgb_safe = linear_to_srgb(rgb_lin_safe).clamp(0.0, 1.0).cpu().numpy()

    return lab_safe.cpu().numpy(), rgb_srgb_safe

# ============================================================
# Hue disk + flow images from a linear RGB ring
# ============================================================

def build_hue_disk_from_ring(rgb_ring_lin: np.ndarray,
                             center_lin: np.ndarray,
                             size: int = 512) -> np.ndarray:
    n_hues = rgb_ring_lin.shape[0]
    N = size
    y, x = np.mgrid[0:N, 0:N]
    x = (x + 0.5) / N * 2 - 1
    y = (y + 0.5) / N * 2 - 1
    r = np.sqrt(x * x + y * y)
    theta = np.mod(np.arctan2(y, x), 2 * np.pi)

    hue_f = theta / (2 * np.pi) * n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)

    ring0 = rgb_ring_lin[idx0]
    ring1 = rgb_ring_lin[idx1]
    ring_interp = (1 - t[..., None]) * ring0 + t[..., None] * ring1

    r_exp = r[..., None]
    rgb_lin = (1 - r_exp) * center_lin + r_exp * ring_interp
    rgb_lin[r > 1] = 1.0

    return np.clip(linear_to_srgb_np(rgb_lin), 0, 1)

def make_flow_images_for_ring(rgb_ring_lin: np.ndarray,
                              center_lin: np.ndarray,
                              flows: torch.Tensor):
    n_hues = rgb_ring_lin.shape[0]

    disk = build_hue_disk_from_ring(rgb_ring_lin, center_lin, size=512)

    u = flows[0].cpu().numpy()
    v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2 * np.pi)
    mag = np.sqrt(u * u + v * v)
    mag /= (mag.max() + 1e-8)

    hue_f = angle / (2 * np.pi) * n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)

    ring0 = rgb_ring_lin[idx0]
    ring1 = rgb_ring_lin[idx1]
    ring_interp = (1 - t[..., None]) * ring0 + t[..., None] * ring1

    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow), 0, 1)

    alpha = mag[..., None]
    white = np.ones_like(flow_rgb)
    black = np.zeros_like(flow_rgb)
    flow_white = alpha * flow_rgb + (1 - alpha) * white
    flow_black = alpha * flow_rgb + (1 - alpha) * black

    return disk, flow_rgb, flow_white, flow_black

# ============================================================
# Warped sRGB cube faces for OKLab trajectory
# ============================================================

def make_face_oklab(face: str, n: int = 96):
    t = np.linspace(0.0, 1.0, n, dtype=np.float32)
    u, v = np.meshgrid(t, t, indexing="ij")
    srgb = np.zeros((n, n, 3), dtype=np.float32)

    if   face == "R0": srgb[...,0]=0.0; srgb[...,1]=u; srgb[...,2]=v
    elif face == "R1": srgb[...,0]=1.0; srgb[...,1]=u; srgb[...,2]=v
    elif face == "G0": srgb[...,1]=0.0; srgb[...,0]=u; srgb[...,2]=v
    elif face == "G1": srgb[...,1]=1.0; srgb[...,0]=u; srgb[...,2]=v
    elif face == "B0": srgb[...,2]=0.0; srgb[...,0]=u; srgb[...,1]=v
    elif face == "B1": srgb[...,2]=1.0; srgb[...,0]=u; srgb[...,1]=v
    else:
        raise ValueError("bad face name")

    srgb_t = torch.from_numpy(srgb.reshape(-1,3)).to(device=device, dtype=dtype)
    rgb_lin = srgb_to_linear(srgb_t)
    lab = linear_to_oklab(rgb_lin).cpu().numpy().reshape(n,n,3)
    return lab[...,0], lab[...,1], lab[...,2]

faces = ["R0","R1","G0","G1","B0","B1"]
face_surfaces = {f: make_face_oklab(f, n=96) for f in faces}

# ============================================================
# 3D OKLab trajectory: Fourier-smoothed vs HSV hex vs sinebow vs HSV
# ============================================================

def plot_oklab_trajectory_hex_fourier_sine_hsv(
    rgb_fourier_srgb: np.ndarray,
    rgb_hex_srgb: np.ndarray,
    rgb_sine_srgb: np.ndarray,
    rgb_hsv_srgb: np.ndarray,
):
    def srgb_ring_to_lab(srgb_ring):
        rgb_lin = srgb_to_linear(torch.from_numpy(srgb_ring).to(device=device, dtype=dtype))
        lab = linear_to_oklab(rgb_lin).cpu().numpy()
        return lab

    lab_fourier = srgb_ring_to_lab(rgb_fourier_srgb)
    lab_hex     = srgb_ring_to_lab(rgb_hex_srgb)
    lab_sine    = srgb_ring_to_lab(rgb_sine_srgb)
    lab_hsv     = srgb_ring_to_lab(rgb_hsv_srgb)

    fig = go.Figure()

    # warped cube faces
    for f in faces:
        Lg, ag, bg = face_surfaces[f]
        fig.add_trace(go.Surface(
            x=Lg, y=ag, z=bg,
            surfacecolor=np.zeros_like(Lg),
            colorscale=[[0,"rgb(120,120,120)"],[1,"rgb(120,120,120)"]],
            showscale=False,
            opacity=0.15,
            hoverinfo="skip",
            lighting=dict(ambient=1.0, diffuse=0.0, specular=0.0),
            showlegend=False,
        ))

    # cube edges
    verts = np.array(list(itertools.product([0.0,1.0],[0.0,1.0],[0.0,1.0])), dtype=np.float32)
    edges = [(i,j) for i in range(8) for j in range(i+1,8)
             if np.sum(np.abs(verts[i]-verts[j])) == 1.0]
    t_edge = np.linspace(0.0,1.0,64,dtype=np.float32)
    for i,j in edges:
        p0 = verts[i]; p1 = verts[j]
        srgb_line = p0 + (p1-p0)[None,:] * t_edge[:,None]
        lab_edge = linear_to_oklab(
            srgb_to_linear(torch.from_numpy(srgb_line).to(device=device, dtype=dtype))
        ).cpu().numpy()
        fig.add_trace(go.Scatter3d(
            x=lab_edge[:,0], y=lab_edge[:,1], z=lab_edge[:,2],
            mode="lines",
            line=dict(width=3, color="rgb(128,128,128)"),
            hoverinfo="skip",
            showlegend=False,
        ))

    # Fourier-smoothed HSV hex (colored along curve)
    fig.add_trace(go.Scatter3d(
        x=lab_fourier[:,0], y=lab_fourier[:,1], z=lab_fourier[:,2],
        mode="lines+markers",
        line=dict(width=6, color="#ffffff"),
        marker=dict(size=4, color=rgb_fourier_srgb),
        name="Fourier-smoothed HSV hex",
    ))

    # Raw HSV hex path (corners)
    fig.add_trace(go.Scatter3d(
        x=lab_hex[:,0], y=lab_hex[:,1], z=lab_hex[:,2],
        mode="lines",
        line=dict(width=3, color="#ff00ff"),
        name="HSV hex corners",
    ))

    # sinebow, colored along curve
    fig.add_trace(go.Scatter3d(
        x=lab_sine[:,0], y=lab_sine[:,1], z=lab_sine[:,2],
        mode="lines+markers",
        line=dict(width=2, color="#888888"),
        marker=dict(size=3, color=rgb_sine_srgb),
        name="sinebow",
    ))

    # HSV smooth wheel, colored along curve
    fig.add_trace(go.Scatter3d(
        x=lab_hsv[:,0], y=lab_hsv[:,1], z=lab_hsv[:,2],
        mode="lines+markers",
        line=dict(width=2, color="#cccccc"),
        marker=dict(size=3, color=rgb_hsv_srgb),
        name="HSV",
    ))

    gray = "rgb(200,200,200)"
    axis_style = dict(
        showbackground=False,
        gridcolor="rgba(200,200,200,0.25)",
        zerolinecolor="rgba(200,200,200,0.35)",
        tickfont=dict(color=gray),
    )

    fig.update_layout(
        width=1000, height=780,
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        scene=dict(
            bgcolor="rgba(0,0,0,0)",
            xaxis=dict(axis_style, title=dict(text="L", font=dict(color=gray))),
            yaxis=dict(axis_style, title=dict(text="a", font=dict(color=gray))),
            zaxis=dict(axis_style, title=dict(text="b", font=dict(color=gray))),
            aspectmode="data",
        ),
        font=dict(color=gray),
        legend=dict(
            font=dict(color=gray, size=14),
            bgcolor="rgba(0,0,0,0)"
        ),
        title=dict(
            text="OKLab: warped sRGB cube + Fourier HSV / hex / sinebow / HSV",
            font=dict(color=gray, size=20),
        )
    )

    return HTML(pio.to_html(fig, include_plotlyjs="cdn", full_html=False))

# ============================================================
# Main: Fourier-smoothed HSV hex vs sinebow vs HSV on Omnipose
# ============================================================

def visualize_fourier_hex_vs_sinebow_hsv_on_omnipose(
    n_hues: int = 72,
    n_harmonics: int = 4,
    device_str: str = "cpu",
):
    dev = torch.device(device_str)

    # 1) base rings in sRGB
    rgb_hex_srgb = build_hsv_hex_path_srgb(n_samples=n_hues)

    # sinebow
    t = np.linspace(0, 1, n_hues, endpoint=False)
    sine_r = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3))
    sine_g = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3))
    sine_b = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_srgb = np.stack([sine_r, sine_g, sine_b], axis=1)

    # HSV smooth wheel
    hsv_vals = np.stack([t, np.ones_like(t), np.ones_like(t)], axis=1)
    rgb_hsv_srgb = hsv_to_rgb(hsv_vals)

    # 2) Fourier-smoothed HSV hex in OKLab
    lab_fourier, rgb_fourier_srgb = fit_fourier_oklab_from_srgb(
        rgb_hex_srgb, n_harmonics=n_harmonics, n_samples=n_hues
    )

    # 3) Trajectory plot in OKLab
    traj_html = plot_oklab_trajectory_hex_fourier_sine_hsv(
        rgb_fourier_srgb,
        rgb_hex_srgb,
        rgb_sine_srgb,
        rgb_hsv_srgb,
    )
    display(traj_html)

    # 4) Convert all rings to linear RGB for disks/flows
    rgb_fourier_lin = srgb_to_linear_np(rgb_fourier_srgb)
    rgb_sine_lin    = srgb_to_linear_np(rgb_sine_srgb)
    rgb_hsv_lin     = srgb_to_linear_np(rgb_hsv_srgb)

    center_srgb = np.array([0.5, 0.5, 0.5], dtype=float)
    center_lin = srgb_to_linear_np(center_srgb)

    # 5) Load Omnipose E. coli flows
    omnidir = Path(omnirefactor.__file__).parent.parent.parent
    basedir = os.path.join(omnidir, "docs", "_static")
    name = "ecoli"
    ext = ".tif"

    image = io.imread(os.path.join(basedir, name + ext))
    masks = io.imread(os.path.join(basedir, name + "_labels" + ext))
    slc = ocdkit.measure.crop_bbox(masks > 0, pad=0)[0]
    masks = fastremap.renumber(masks[slc])[0]
    image = image[slc]

    flows_all = omnirefactor.core.masks_to_flows(masks, device="cpu")[4].to(dev)
    flows = flows_all

    # 6) Build images for each colormap
    disk_fourier, flow_fourier, flow_fourier_white, flow_fourier_black = make_flow_images_for_ring(
        rgb_fourier_lin, center_lin, flows
    )
    disk_sine, flow_sine, flow_sine_white, flow_sine_black = make_flow_images_for_ring(
        rgb_sine_lin, center_lin, flows
    )
    disk_hsv, flow_hsv, flow_hsv_white, flow_hsv_black = make_flow_images_for_ring(
        rgb_hsv_lin, center_lin, flows
    )

    # 7) Plot 3×4 grid
    fig, axes = plt.subplots(3, 4, figsize=(16, 10))
    row_titles = [
        f"Fourier-smoothed HSV hex (M={n_harmonics})",
        "sinebow",
        "HSV",
    ]

    # Row 0: Fourier-smoothed HSV hex
    axes[0,0].imshow(disk_fourier);           axes[0,0].set_title("Hue disk");                  axes[0,0].axis("off")
    axes[0,1].imshow(flow_fourier);           axes[0,1].set_title("Flow mag → chroma");         axes[0,1].axis("off")
    axes[0,2].imshow(flow_fourier_white);     axes[0,2].set_title("Flow mag → alpha (white)");  axes[0,2].axis("off")
    axes[0,3].imshow(flow_fourier_black);     axes[0,3].set_title("Flow mag → alpha (black)");  axes[0,3].axis("off")

    # Row 1: sinebow
    axes[1,0].imshow(disk_sine);              axes[1,0].axis("off")
    axes[1,1].imshow(flow_sine);              axes[1,1].axis("off")
    axes[1,2].imshow(flow_sine_white);        axes[1,2].axis("off")
    axes[1,3].imshow(flow_sine_black);        axes[1,3].axis("off")

    # Row 2: HSV
    axes[2,0].imshow(disk_hsv);               axes[2,0].axis("off")
    axes[2,1].imshow(flow_hsv);               axes[2,1].axis("off")
    axes[2,2].imshow(flow_hsv_white);         axes[2,2].axis("off")
    axes[2,3].imshow(flow_hsv_black);         axes[2,3].axis("off")

    for i, title in enumerate(row_titles):
        axes[i,0].set_ylabel(title, fontsize=12)

    plt.tight_layout()
    plt.show()

# ============================================================
# Run everything
# ============================================================

if __name__ == "__main__":
    N_HUES = 72        # discrete hues
    N_HARMONICS = 4    # Fourier components

    visualize_fourier_hex_vs_sinebow_hsv_on_omnipose(
        n_hues=N_HUES,
        n_harmonics=N_HARMONICS,
        device_str="cpu",
    )

In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import hsv_to_rgb

import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display

import omnirefactor
from omnirefactor.plot import imshow
from omnirefactor import io
import fastremap
from pathlib import Path
import os
import itertools

# ============================================================
# Global config
# ============================================================

dtype = torch.float32
device = torch.device("cpu")

# ============================================================
# Torch: sRGB <-> linear, OKLab
# ============================================================

def srgb_to_linear(srgb: torch.Tensor) -> torch.Tensor:
    srgb = srgb.to(dtype=dtype).clamp(0.0, 1.0)
    threshold = 0.04045
    low = srgb <= threshold
    high = ~low
    out = torch.zeros_like(srgb)
    out[low] = srgb[low] / 12.92
    out[high] = ((srgb[high] + 0.055) / 1.055) ** 2.4
    return out

def linear_to_srgb(linear: torch.Tensor) -> torch.Tensor:
    linear = linear.to(dtype=dtype)
    threshold = 0.0031308
    low = linear <= threshold
    high = ~low
    out = torch.zeros_like(linear)
    out[low] = 12.92 * linear[low]
    out[high] = 1.055 * (linear[high].clamp(min=0.0)) ** (1.0 / 2.4) - 0.055
    return out

M1 = torch.tensor([
    [0.4122214708, 0.5363325363, 0.0514459929],
    [0.2119034982, 0.6806995451, 0.1073969566],
    [0.0883024619, 0.2817188376, 0.6299787005],
], dtype=dtype, device=device)

M2 = torch.tensor([
    [ 0.2104542553,  0.7936177850, -0.0040720468],
    [ 1.9779984951, -2.4285922050,  0.4505937099],
    [ 0.0259040371,  0.7827717662, -0.8086757660],
], dtype=dtype, device=device)

M1_inv = torch.tensor([
    [ 4.0767416621, -3.3077115913,  0.2309699292],
    [-1.2684380046,  2.6097574011, -0.3413193965],
    [-0.0041960863, -0.7034186147,  1.7076147010],
], dtype=dtype, device=device)

M2_inv = torch.tensor([
    [1.0000000000,  0.3963377774,  0.2158037573],
    [1.0000000000, -0.1055613458, -0.0638541728],
    [1.0000000000, -0.0894841775, -1.2914855480],
], dtype=dtype, device=device)

def linear_to_oklab(rgb_lin: torch.Tensor) -> torch.Tensor:
    shape = rgb_lin.shape
    x = rgb_lin.reshape(-1, 3)
    lms = x @ M1.T
    lms = torch.clamp(lms, min=0.0)
    lms_cbrt = torch.sign(lms) * torch.abs(lms) ** (1.0 / 3.0)
    lab = lms_cbrt @ M2.T
    return lab.reshape(shape)

def oklab_to_linear(lab: torch.Tensor) -> torch.Tensor:
    shape = lab.shape
    x = lab.reshape(-1, 3)
    lms_cbrt = x @ M2_inv.T
    lms = lms_cbrt ** 3
    rgb_lin = lms @ M1_inv.T
    return rgb_lin.reshape(shape)

# ============================================================
# NumPy sRGB <-> linear (for disks / flows)
# ============================================================

def srgb_to_linear_np(rgb):
    rgb = np.asarray(rgb, dtype=float)
    a = 0.055
    return np.where(rgb <= 0.04045, rgb / 12.92,
                    ((rgb + a) / (1 + a))**2.4)

def linear_to_srgb_np(rgb):
    rgb = np.asarray(rgb, dtype=float)
    a = 0.055
    return np.where(
        rgb <= 0.0031308,
        12.92 * rgb,
        (1 + a) * np.power(np.clip(rgb, 0, None), 1 / 2.4) - a,
    )

# ============================================================
# HSV hexagon path in sRGB (RGB + CMY corners)
# ============================================================

def build_hsv_hex_path_srgb(n_samples: int) -> np.ndarray:
    """
    Classic HSV path: R -> Y -> G -> C -> B -> M -> R.
    Returns [n_samples,3] sRGB.
    """
    t = np.linspace(0, 1, n_samples, endpoint=False)
    hsv_vals = np.stack([t, np.ones_like(t), np.ones_like(t)], axis=1)
    rgb = hsv_to_rgb(hsv_vals)
    return rgb

# ============================================================
# Fourier smoothing in OKLab with arc-length reparameterization
# ============================================================

def build_fourier_hsv_hex_ring(n_hues: int,
                               n_harmonics: int,
                               oversample: int = 8) -> (np.ndarray, np.ndarray, np.ndarray, np.ndarray):
    """
    Build a Fourier-smoothed cyclic colormap:

      1) HSV hexagon path in sRGB with N_hi = n_hues * oversample.
      2) Convert to OKLab.
      3) Reparameterize by cumulative arc length in OKLab.
      4) Resample to n_hues points at equal arc steps (lab_arc, rgb_arc).
      5) Fit truncated Fourier series L/a/b(t) with n_harmonics.
      6) Evaluate smooth L/a/b on the same t grid.
      7) Apply global OKLab scaling around center to guarantee in-gamut.

    Returns:
        lab_safe      : [n_hues,3] OKLab Fourier-smoothed ring
        rgb_srgb_safe : [n_hues,3] sRGB Fourier-smoothed ring
        lab_arc       : [n_hues,3] OKLab equal-arc HSV hex samples
        rgb_arc       : [n_hues,3] sRGB equal-arc HSV hex samples
    """
    # 1) high-res HSV hex
    N_hi = n_hues * oversample
    rgb_hi = build_hsv_hex_path_srgb(N_hi)  # [N_hi,3], sRGB

    # 2) convert to OKLab
    rgb_t = torch.from_numpy(rgb_hi).to(device=device, dtype=dtype)
    lab_t = linear_to_oklab(srgb_to_linear(rgb_t))
    lab_hi = lab_t.cpu().numpy()  # [N_hi,3]

    # 3) cumulative arc length in OKLab (closed loop)
    diffs = lab_hi[(np.arange(N_hi)+1) % N_hi] - lab_hi
    seg_len = np.linalg.norm(diffs, axis=1)
    s_vertices = np.concatenate([[0.0], np.cumsum(seg_len)])  # length N_hi+1
    S = s_vertices[-1]
    t_vertices = s_vertices / S  # [0..1]

    # 4) equal-arc resampling to n_hues points
    t_out = np.linspace(0.0, 1.0, n_hues, endpoint=False)
    lab_arc = np.zeros((n_hues, 3), dtype=float)
    rgb_arc = np.zeros((n_hues, 3), dtype=float)

    lab_ext = np.vstack([lab_hi, lab_hi[0]])
    rgb_ext = np.vstack([rgb_hi, rgb_hi[0]])

    for k, tk in enumerate(t_out):
        j = np.searchsorted(t_vertices, tk, side="right") - 1
        j = np.clip(j, 0, N_hi - 1)
        t0 = t_vertices[j]
        t1 = t_vertices[j+1]
        u = 0.0 if t1 <= t0 else (tk - t0) / (t1 - t0)
        lab_arc[k] = (1 - u) * lab_ext[j] + u * lab_ext[j+1]
        rgb_arc[k] = (1 - u) * rgb_ext[j] + u * rgb_ext[j+1]

    # 5) Fourier fit on arc-length parameter t_out
    M = n_harmonics
    cols = [np.ones_like(t_out)]
    for k in range(1, M+1):
        cols.append(np.cos(2*np.pi*k*t_out))
        cols.append(np.sin(2*np.pi*k*t_out))
    X = np.stack(cols, axis=1)  # [n_hues, 1+2M]

    coeffs = []
    for c_idx in range(3):
        y = lab_arc[:, c_idx]
        beta, *_ = np.linalg.lstsq(X, y, rcond=None)
        coeffs.append(beta)
    coeffs = np.stack(coeffs, axis=0)  # [3,K]

    lab_smooth = (coeffs @ X.T).T  # [n_hues,3]

    # 6) global OKLab scaling around center to guarantee in-gamut
    lab_t2 = torch.from_numpy(lab_smooth).to(device=device, dtype=dtype)
    center = lab_t2.mean(dim=0)
    offsets = lab_t2 - center[None, :]

    def is_safe(scale: float) -> bool:
        lab_scaled = center[None, :] + scale * offsets
        rgb_lin = oklab_to_linear(lab_scaled)
        return bool(((rgb_lin >= -1e-6) & (rgb_lin <= 1.0 + 1e-6)).all())

    s_lo, s_hi = 0.0, 1.0
    if not is_safe(1.0):
        for _ in range(40):
            mid = 0.5 * (s_lo + s_hi)
            if is_safe(float(mid)):
                s_lo = mid
            else:
                s_hi = mid
        s_opt = s_lo
    else:
        s_opt = 1.0

    lab_safe = center[None, :] + s_opt * offsets
    rgb_lin_safe = oklab_to_linear(lab_safe)
    rgb_srgb_safe = linear_to_srgb(rgb_lin_safe).clamp(0.0, 1.0).cpu().numpy()

    return lab_safe.cpu().numpy(), rgb_srgb_safe, lab_arc, rgb_arc

# ============================================================
# Hue disk + flow images from a linear RGB ring
# ============================================================

def build_hue_disk_from_ring(rgb_ring_lin: np.ndarray,
                             center_lin: np.ndarray,
                             size: int = 512) -> np.ndarray:
    n_hues = rgb_ring_lin.shape[0]
    N = size
    y, x = np.mgrid[0:N, 0:N]
    x = (x + 0.5) / N * 2 - 1
    y = (y + 0.5) / N * 2 - 1
    r = np.sqrt(x * x + y * y)
    theta = np.mod(np.arctan2(y, x), 2 * np.pi)

    hue_f = theta / (2 * np.pi) * n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)

    ring0 = rgb_ring_lin[idx0]
    ring1 = rgb_ring_lin[idx1]
    ring_interp = (1 - t[..., None]) * ring0 + t[..., None] * ring1

    r_exp = r[..., None]
    rgb_lin = (1 - r_exp) * center_lin + r_exp * ring_interp
    rgb_lin[r > 1] = 1.0

    return np.clip(linear_to_srgb_np(rgb_lin), 0, 1)

def make_flow_images_for_ring(rgb_ring_lin: np.ndarray,
                              center_lin: np.ndarray,
                              flows: torch.Tensor):
    n_hues = rgb_ring_lin.shape[0]

    disk = build_hue_disk_from_ring(rgb_ring_lin, center_lin, size=512)

    u = flows[0].cpu().numpy()
    v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2 * np.pi)
    mag = np.sqrt(u * u + v * v)
    mag /= (mag.max() + 1e-8)

    hue_f = angle / (2 * np.pi) * n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)

    ring0 = rgb_ring_lin[idx0]
    ring1 = rgb_ring_lin[idx1]
    ring_interp = (1 - t[..., None]) * ring0 + t[..., None] * ring1

    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow), 0, 1)

    alpha = mag[..., None]
    white = np.ones_like(flow_rgb)
    black = np.zeros_like(flow_rgb)
    flow_white = alpha * flow_rgb + (1 - alpha) * white
    flow_black = alpha * flow_rgb + (1 - alpha) * black

    return disk, flow_rgb, flow_white, flow_black

# ============================================================
# Warped sRGB cube faces for OKLab trajectory
# ============================================================

def make_face_oklab(face: str, n: int = 96):
    t = np.linspace(0.0, 1.0, n, dtype=np.float32)
    u, v = np.meshgrid(t, t, indexing="ij")
    srgb = np.zeros((n, n, 3), dtype=np.float32)

    if   face == "R0": srgb[...,0]=0.0; srgb[...,1]=u; srgb[...,2]=v
    elif face == "R1": srgb[...,0]=1.0; srgb[...,1]=u; srgb[...,2]=v
    elif face == "G0": srgb[...,1]=0.0; srgb[...,0]=u; srgb[...,2]=v
    elif face == "G1": srgb[...,1]=1.0; srgb[...,0]=u; srgb[...,2]=v
    elif face == "B0": srgb[...,2]=0.0; srgb[...,0]=u; srgb[...,1]=v
    elif face == "B1": srgb[...,2]=1.0; srgb[...,0]=u; srgb[...,1]=v
    else:
        raise ValueError("bad face name")

    srgb_t = torch.from_numpy(srgb.reshape(-1,3)).to(device=device, dtype=dtype)
    rgb_lin = srgb_to_linear(srgb_t)
    lab = linear_to_oklab(rgb_lin).cpu().numpy().reshape(n,n,3)
    return lab[...,0], lab[...,1], lab[...,2]

faces = ["R0","R1","G0","G1","B0","B1"]
face_surfaces = {f: make_face_oklab(f, n=96) for f in faces}

# ============================================================
# 3D OKLab trajectory: Fourier-smoothed vs arc-length HSV hex vs sinebow vs HSV
# ============================================================
def plot_oklab_trajectory_hex_fourier_sine_hsv(
    rgb_fourier_srgb: np.ndarray,
    rgb_arc_srgb: np.ndarray,   # reparametrized HSV hex samples (equal arc)
    rgb_sine_srgb: np.ndarray,
    rgb_hsv_srgb: np.ndarray,
    tick_len: float = 0.003,    # length of the small orthogonal ticks in OKLab units
):
    def srgb_ring_to_lab(srgb_ring):
        rgb_lin = srgb_to_linear(torch.from_numpy(srgb_ring).to(device=device, dtype=dtype))
        lab = linear_to_oklab(rgb_lin).cpu().numpy()
        return lab

    lab_fourier = srgb_ring_to_lab(rgb_fourier_srgb)
    lab_arc     = srgb_ring_to_lab(rgb_arc_srgb)
    lab_sine    = srgb_ring_to_lab(rgb_sine_srgb)
    lab_hsv     = srgb_ring_to_lab(rgb_hsv_srgb)

    fig = go.Figure()

    # ------------------------------------------------------------------
    # warped cube faces
    # ------------------------------------------------------------------
    for f in faces:
        Lg, ag, bg = face_surfaces[f]
        fig.add_trace(go.Surface(
            x=Lg, y=ag, z=bg,
            surfacecolor=np.zeros_like(Lg),
            colorscale=[[0,"rgb(120,120,120)"],[1,"rgb(120,120,120)"]],
            showscale=False,
            opacity=0.15,
            hoverinfo="skip",
            lighting=dict(ambient=1.0, diffuse=0.0, specular=0.0),
            showlegend=False,
        ))

    # ------------------------------------------------------------------
    # cube edges
    # ------------------------------------------------------------------
    verts = np.array(list(itertools.product([0.0,1.0],[0.0,1.0],[0.0,1.0])), dtype=np.float32)
    edges = [(i,j) for i in range(8) for j in range(i+1,8)
             if np.sum(np.abs(verts[i]-verts[j])) == 1.0]
    t_edge = np.linspace(0.0,1.0,64,dtype=np.float32)
    for i,j in edges:
        p0 = verts[i]; p1 = verts[j]
        srgb_line = p0 + (p1-p0)[None,:] * t_edge[:,None]
        lab_edge = linear_to_oklab(
            srgb_to_linear(torch.from_numpy(srgb_line).to(device=device, dtype=dtype))
        ).cpu().numpy()
        fig.add_trace(go.Scatter3d(
            x=lab_edge[:,0], y=lab_edge[:,1], z=lab_edge[:,2],
            mode="lines",
            line=dict(width=3, color="rgb(128,128,128)"),
            hoverinfo="skip",
            showlegend=False,
        ))

    # ------------------------------------------------------------------
    # Fourier-smoothed HSV hex (colored curve)
    # ------------------------------------------------------------------
    fig.add_trace(go.Scatter3d(
        x=lab_fourier[:,0], y=lab_fourier[:,1], z=lab_fourier[:,2],
        mode="lines+markers",
        line=dict(width=6, color="#ffffff"),
        marker=dict(size=4, color=rgb_fourier_srgb),
        name="Fourier-smoothed HSV hex",
    ))

    # ------------------------------------------------------------------
    # Reparameterized HSV samples: **orthogonal tick marks**
    # ------------------------------------------------------------------
    # Compute tangents along lab_arc
    n = lab_arc.shape[0]
    tangents = np.zeros_like(lab_arc)
    for i in range(n):
        p_prev = lab_arc[(i-1) % n]
        p_next = lab_arc[(i+1) % n]
        tangents[i] = p_next - p_prev
    # Normalize tangents
    tangents /= np.linalg.norm(tangents, axis=1, keepdims=True) + 1e-12

    # Build normals (arbitrary but consistent) by crossing with a stable vector
    # If tangent is too close to some axis, choose another axis dynamically.
    normals = np.zeros_like(lab_arc)
    for i in range(n):
        t = tangents[i]
        # pick a base vector not parallel to t
        base = np.array([1,0,0])
        if abs(np.dot(base,t)) > 0.8:
            base = np.array([0,1,0])
        nrm = np.cross(t, base)
        nrm_n = np.linalg.norm(nrm)
        if nrm_n < 1e-12:
            # fallback
            base = np.array([0,0,1])
            nrm = np.cross(t, base)
            nrm_n = np.linalg.norm(nrm)
        nrm /= (nrm_n + 1e-12)
        normals[i] = nrm

    # Build tick segment list
    xs, ys, zs = [], [], []
    for p, nrm in zip(lab_arc, normals):
        p1 = p + tick_len * nrm
        p2 = p - tick_len * nrm
        xs.extend([p1[0], p2[0], None])
        ys.extend([p1[1], p2[1], None])
        zs.extend([p1[2], p2[2], None])

    fig.add_trace(go.Scatter3d(
        x=xs, y=ys, z=zs,
        mode="lines",
        line=dict(width=2, color="rgb(160,160,160)"),
        name="HSV hex (equal-arc ticks)",
        hoverinfo="skip",
    ))

    # ------------------------------------------------------------------
    # sinebow (colored)
    # ------------------------------------------------------------------
    fig.add_trace(go.Scatter3d(
        x=lab_sine[:,0], y=lab_sine[:,1], z=lab_sine[:,2],
        mode="lines+markers",
        line=dict(width=2, color="#888888"),
        marker=dict(size=3, color=rgb_sine_srgb),
        name="sinebow",
    ))

    # ------------------------------------------------------------------
    # HSV smooth wheel (colored)
    # ------------------------------------------------------------------
    fig.add_trace(go.Scatter3d(
        x=lab_hsv[:,0], y=lab_hsv[:,1], z=lab_hsv[:,2],
        mode="lines+markers",
        line=dict(width=2, color="#cccccc"),
        marker=dict(size=3, color=rgb_hsv_srgb),
        name="HSV",
    ))

    # ------------------------------------------------------------------
    # layout
    # ------------------------------------------------------------------
    gray = "rgb(200,200,200)"
    axis_style = dict(
        showbackground=False,
        gridcolor="rgba(200,200,200,0.25)",
        zerolinecolor="rgba(200,200,200,0.35)",
        tickfont=dict(color=gray),
    )

    fig.update_layout(
        width=1000, height=780,
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        scene=dict(
            bgcolor="rgba(0,0,0,0)",
            xaxis=dict(axis_style, title=dict(text="L", font=dict(color=gray))),
            yaxis=dict(axis_style, title=dict(text="a", font=dict(color=gray))),
            zaxis=dict(axis_style, title=dict(text="b", font=dict(color=gray))),
            aspectmode="data",
        ),
        font=dict(color=gray),
        legend=dict(
            font=dict(color=gray, size=14),
            bgcolor="rgba(0,0,0,0)"
        ),
        title=dict(
            text="OKLab: warped sRGB cube + Fourier HSV / arc-length ticks / sinebow / HSV",
            font=dict(color=gray, size=20),
        )
    )

    return HTML(pio.to_html(fig, include_plotlyjs="cdn", full_html=False))
# ============================================================
# Main: Fourier-smoothed HSV hex vs sinebow vs HSV on Omnipose
# ============================================================

def visualize_fourier_hex_vs_sinebow_hsv_on_omnipose(
    n_hues: int = 72,
    n_harmonics: int = 4,
    device_str: str = "cpu",
):
    dev = torch.device(device_str)

    # 1) base HSV wheel (smooth)
    t = np.linspace(0, 1, n_hues, endpoint=False)
    hsv_vals = np.stack([t, np.ones_like(t), np.ones_like(t)], axis=1)
    rgb_hsv_srgb = hsv_to_rgb(hsv_vals)

    # sinebow
    sine_r = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3))
    sine_g = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3))
    sine_b = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_srgb = np.stack([sine_r, sine_g, sine_b], axis=1)

    # 2) Fourier-smoothed HSV hex colormap (arc-length reparam in OKLab)
    lab_fourier, rgb_fourier_srgb, lab_arc, rgb_arc_srgb = build_fourier_hsv_hex_ring(
        n_hues=n_hues,
        n_harmonics=n_harmonics,
        oversample=8,
    )

    # 3) Trajectory plot in OKLab
    traj_html = plot_oklab_trajectory_hex_fourier_sine_hsv(
        rgb_fourier_srgb,
        rgb_arc_srgb,
        rgb_sine_srgb,
        rgb_hsv_srgb,
    )
    display(traj_html)

    # 4) Convert rings to linear RGB for disks/flows
    rgb_fourier_lin = srgb_to_linear_np(rgb_fourier_srgb)
    rgb_sine_lin    = srgb_to_linear_np(rgb_sine_srgb)
    rgb_hsv_lin     = srgb_to_linear_np(rgb_hsv_srgb)

    center_srgb = np.array([0.5, 0.5, 0.5], dtype=float)
    center_lin = srgb_to_linear_np(center_srgb)

    # 5) Load Omnipose E. coli flows
    omnidir = Path(omnirefactor.__file__).parent.parent.parent
    basedir = os.path.join(omnidir, "docs", "_static")
    name = "ecoli"
    ext = ".tif"

    image = io.imread(os.path.join(basedir, name + ext))
    masks = io.imread(os.path.join(basedir, name + "_labels" + ext))
    slc = ocdkit.measure.crop_bbox(masks > 0, pad=0)[0]
    masks = fastremap.renumber(masks[slc])[0]
    image = image[slc]

    flows_all = omnirefactor.core.masks_to_flows(masks, device="cpu")[4].to(dev)
    flows = flows_all

    # 6) Build images for each colormap
    disk_fourier, flow_fourier, flow_fourier_white, flow_fourier_black = make_flow_images_for_ring(
        rgb_fourier_lin, center_lin, flows
    )
    disk_sine, flow_sine, flow_sine_white, flow_sine_black = make_flow_images_for_ring(
        rgb_sine_lin, center_lin, flows
    )
    disk_hsv, flow_hsv, flow_hsv_white, flow_hsv_black = make_flow_images_for_ring(
        rgb_hsv_lin, center_lin, flows
    )

    # 7) Plot 3×4 grid
    fig, axes = plt.subplots(3, 4, figsize=(16, 10))
    row_titles = [
        f"Fourier-smoothed HSV hex (M={n_harmonics})",
        "sinebow",
        "HSV",
    ]

    # Row 0: Fourier-smoothed HSV hex
    axes[0,0].imshow(disk_fourier);           axes[0,0].set_title("Hue disk");                  axes[0,0].axis("off")
    axes[0,1].imshow(flow_fourier);           axes[0,1].set_title("Flow mag → chroma");         axes[0,1].axis("off")
    axes[0,2].imshow(flow_fourier_white);     axes[0,2].set_title("Flow mag → alpha (white)");  axes[0,2].axis("off")
    axes[0,3].imshow(flow_fourier_black);     axes[0,3].set_title("Flow mag → alpha (black)");  axes[0,3].axis("off")

    # Row 1: sinebow
    axes[1,0].imshow(disk_sine);              axes[1,0].axis("off")
    axes[1,1].imshow(flow_sine);              axes[1,1].axis("off")
    axes[1,2].imshow(flow_sine_white);        axes[1,2].axis("off")
    axes[1,3].imshow(flow_sine_black);        axes[1,3].axis("off")

    # Row 2: HSV
    axes[2,0].imshow(disk_hsv);               axes[2,0].axis("off")
    axes[2,1].imshow(flow_hsv);               axes[2,1].axis("off")
    axes[2,2].imshow(flow_hsv_white);         axes[2,2].axis("off")
    axes[2,3].imshow(flow_hsv_black);         axes[2,3].axis("off")

    for i, title in enumerate(row_titles):
        axes[i,0].set_ylabel(title, fontsize=12)

    plt.tight_layout()
    plt.show()

# ============================================================
# Run everything
# ============================================================

if __name__ == "__main__":
    N_HUES = 150        # discrete hues
    N_HARMONICS = 2    # Fourier components

    visualize_fourier_hex_vs_sinebow_hsv_on_omnipose(
        n_hues=N_HUES,
        n_harmonics=N_HARMONICS,
        device_str="cpu",
    )

In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import hsv_to_rgb

import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display

import omnirefactor
from omnirefactor.plot import imshow
from omnirefactor import io
import fastremap
from pathlib import Path
import os
import itertools

# ============================================================
# Global config
# ============================================================

dtype = torch.float32
device = torch.device("cpu")

# ============================================================
# NumPy sRGB <-> linear, XYZ
# ============================================================

# ============================================================
# NumPy sRGB <-> linear, XYZ  (fixed)
# ============================================================

def srgb_to_linear_np(rgb):
    """
    Robust sRGB -> linear RGB.

    - Accepts values slightly outside [0,1] (for diagnostics).
    - Avoids invalid fractional powers for negative bases by clamping
      only the high-branch base to >= 0.
    - DOES NOT globally clamp rgb to [0,1], so gamut checks remain meaningful.
    """
    rgb = np.asarray(rgb, dtype=float)
    a = 0.055
    thresh = 0.04045

    low  = rgb <= thresh
    high = ~low

    out = np.zeros_like(rgb)
    out[low]  = rgb[low] / 12.92

    base = (rgb[high] + a) / (1 + a)
    # forbid negative bases for fractional power, but allow >1
    base = np.maximum(base, 0.0)
    out[high] = base ** 2.4

    return out

def linear_to_srgb_np(rgb):
    """
    linear RGB -> sRGB (no pre-clamp; caller is responsible for gamut).
    """
    rgb = np.asarray(rgb, dtype=float)
    a = 0.055
    return np.where(
        rgb <= 0.0031308,
        12.92 * rgb,
        (1 + a) * np.power(np.clip(rgb, 0, None), 1 / 2.4) - a,
    )

# sRGB (D65) to XYZ (D65)
M_RGB_TO_XYZ_np = np.array([
    [0.4124564, 0.3575761, 0.1804375],
    [0.2126729, 0.7151522, 0.0721750],
    [0.0193339, 0.1191920, 0.9503041],
], dtype=float)
M_XYZ_TO_RGB_np = np.linalg.inv(M_RGB_TO_XYZ_np)

def srgb_to_xyz_np(srgb):
    rgb_lin = srgb_to_linear_np(srgb)
    return rgb_lin @ M_RGB_TO_XYZ_np.T

def xyz_to_srgb_np(xyz):
    rgb_lin = xyz @ M_XYZ_TO_RGB_np.T
    return linear_to_srgb_np(rgb_lin)

    
# ============================================================
# JzAzBz (Safdar et al. 2017) in NumPy
# ============================================================

# ============================================================
# Robust JzAzBz (Safdar et al. 2017) with epsilon clamps
# ============================================================

# Constants (same as before)
b_  = 1.15
g_  = 0.66
c1_ = 3424 / 2**12
c2_ = 2413 / 2**7
c3_ = 2392 / 2**7
n__ = 2610 / 2**14
p__ = 1.7 * 2523 / 2**5
d_  = -0.56
d0_ = 1.6295499532821566e-11

M1_np = np.array([
    [ 0.41478972,  0.579999,    0.0146480],
    [-0.2015100,   1.120649,    0.0531008],
    [-0.0166008,   0.264800,    0.6684799],
], dtype=float)

M2_np = np.array([
    [ 0.5,        0.5,        0.0       ],
    [ 3.524000,  -4.066708,   0.542708  ],
    [ 0.199076,   1.096799,  -1.295875  ],
], dtype=float)

M2_INV_np = np.linalg.inv(M2_np)
M1_INV_np = np.linalg.inv(M1_np)

def xyz_to_jzazbz_np(xyz):
    """
    XYZ (D65, relative) -> JzAzBz (Safdar 2017).
    With numeric clamps to avoid negative bases in fractional powers.
    """
    xyz = np.asarray(xyz, dtype=float)

    # Pre-adaptation of XYZ
    X, Y, Z = xyz[...,0], xyz[...,1], xyz[...,2]
    Xp = b_ * X - (b_ - 1.0) * Z
    Yp = g_ * Y - (g_ - 1.0) * X
    Zp = Z
    XYZ = np.stack([Xp, Yp, Zp], axis=-1)

    # Scale to 1e4 as in paper
    XYZ_4 = XYZ * 1e4
    LMS = XYZ_4 @ M1_np.T
    # Guarantee positive, nonzero LMS ratios
    LMS_ratio = np.clip(LMS / 1e4, 1e-10, None)

    # Nonlinear step with clamps
    LMS_ratio_n = np.power(LMS_ratio, n__)
    num = c1_ + c2_ * LMS_ratio_n
    den = 1.0 + c3_ * LMS_ratio_n
    ratio = np.clip(num / den, 1e-10, None)
    LMS_p = np.power(ratio, p__)

    Izazbz = LMS_p @ M2_np.T
    Iz = Izazbz[...,0]
    az = Izazbz[...,1]
    bz = Izazbz[...,2]
    Jz = ((1 + d_) * Iz) / (1 + d_ * Iz) - d0_

    return np.stack([Jz, az, bz], axis=-1)

def jzazbz_to_xyz_np(jzazbz):
    """
    JzAzBz -> XYZ (D65, relative).
    Inverse with numeric clamps to avoid invalid powers.
    """
    jzazbz = np.asarray(jzazbz, dtype=float)
    Jz = jzazbz[...,0]
    az = jzazbz[...,1]
    bz = jzazbz[...,2]

    # Invert Jz -> Iz
    Iz = (Jz + d0_) / (1 + d_ - d_ * (Jz + d0_))
    Izazbz = np.stack([Iz, az, bz], axis=-1)

    LMS_p = Izazbz @ M2_INV_np.T
    # Clamp to positive before taking 1/p__ power
    LMS_p = np.clip(LMS_p, 1e-10, None)

    LMS_c = np.power(LMS_p, 1.0 / p__)
    num = np.clip(LMS_c - c1_, 1e-10, None)
    den = np.clip(c2_ - c3_ * LMS_c, 1e-10, None)
    LMS_ratio = np.power(num / den, 1.0 / n__)
    LMS = LMS_ratio * 1e4
    XYZ_4 = LMS @ M1_INV_np.T
    XYZ = XYZ_4 / 1e4

    # Undo pre-adaptation
    Xp, Yp, Zp = XYZ[...,0], XYZ[...,1], XYZ[...,2]
    Z = Zp
    X = (Xp + (b_ - 1.0) * Z) / b_
    Y = (Yp + (g_ - 1.0) * X) / g_

    return np.stack([X, Y, Z], axis=-1)

# ============================================================
# HSV hexagon path in sRGB (RGB + CMY corners)
# ============================================================

def build_hsv_hex_path_srgb(n_samples: int) -> np.ndarray:
    """
    Classic HSV path: R -> Y -> G -> C -> B -> M -> R.
    Returns [n_samples,3] sRGB.
    """
    t = np.linspace(0, 1, n_samples, endpoint=False)
    hsv_vals = np.stack([t, np.ones_like(t), np.ones_like(t)], axis=1)
    rgb = hsv_to_rgb(hsv_vals)
    return rgb

# ============================================================
# Fourier smoothing in JzAzBz with arc-length reparameterization
# ============================================================
def build_fourier_hsv_hex_ring_jz(
    n_hues: int,
    n_harmonics: int,
    oversample: int = 8
) -> (np.ndarray, np.ndarray, np.ndarray, np.ndarray):
    """
    Build a Fourier-smoothed cyclic colormap in JzAzBz:

      1) HSV hexagon path in sRGB with N_hi = n_hues * oversample.
      2) Convert to JzAzBz.
      3) Reparameterize by cumulative arc length in JzAzBz.
      4) Resample to n_hues points at equal arc steps (jz_arc, rgb_arc).
      5) Fit truncated Fourier series Jz/az/bz(t) with n_harmonics.
      6) Evaluate smooth Jz/az/bz on the same t grid.
      7) Shift Jz so that mean(Jz) ≈ 0.5.
      8) Apply global JzAzBz scaling around center to guarantee in-gamut.

    Returns:
        jz_safe       : [n_hues,3] JzAzBz Fourier-smoothed ring (after shift+scale)
        rgb_srgb_safe : [n_hues,3] sRGB ring for that path
        jz_arc        : [n_hues,3] JzAzBz equal-arc HSV hex samples
        rgb_arc       : [n_hues,3] sRGB equal-arc HSV hex samples
    """
    # 1) high-res HSV hex in sRGB
    N_hi = n_hues * oversample
    rgb_hi = build_hsv_hex_path_srgb(N_hi)  # [N_hi,3], sRGB

    # 2) convert to JzAzBz
    xyz_hi = srgb_to_xyz_np(rgb_hi)
    jz_hi  = xyz_to_jzazbz_np(xyz_hi)       # [N_hi,3]

    # 3) cumulative arc length in JzAzBz (closed loop)
    diffs    = jz_hi[(np.arange(N_hi) + 1) % N_hi] - jz_hi
    seg_len  = np.linalg.norm(diffs, axis=1)
    s_vertices = np.concatenate([[0.0], np.cumsum(seg_len)])
    S        = s_vertices[-1]
    t_vertices = s_vertices / S             # parameter in [0,1]

    # 4) equal-arc resampling to n_hues points
    t_out = np.linspace(0.0, 1.0, n_hues, endpoint=False)
    jz_arc  = np.zeros((n_hues, 3), dtype=float)
    rgb_arc = np.zeros((n_hues, 3), dtype=float)

    jz_ext  = np.vstack([jz_hi,  jz_hi[0]])
    rgb_ext = np.vstack([rgb_hi, rgb_hi[0]])

    for k, tk in enumerate(t_out):
        j = np.searchsorted(t_vertices, tk, side="right") - 1
        j = np.clip(j, 0, N_hi - 1)
        t0 = t_vertices[j]
        t1 = t_vertices[j+1]
        u  = 0.0 if t1 <= t0 else (tk - t0) / (t1 - t0)
        jz_arc[k]  = (1.0 - u) * jz_ext[j]  + u * jz_ext[j+1]
        rgb_arc[k] = (1.0 - u) * rgb_ext[j] + u * rgb_ext[j+1]

    # 5) Fourier fit on arc-length parameter t_out
    M = n_harmonics
    cols = [np.ones_like(t_out)]
    for k in range(1, M+1):
        cols.append(np.cos(2*np.pi*k*t_out))
        cols.append(np.sin(2*np.pi*k*t_out))
    X = np.stack(cols, axis=1)  # [n_hues, 1+2M]

    coeffs = []
    for c_idx in range(3):
        y = jz_arc[:, c_idx]
        beta, *_ = np.linalg.lstsq(X, y, rcond=None)
        coeffs.append(beta)
    coeffs = np.stack(coeffs, axis=0)       # [3, 1+2M]

    jz_smooth = (coeffs @ X.T).T            # [n_hues,3]

    # 6) shift Jz so mean luminance ~ 0.5
    mean_J = jz_smooth[:, 0].mean()
    # print('jj',mean_J)
    # delta  = 0.5 - mean_J
    # jz_smooth[:, 0] += delta

    # 7) global JzAzBz scaling around center to guarantee in-gamut
    jz_t   = torch.from_numpy(jz_smooth).to(device=device, dtype=dtype)
    center = jz_t.mean(dim=0)
    offsets = jz_t - center[None, :]

    def is_safe(scale: float) -> bool:
        jz_scaled = center[None, :] + scale * offsets
        jz_np = jz_scaled.cpu().numpy()
        xyz   = jzazbz_to_xyz_np(jz_np)
        srgb  = xyz_to_srgb_np(xyz)
        # IMPORTANT: check sRGB bounds directly, not after linearization
        return np.all((srgb >= -1e-6) & (srgb <= 1.0 + 1e-6))

    s_lo, s_hi = 0.0, 1.0
    if not is_safe(1.0):
        for _ in range(40):
            mid = 0.5 * (s_lo + s_hi)
            if is_safe(float(mid)):
                s_lo = mid
            else:
                s_hi = mid
        s_opt = s_lo
    else:
        s_opt = 1.0

    jz_safe_t = center[None, :] + s_opt * offsets
    jz_safe   = jz_safe_t.cpu().numpy()
    xyz_safe  = jzazbz_to_xyz_np(jz_safe)
    rgb_srgb_safe = np.clip(xyz_to_srgb_np(xyz_safe), 0, 1)

    return jz_safe, rgb_srgb_safe, jz_arc, rgb_arc

# ============================================================
# Hue disk + flow images from a linear RGB ring
# ============================================================

def build_hue_disk_from_ring(rgb_ring_lin: np.ndarray,
                             center_lin: np.ndarray,
                             size: int = 512) -> np.ndarray:
    n_hues = rgb_ring_lin.shape[0]
    N = size
    y, x = np.mgrid[0:N, 0:N]
    x = (x + 0.5) / N * 2 - 1
    y = (y + 0.5) / N * 2 - 1
    r = np.sqrt(x * x + y * y)
    theta = np.mod(np.arctan2(y, x), 2 * np.pi)

    hue_f = theta / (2 * np.pi) * n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)

    ring0 = rgb_ring_lin[idx0]
    ring1 = rgb_ring_lin[idx1]
    ring_interp = (1 - t[..., None]) * ring0 + t[..., None] * ring1

    r_exp = r[..., None]
    rgb_lin = (1 - r_exp) * center_lin + r_exp * ring_interp
    rgb_lin[r > 1] = 1.0

    return np.clip(linear_to_srgb_np(rgb_lin), 0, 1)

def make_flow_images_for_ring(rgb_ring_lin: np.ndarray,
                              center_lin: np.ndarray,
                              flows: torch.Tensor):
    n_hues = rgb_ring_lin.shape[0]

    disk = build_hue_disk_from_ring(rgb_ring_lin, center_lin, size=512)

    u = flows[0].cpu().numpy()
    v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2 * np.pi)
    mag = np.sqrt(u * u + v * v)
    mag /= (mag.max() + 1e-8)

    hue_f = angle / (2 * np.pi) * n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)

    ring0 = rgb_ring_lin[idx0]
    ring1 = rgb_ring_lin[idx1]
    ring_interp = (1 - t[..., None]) * ring0 + t[..., None] * ring1

    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow), 0, 1)

    alpha = mag[..., None]
    white = np.ones_like(flow_rgb)
    black = np.zeros_like(flow_rgb)
    flow_white = alpha * flow_rgb + (1 - alpha) * white
    flow_black = alpha * flow_rgb + (1 - alpha) * black

    return disk, flow_rgb, flow_white, flow_black

# ============================================================
# Warped sRGB cube faces in JzAzBz for trajectory plot
# ============================================================

def make_face_jz(face: str, n: int = 64):
    t = np.linspace(0.0, 1.0, n, dtype=float)
    u, v = np.meshgrid(t, t, indexing="ij")
    srgb = np.zeros((n, n, 3), dtype=float)

    if   face == "R0": srgb[...,0]=0.0; srgb[...,1]=u; srgb[...,2]=v
    elif face == "R1": srgb[...,0]=1.0; srgb[...,1]=u; srgb[...,2]=v
    elif face == "G0": srgb[...,1]=0.0; srgb[...,0]=u; srgb[...,2]=v
    elif face == "G1": srgb[...,1]=1.0; srgb[...,0]=u; srgb[...,2]=v
    elif face == "B0": srgb[...,2]=0.0; srgb[...,0]=u; srgb[...,1]=v
    elif face == "B1": srgb[...,2]=1.0; srgb[...,0]=u; srgb[...,1]=v
    else:
        raise ValueError("bad face name")

    xyz = srgb_to_xyz_np(srgb.reshape(-1,3)).reshape(n,n,3)
    jz = xyz_to_jzazbz_np(xyz.reshape(-1,3)).reshape(n,n,3)
    Jg = jz[...,0]
    Az = jz[...,1]
    Bz = jz[...,2]
    return Jg, Az, Bz

faces = ["R0","R1","G0","G1","B0","B1"]
face_surfaces_jz = {f: make_face_jz(f, n=64) for f in faces}

# ============================================================
# 3D JzAzBz trajectory: Fourier-smoothed vs arc-length HSV hex vs sinebow vs HSV
# ============================================================

def plot_jz_trajectory_hex_fourier_sine_hsv(
    jz_fourier: np.ndarray,
    rgb_fourier_srgb: np.ndarray,
    jz_arc: np.ndarray,
    rgb_arc_srgb: np.ndarray,
    jz_sine: np.ndarray,
    rgb_sine_srgb: np.ndarray,
    jz_hsv: np.ndarray,
    rgb_hsv_srgb: np.ndarray,
    tick_len: float = 0.003,
):
    fig = go.Figure()

    # warped cube faces in JzAzBz
    for f in faces:
        Jg, Az, Bz = face_surfaces_jz[f]
        fig.add_trace(go.Surface(
            x=Jg, y=Az, z=Bz,
            surfacecolor=np.zeros_like(Jg),
            colorscale=[[0,"rgb(120,120,120)"],[1,"rgb(120,120,120)"]],
            showscale=False,
            opacity=0.15,
            hoverinfo="skip",
            lighting=dict(ambient=1.0, diffuse=0.0, specular=0.0),
            showlegend=False,
        ))

    # cube edges in JzAzBz
    verts = np.array(list(itertools.product([0.0,1.0],[0.0,1.0],[0.0,1.0])), dtype=float)
    edges = [(i,j) for i in range(8) for j in range(i+1,8)
             if np.sum(np.abs(verts[i]-verts[j])) == 1.0]
    t_edge = np.linspace(0.0,1.0,64)
    for i,j in edges:
        p0 = verts[i]; p1 = verts[j]
        srgb_line = p0 + (p1-p0)[None,:] * t_edge[:,None]
        xyz_line = srgb_to_xyz_np(srgb_line)
        jz_line = xyz_to_jzazbz_np(xyz_line)
        fig.add_trace(go.Scatter3d(
            x=jz_line[:,0], y=jz_line[:,1], z=jz_line[:,2],
            mode="lines",
            line=dict(width=3, color="rgb(128,128,128)"),
            hoverinfo="skip",
            showlegend=False,
        ))

    # Fourier-smoothed HSV hex (colored along curve)
    fig.add_trace(go.Scatter3d(
        x=jz_fourier[:,0], y=jz_fourier[:,1], z=jz_fourier[:,2],
        mode="lines+markers",
        line=dict(width=6, color="#ffffff"),
        marker=dict(size=4, color=rgb_fourier_srgb),
        name="Fourier-smoothed HSV hex (Jz)",
    ))

    # Arc-length reparametrized HSV hex samples as small orthogonal gray ticks
    n = jz_arc.shape[0]
    tangents = np.zeros_like(jz_arc)
    for i in range(n):
        p_prev = jz_arc[(i-1) % n]
        p_next = jz_arc[(i+1) % n]
        tangents[i] = p_next - p_prev
    tangents /= np.linalg.norm(tangents, axis=1, keepdims=True) + 1e-12

    normals = np.zeros_like(jz_arc)
    for i in range(n):
        t = tangents[i]
        base = np.array([1.0,0.0,0.0])
        if abs(np.dot(base,t)) > 0.8:
            base = np.array([0.0,1.0,0.0])
        nrm = np.cross(t, base)
        nrm_n = np.linalg.norm(nrm)
        if nrm_n < 1e-12:
            base = np.array([0.0,0.0,1.0])
            nrm = np.cross(t, base)
            nrm_n = np.linalg.norm(nrm)
        normals[i] = nrm / (nrm_n + 1e-12)

    xs, ys, zs = [], [], []
    for p, nrm in zip(jz_arc, normals):
        p1 = p + tick_len * nrm
        p2 = p - tick_len * nrm
        xs.extend([p1[0], p2[0], None])
        ys.extend([p1[1], p2[1], None])
        zs.extend([p1[2], p2[2], None])

    fig.add_trace(go.Scatter3d(
        x=xs, y=ys, z=zs,
        mode="lines",
        line=dict(width=2, color="rgb(160,160,160)"),
        name="HSV hex (equal-arc ticks, Jz)",
        hoverinfo="skip",
    ))

    # sinebow, colored along Jz curve
    fig.add_trace(go.Scatter3d(
        x=jz_sine[:,0], y=jz_sine[:,1], z=jz_sine[:,2],
        mode="lines+markers",
        line=dict(width=2, color="#888888"),
        marker=dict(size=3, color=rgb_sine_srgb),
        name="sinebow",
    ))

    # HSV smooth wheel, colored along curve
    fig.add_trace(go.Scatter3d(
        x=jz_hsv[:,0], y=jz_hsv[:,1], z=jz_hsv[:,2],
        mode="lines+markers",
        line=dict(width=2, color="#cccccc"),
        marker=dict(size=3, color=rgb_hsv_srgb),
        name="HSV",
    ))

    gray = "rgb(200,200,200)"
    axis_style = dict(
        showbackground=False,
        gridcolor="rgba(200,200,200,0.25)",
        zerolinecolor="rgba(200,200,200,0.35)",
        tickfont=dict(color=gray),
    )

    fig.update_layout(
        width=1000, height=780,
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        scene=dict(
            bgcolor="rgba(0,0,0,0)",
            xaxis=dict(axis_style, title=dict(text="Jz", font=dict(color=gray))),
            yaxis=dict(axis_style, title=dict(text="az", font=dict(color=gray))),
            zaxis=dict(axis_style, title=dict(text="bz", font=dict(color=gray))),
            aspectmode="data",
        ),
        font=dict(color=gray),
        legend=dict(
            font=dict(color=gray, size=14),
            bgcolor="rgba(0,0,0,0)"
        ),
        title=dict(
            text="JzAzBz: warped sRGB cube + Fourier HSV / equal-arc ticks / sinebow / HSV",
            font=dict(color=gray, size=20),
        )
    )

    return HTML(pio.to_html(fig, include_plotlyjs="cdn", full_html=False))

def shift_Jz_to_mean_0p5(jz_ring: np.ndarray) -> np.ndarray:
    """
    Shift the Jz component of a JzAzBz ring so that its mean luminance is 0.5.
    
    jz_ring: [N,3] Jz, az, bz
    Returns: shifted_ring [N,3]
    """
    jz = jz_ring[:,0]
    mean_jz = np.mean(jz)
    delta = 0.5 - mean_jz

    jz_shifted = jz + delta
    shifted_ring = np.stack([jz_shifted, jz_ring[:,1], jz_ring[:,2]], axis=1)
    return shifted_ring

# ============================================================
# Main: Fourier-smoothed HSV hex vs sinebow vs HSV on Omnipose (JzAzBz colormap)
# ============================================================

def find_reddest_index(rgb_ring_srgb: np.ndarray) -> int:
    """
    Return the index of the reddest color in an sRGB ring.
    Use red-dominance metric:  R - 0.5*(G+B).
    """
    rgb = np.asarray(rgb_ring_srgb)
    r = rgb[:,0]
    g = rgb[:,1]
    b = rgb[:,2]
    score = r - 0.5*(g + b)
    return int(np.argmax(score))


def find_greenest_index(rgb_ring_srgb: np.ndarray) -> int:
    """
    Return the index of the greenest color in an sRGB ring.
    Use green-dominance metric: G - 0.5*(R+B).
    """
    rgb = np.asarray(rgb_ring_srgb)
    r = rgb[:,0]
    g = rgb[:,1]
    b = rgb[:,2]
    score = g - 0.5*(r + b)
    return int(np.argmax(score))


def align_ring_to_reference(rgb_ring_srgb: np.ndarray,
                            ref_green_idx: int) -> np.ndarray:
    """
    Align an sRGB ring to a canonical reference where:
      - reddest color is at index 0
      - greenest color lies in the same direction as ref_green_idx.

    Steps:
      1) Roll so reddest index -> 0.
      2) Compute green index in this orientation (g1).
      3) Also consider reversed orientation (flip), compute g2.
      4) Choose orientation (original vs reversed) whose green index
         is closer (modulo wrapping) to ref_green_idx.

    Returns aligned ring.
    """
    N = rgb_ring_srgb.shape[0]
    ring = np.asarray(rgb_ring_srgb)

    # Roll reddest to 0
    idx_red = find_reddest_index(ring)
    ring_r0 = np.roll(ring, -idx_red, axis=0)

    # orientation 1: as-is
    g1 = find_greenest_index(ring_r0)

    # orientation 2: reversed (then re-roll reddest back to 0 implicitly)
    ring_rev = ring_r0[::-1].copy()
    g2 = find_greenest_index(ring_rev)

    def circ_dist(a, b):
        d = abs(a - b)
        return min(d, N - d)

    if circ_dist(g1, ref_green_idx) <= circ_dist(g2, ref_green_idx):
        return ring_r0
    else:
        return ring_rev


def visualize_fourier_hex_vs_sinebow_hsv_on_omnipose_jz(
    n_hues: int = 72,
    n_harmonics: int = 4,
    device_str: str = "cpu",
):
    dev = torch.device(device_str)

    # 1) base HSV wheel (smooth) in sRGB
    t = np.linspace(0, 1, n_hues, endpoint=False)
    hsv_vals = np.stack([t, np.ones_like(t), np.ones_like(t)], axis=1)
    rgb_hsv_srgb_raw = hsv_to_rgb(hsv_vals)
    xyz_hsv_raw = srgb_to_xyz_np(rgb_hsv_srgb_raw)
    jz_hsv_raw = xyz_to_jzazbz_np(xyz_hsv_raw)

    # sinebow in sRGB
    sine_r = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3))
    sine_g = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3))
    sine_b = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_srgb_raw = np.stack([sine_r, sine_g, sine_b], axis=1)
    xyz_sine_raw = srgb_to_xyz_np(rgb_sine_srgb_raw)
    jz_sine_raw = xyz_to_jzazbz_np(xyz_sine_raw)

    # 2) Fourier-smoothed HSV hex colormap in JzAzBz (use only sRGB and arc)
    jz_fourier_raw, rgb_fourier_srgb_raw, jz_arc_raw, rgb_arc_srgb_raw = build_fourier_hsv_hex_ring_jz(
        n_hues=n_hues,
        n_harmonics=n_harmonics,
        oversample=8,
    )

    # ------------------------------------------------------------
    # 3) Canonical alignment: use HSV as reference
    #    Step 1: align HSV ring to itself (red at 0, green forward)
    # ------------------------------------------------------------
    idx_red_h = find_reddest_index(rgb_hsv_srgb_raw)
    rgb_hsv_r0 = np.roll(rgb_hsv_srgb_raw, -idx_red_h, axis=0)
    g_h = find_greenest_index(rgb_hsv_r0)
    # Ensure green is in the "forward" half (0..N/2)
    if g_h > n_hues // 2:
        rgb_hsv_r0 = rgb_hsv_r0[::-1].copy()
        g_h = find_greenest_index(rgb_hsv_r0)
    rgb_hsv_srgb = rgb_hsv_r0
    jz_hsv = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_hsv_srgb))

    # Align arc-length HSV samples (rgb_arc_srgb_raw) to this same orientation
    rgb_arc_r0 = np.roll(rgb_arc_srgb_raw, -idx_red_h, axis=0)
    if g_h > n_hues // 2:
        rgb_arc_r0 = rgb_arc_r0[::-1].copy()
    rgb_arc_srgb = rgb_arc_r0
    jz_arc = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_arc_srgb))

    # ------------------------------------------------------------
    # 4) Align Fourier and sinebow to the same red/green phase as HSV
    # ------------------------------------------------------------
    rgb_fourier_srgb = align_ring_to_reference(rgb_fourier_srgb_raw, ref_green_idx=g_h)
    rgb_sine_srgb    = align_ring_to_reference(rgb_sine_srgb_raw,    ref_green_idx=g_h)

    jz_fourier = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_fourier_srgb))
    jz_sine    = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_sine_srgb))

    # ------------------------------------------------------------
    # 5) Trajectory plot in JzAzBz (using aligned rings)
    # ------------------------------------------------------------
    traj_html = plot_jz_trajectory_hex_fourier_sine_hsv(
        jz_fourier, rgb_fourier_srgb,
        jz_arc, rgb_arc_srgb,
        jz_sine, rgb_sine_srgb,
        jz_hsv, rgb_hsv_srgb,
    )
    display(traj_html)

    # ------------------------------------------------------------
    # 6) Convert aligned rings to linear RGB for disks/flows
    # ------------------------------------------------------------
    rgb_fourier_lin = srgb_to_linear_np(rgb_fourier_srgb)
    rgb_sine_lin    = srgb_to_linear_np(rgb_sine_srgb)
    rgb_hsv_lin     = srgb_to_linear_np(rgb_hsv_srgb)

    center_srgb = np.array([0.5, 0.5, 0.5], dtype=float)
    center_lin = srgb_to_linear_np(center_srgb)

    # 7) Load Omnipose E. coli flows
    omnidir = Path(omnirefactor.__file__).parent.parent.parent
    basedir = os.path.join(omnidir, "docs", "_static")
    name = "ecoli"
    ext = ".tif"

    image = io.imread(os.path.join(basedir, name + ext))
    masks = io.imread(os.path.join(basedir, name + "_labels" + ext))
    slc = ocdkit.measure.crop_bbox(masks > 0, pad=0)[0]
    masks = fastremap.renumber(masks[slc])[0]
    image = image[slc]

    flows_all = omnirefactor.core.masks_to_flows(masks, device="cpu")[4].to(dev)
    flows = flows_all

    # 8) Build images for each colormap (phase 0)
    disk_fourier, flow_fourier, flow_fourier_white, flow_fourier_black = make_flow_images_for_ring(
        rgb_fourier_lin, center_lin, flows
    )
    disk_sine, flow_sine, flow_sine_white, flow_sine_black = make_flow_images_for_ring(
        rgb_sine_lin, center_lin, flows
    )
    disk_hsv, flow_hsv, flow_hsv_white, flow_hsv_black = make_flow_images_for_ring(
        rgb_hsv_lin, center_lin, flows
    )

    # 9) Build rotated (+90°) colormaps by rolling hue index by 1/4 turn
    shift = n_hues // 4

    rgb_fourier_lin_rot = np.roll(rgb_fourier_lin, shift, axis=0)
    rgb_sine_lin_rot    = np.roll(rgb_sine_lin,    shift, axis=0)
    rgb_hsv_lin_rot     = np.roll(rgb_hsv_lin,     shift, axis=0)

    # Rotated white/black composites
    _, _, flow_fourier_white_rot, flow_fourier_black_rot = make_flow_images_for_ring(
        rgb_fourier_lin_rot, center_lin, flows
    )
    _, _, flow_sine_white_rot, flow_sine_black_rot = make_flow_images_for_ring(
        rgb_sine_lin_rot, center_lin, flows
    )
    _, _, flow_hsv_white_rot, flow_hsv_black_rot = make_flow_images_for_ring(
        rgb_hsv_lin_rot, center_lin, flows
    )

    # ============================================================
    # 10) Single unified figure: 3 rows × 6 columns
    # Each row: [disk | chroma | 0° white | 0° black | +90° white | +90° black]
    # ============================================================

    fig, axes = plt.subplots(3, 6, figsize=(26, 8))
    row_titles = [
        f"Fourier-smoothed HSV hex (Jz, M={n_harmonics})",
        "sinebow",
        "HSV",
    ]

    # Row 0: Fourier-Jz
    axes[0,0].imshow(disk_fourier);           axes[0,0].axis("off")
    axes[0,1].imshow(flow_fourier);           axes[0,1].axis("off")
    axes[0,2].imshow(flow_fourier_white);     axes[0,2].axis("off")
    axes[0,3].imshow(flow_fourier_black);     axes[0,3].axis("off")
    axes[0,4].imshow(flow_fourier_white_rot); axes[0,4].axis("off")
    axes[0,5].imshow(flow_fourier_black_rot); axes[0,5].axis("off")

    # Row 1: sinebow
    axes[1,0].imshow(disk_sine);              axes[1,0].axis("off")
    axes[1,1].imshow(flow_sine);              axes[1,1].axis("off")
    axes[1,2].imshow(flow_sine_white);        axes[1,2].axis("off")
    axes[1,3].imshow(flow_sine_black);        axes[1,3].axis("off")
    axes[1,4].imshow(flow_sine_white_rot);    axes[1,4].axis("off")
    axes[1,5].imshow(flow_sine_black_rot);    axes[1,5].axis("off")

    # Row 2: HSV
    axes[2,0].imshow(disk_hsv);               axes[2,0].axis("off")
    axes[2,1].imshow(flow_hsv);               axes[2,1].axis("off")
    axes[2,2].imshow(flow_hsv_white);         axes[2,2].axis("off")
    axes[2,3].imshow(flow_hsv_black);         axes[2,3].axis("off")
    axes[2,4].imshow(flow_hsv_white_rot);     axes[2,4].axis("off")
    axes[2,5].imshow(flow_hsv_black_rot);     axes[2,5].axis("off")

    # Row labels
    for i, title in enumerate(row_titles):
        axes[i,0].set_ylabel(title, fontsize=12)

    # Column labels
    axes[0,0].set_title("Hue disk (0°)")
    axes[0,1].set_title("Flow → chroma (0°)")
    axes[0,2].set_title("Alpha on white (0°)")
    axes[0,3].set_title("Alpha on black (0°)")
    axes[0,4].set_title("Alpha on white (+90°)")
    axes[0,5].set_title("Alpha on black (+90°)")

    plt.tight_layout()
    plt.show()

# ============================================================
# Run everything
# ============================================================

if __name__ == "__main__":
    N_HUES = 72        # discrete hues
    N_HARMONICS = 1    # Fourier components

    visualize_fourier_hex_vs_sinebow_hsv_on_omnipose_jz(
        n_hues=N_HUES,
        n_harmonics=N_HARMONICS,
        device_str="cpu",
    )

In [ ]:
if the ellipse is centered on gray, then this moight satisfy the deltaE condiiton by default and might be why it looks so nice

In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import hsv_to_rgb

import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display

import omnirefactor
from omnirefactor.plot import imshow
from omnirefactor import io
import fastremap
from pathlib import Path
import os
import itertools

# ============================================================
# Global config
# ============================================================

dtype = torch.float32
device = torch.device("cpu")

# ============================================================
# NumPy sRGB <-> linear, XYZ  (robust)
# ============================================================

def srgb_to_linear_np(rgb):
    """
    Robust sRGB -> linear RGB.
    Allows values slightly outside [0,1] but avoids invalid powers.
    """
    rgb = np.asarray(rgb, dtype=float)
    a = 0.055
    thresh = 0.04045

    low  = rgb <= thresh
    high = ~low

    out = np.zeros_like(rgb)
    out[low]  = rgb[low] / 12.92

    base = (rgb[high] + a) / (1 + a)
    base = np.maximum(base, 0.0)
    out[high] = base ** 2.4

    return out

def linear_to_srgb_np(rgb):
    """
    linear RGB -> sRGB (no gamut clamp).
    """
    rgb = np.asarray(rgb, dtype=float)
    a = 0.055
    return np.where(
        rgb <= 0.0031308,
        12.92 * rgb,
        (1 + a) * np.power(np.clip(rgb, 0, None), 1 / 2.4) - a,
    )

M_RGB_TO_XYZ_np = np.array([
    [0.4124564, 0.3575761, 0.1804375],
    [0.2126729, 0.7151522, 0.0721750],
    [0.0193339, 0.1191920, 0.9503041],
], dtype=float)
M_XYZ_TO_RGB_np = np.linalg.inv(M_RGB_TO_XYZ_np)

def srgb_to_xyz_np(srgb):
    rgb_lin = srgb_to_linear_np(srgb)
    return rgb_lin @ M_RGB_TO_XYZ_np.T

def xyz_to_srgb_np(xyz):
    rgb_lin = xyz @ M_XYZ_TO_RGB_np.T
    return linear_to_srgb_np(rgb_lin)

# ============================================================
# JzAzBz (Safdar) with clamps
# ============================================================

b_  = 1.15
g_  = 0.66
c1_ = 3424 / 2**12
c2_ = 2413 / 2**7
c3_ = 2392 / 2**7
n__ = 2610 / 2**14
p__ = 1.7 * 2523 / 2**5
d_  = -0.56
d0_ = 1.6295499532821566e-11

M1_np = np.array([
    [ 0.41478972,  0.579999,    0.0146480],
    [-0.2015100,   1.120649,    0.0531008],
    [-0.0166008,   0.264800,    0.6684799],
], dtype=float)

M2_np = np.array([
    [ 0.5,        0.5,        0.0       ],
    [ 3.524000,  -4.066708,   0.542708  ],
    [ 0.199076,   1.096799,  -1.295875  ],
], dtype=float)

M2_INV_np = np.linalg.inv(M2_np)
M1_INV_np = np.linalg.inv(M1_np)

def xyz_to_jzazbz_np(xyz):
    xyz = np.asarray(xyz, dtype=float)
    X, Y, Z = xyz[...,0], xyz[...,1], xyz[...,2]
    Xp = b_ * X - (b_ - 1.0) * Z
    Yp = g_ * Y - (g_ - 1.0) * X
    Zp = Z
    XYZ = np.stack([Xp, Yp, Zp], axis=-1)

    XYZ_4 = XYZ * 1e4
    LMS = XYZ_4 @ M1_np.T
    LMS_ratio = np.clip(LMS / 1e4, 1e-10, None)

    LMS_ratio_n = np.power(LMS_ratio, n__)
    num = c1_ + c2_ * LMS_ratio_n
    den = 1.0 + c3_ * LMS_ratio_n
    ratio = np.clip(num / den, 1e-10, None)
    LMS_p = np.power(ratio, p__)

    Izazbz = LMS_p @ M2_np.T
    Iz = Izazbz[...,0]
    az = Izazbz[...,1]
    bz = Izazbz[...,2]
    Jz = ((1 + d_) * Iz) / (1 + d_ * Iz) - d0_
    return np.stack([Jz, az, bz], axis=-1)

def jzazbz_to_xyz_np(jzazbz):
    jzazbz = np.asarray(jzazbz, dtype=float)
    Jz = jzazbz[...,0]
    az = jzazbz[...,1]
    bz = jzazbz[...,2]

    Iz = (Jz + d0_) / (1 + d_ - d_ * (Jz + d0_))
    Izazbz = np.stack([Iz, az, bz], axis=-1)

    LMS_p = Izazbz @ M2_INV_np.T
    LMS_p = np.clip(LMS_p, 1e-10, None)

    LMS_c = np.power(LMS_p, 1.0 / p__)
    num = np.clip(LMS_c - c1_, 1e-10, None)
    den = np.clip(c2_ - c3_ * LMS_c, 1e-10, None)
    LMS_ratio = np.power(num / den, 1.0 / n__)
    LMS = LMS_ratio * 1e4
    XYZ_4 = LMS @ M1_INV_np.T
    XYZ = XYZ_4 / 1e4

    Xp, Yp, Zp = XYZ[...,0], XYZ[...,1], XYZ[...,2]
    Z = Zp
    X = (Xp + (b_ - 1.0) * Z) / b_
    Y = (Yp + (g_ - 1.0) * X) / g_
    return np.stack([X, Y, Z], axis=-1)

# ============================================================
# HSV hexagon path in sRGB (RGB + CMY corners)
# ============================================================

def build_hsv_hex_path_srgb(n_samples: int) -> np.ndarray:
    t = np.linspace(0, 1, n_samples, endpoint=False)
    hsv_vals = np.stack([t, np.ones_like(t), np.ones_like(t)], axis=1)
    rgb = hsv_to_rgb(hsv_vals)
    return rgb

# ============================================================
# Fourier smoothing in JzAzBz with arc-length reparameterization
# ============================================================

def build_fourier_hsv_hex_ring_jz(
    n_hues: int,
    n_harmonics: int,
    oversample: int = 8
) -> (np.ndarray, np.ndarray, np.ndarray, np.ndarray):
    N_hi = n_hues * oversample
    rgb_hi = build_hsv_hex_path_srgb(N_hi)

    xyz_hi = srgb_to_xyz_np(rgb_hi)
    jz_hi  = xyz_to_jzazbz_np(xyz_hi)  # [N_hi,3]

    diffs    = jz_hi[(np.arange(N_hi) + 1) % N_hi] - jz_hi
    seg_len  = np.linalg.norm(diffs, axis=1)
    s_vertices = np.concatenate([[0.0], np.cumsum(seg_len)])
    S        = s_vertices[-1]
    t_vertices = s_vertices / S

    t_out = np.linspace(0.0, 1.0, n_hues, endpoint=False)
    jz_arc  = np.zeros((n_hues, 3), dtype=float)
    rgb_arc = np.zeros((n_hues, 3), dtype=float)

    jz_ext  = np.vstack([jz_hi,  jz_hi[0]])
    rgb_ext = np.vstack([rgb_hi, rgb_hi[0]])

    for k, tk in enumerate(t_out):
        j = np.searchsorted(t_vertices, tk, side="right") - 1
        j = np.clip(j, 0, N_hi - 1)
        t0 = t_vertices[j]
        t1 = t_vertices[j+1]
        u  = 0.0 if t1 <= t0 else (tk - t0) / (t1 - t0)
        jz_arc[k]  = (1.0 - u) * jz_ext[j]  + u * jz_ext[j+1]
        rgb_arc[k] = (1.0 - u) * rgb_ext[j] + u * rgb_ext[j+1]

    M = n_harmonics
    cols = [np.ones_like(t_out)]
    for k in range(1, M+1):
        cols.append(np.cos(2*np.pi*k*t_out))
        cols.append(np.sin(2*np.pi*k*t_out))
    X = np.stack(cols, axis=1)

    coeffs = []
    for c_idx in range(3):
        y = jz_arc[:, c_idx]
        beta, *_ = np.linalg.lstsq(X, y, rcond=None)
        coeffs.append(beta)
    coeffs = np.stack(coeffs, axis=0)

    jz_smooth = (coeffs @ X.T).T  # [n_hues,3]

    jz_t   = torch.from_numpy(jz_smooth).to(device=device, dtype=dtype)
    center = jz_t.mean(dim=0)
    offsets = jz_t - center[None, :]

    def is_safe(scale: float) -> bool:
        jz_scaled = center[None,:] + scale * offsets
        xyz = jzazbz_to_xyz_np(jz_scaled.cpu().numpy())
        srgb = xyz_to_srgb_np(xyz)
        rgb_lin = srgb_to_linear_np(srgb)
        return np.all((rgb_lin >= -1e-6) & (rgb_lin <= 1.0 + 1e-6))

    s_lo, s_hi = 0.0, 1.0
    if not is_safe(1.0):
        for _ in range(40):
            mid = 0.5*(s_lo + s_hi)
            if is_safe(float(mid)):
                s_lo = mid
            else:
                s_hi = mid
        s_opt = s_lo
    else:
        s_opt = 1.0

    jz_safe_t = center[None,:] + s_opt * offsets
    jz_safe   = jz_safe_t.cpu().numpy()
    xyz_safe  = jzazbz_to_xyz_np(jz_safe)
    rgb_srgb_safe = np.clip(xyz_to_srgb_np(xyz_safe), 0, 1)
    return jz_safe, rgb_srgb_safe, jz_arc, rgb_arc

# ============================================================
# Phase/direction alignment helpers
# ============================================================

def find_reddest_index(rgb_ring_srgb: np.ndarray) -> int:
    rgb = np.asarray(rgb_ring_srgb)
    r, g, b = rgb[:,0], rgb[:,1], rgb[:,2]
    score = r - 0.5*(g + b)
    return int(np.argmax(score))

def find_greenest_index(rgb_ring_srgb: np.ndarray) -> int:
    rgb = np.asarray(rgb_ring_srgb)
    r, g, b = rgb[:,0], rgb[:,1], rgb[:,2]
    score = g - 0.5*(r + b)
    return int(np.argmax(score))

def align_ring_to_reference(rgb_ring_srgb: np.ndarray,
                            ref_green_idx: int) -> np.ndarray:
    ring = np.asarray(rgb_ring_srgb)
    N = ring.shape[0]

    idx_red = find_reddest_index(ring)
    ring_r0 = np.roll(ring, -idx_red, axis=0)

    g1 = find_greenest_index(ring_r0)

    ring_rev = ring_r0[::-1].copy()
    g2 = find_greenest_index(ring_rev)

    def circ_dist(a, b):
        d = abs(a - b)
        return min(d, N - d)

    if circ_dist(g1, ref_green_idx) <= circ_dist(g2, ref_green_idx):
        return ring_r0
    else:
        return ring_rev

# ============================================================
# iso-Jz ring builder from Fourier ring
# ============================================================

def build_isoJ_from_fourier(jz_fourier: np.ndarray,
                            J0: float = 0.5) -> np.ndarray:
    jz_iso = jz_fourier.copy()
    jz_iso[:,0] = J0

    jz_t = torch.from_numpy(jz_iso).to(device=device, dtype=dtype)
    center = jz_t.mean(dim=0)
    offsets = jz_t - center[None,:]

    def is_safe(scale: float) -> bool:
        jz_scaled = center[None,:] + scale * offsets
        xyz = jzazbz_to_xyz_np(jz_scaled.cpu().numpy())
        srgb = xyz_to_srgb_np(xyz)
        rgb_lin = srgb_to_linear_np(srgb)
        return np.all((rgb_lin >= -1e-6) & (rgb_lin <= 1.0 + 1e-6))

    s_lo, s_hi = 0.0, 1.0
    if not is_safe(1.0):
        for _ in range(40):
            mid = 0.5*(s_lo + s_hi)
            if is_safe(float(mid)):
                s_lo = mid
            else:
                s_hi = mid
        s_opt = s_lo
    else:
        s_opt = 1.0

    jz_iso_safe_t = center[None,:] + s_opt * offsets
    jz_iso_safe   = jz_iso_safe_t.cpu().numpy()
    xyz_iso_safe  = jzazbz_to_xyz_np(jz_iso_safe)
    rgb_iso_srgb  = np.clip(xyz_to_srgb_np(xyz_iso_safe), 0, 1)
    return rgb_iso_srgb

# ============================================================
# Flow images and hue disks
# ============================================================

def build_hue_disk_from_ring(rgb_ring_lin: np.ndarray,
                             center_lin: np.ndarray,
                             size: int = 512) -> np.ndarray:
    n_hues = rgb_ring_lin.shape[0]
    N = size
    y, x = np.mgrid[0:N, 0:N]
    x = (x + 0.5)/N*2 - 1
    y = (y + 0.5)/N*2 - 1
    r = np.sqrt(x*x + y*y)
    theta = np.mod(np.arctan2(y, x), 2*np.pi)

    hue_f = theta/(2*np.pi)*n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)

    ring0 = rgb_ring_lin[idx0]
    ring1 = rgb_ring_lin[idx1]
    ring_interp = (1 - t[...,None])*ring0 + t[...,None]*ring1

    r_exp = r[...,None]
    rgb_lin = (1-r_exp)*center_lin + r_exp*ring_interp
    rgb_lin[r>1] = 1.0

    return np.clip(linear_to_srgb_np(rgb_lin), 0, 1)

def make_flow_images_for_ring(rgb_ring_lin: np.ndarray,
                              center_lin: np.ndarray,
                              flows: torch.Tensor):
    n_hues = rgb_ring_lin.shape[0]

    disk = build_hue_disk_from_ring(rgb_ring_lin, center_lin, size=512)

    u = flows[0].cpu().numpy()
    v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi)
    mag = np.sqrt(u*u + v*v)
    mag /= (mag.max() + 1e-8)

    hue_f = angle/(2*np.pi)*n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)

    ring0 = rgb_ring_lin[idx0]
    ring1 = rgb_ring_lin[idx1]
    ring_interp = (1 - t[...,None])*ring0 + t[...,None]*ring1

    r = mag[...,None]
    rgb_lin_flow = (1-r)*center_lin + r*ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow), 0, 1)

    alpha = mag[...,None]
    white = np.ones_like(flow_rgb)
    black = np.zeros_like(flow_rgb)
    flow_white = alpha*flow_rgb + (1-alpha)*white
    flow_black = alpha*flow_rgb + (1-alpha)*black

    return disk, flow_rgb, flow_white, flow_black

# ============================================================
# Warped sRGB cube faces in JzAzBz for trajectory
# ============================================================

def make_face_jz(face: str, n: int = 64):
    t = np.linspace(0.0, 1.0, n, dtype=float)
    u, v = np.meshgrid(t, t, indexing="ij")
    srgb = np.zeros((n, n, 3), dtype=float)

    if   face == "R0": srgb[...,0]=0.0; srgb[...,1]=u; srgb[...,2]=v
    elif face == "R1": srgb[...,0]=1.0; srgb[...,1]=u; srgb[...,2]=v
    elif face == "G0": srgb[...,1]=0.0; srgb[...,0]=u; srgb[...,2]=v
    elif face == "G1": srgb[...,1]=1.0; srgb[...,0]=u; srgb[...,2]=v
    elif face == "B0": srgb[...,2]=0.0; srgb[...,0]=u; srgb[...,1]=v
    elif face == "B1": srgb[...,2]=1.0; srgb[...,0]=u; srgb[...,1]=v
    else:
        raise ValueError("bad face name")

    xyz = srgb_to_xyz_np(srgb.reshape(-1,3)).reshape(n,n,3)
    jz = xyz_to_jzazbz_np(xyz.reshape(-1,3)).reshape(n,n,3)
    Jg = jz[...,0]
    Az = jz[...,1]
    Bz = jz[...,2]
    return Jg, Az, Bz

faces = ["R0","R1","G0","G1","B0","B1"]
face_surfaces_jz = {f: make_face_jz(f, n=64) for f in faces}

# ============================================================
# 3D JzAzBz trajectory plot with ticks
# ============================================================
def plot_jz_trajectory_hex_fourier_sine_hsv(
    jz_fourier: np.ndarray,
    rgb_fourier_srgb: np.ndarray,
    jz_iso: np.ndarray,
    rgb_iso_srgb: np.ndarray,
    jz_arc: np.ndarray,
    rgb_arc_srgb: np.ndarray,
    jz_sine: np.ndarray,
    rgb_sine_srgb: np.ndarray,
    jz_hsv: np.ndarray,
    rgb_hsv_srgb: np.ndarray,
    tick_len: float = 0.003,
):
    fig = go.Figure()

    # warped cube faces
    for f in faces:
        Jg, Az, Bz = face_surfaces_jz[f]
        fig.add_trace(go.Surface(
            x=Jg, y=Az, z=Bz,
            surfacecolor=np.zeros_like(Jg),
            colorscale=[[0,"rgb(120,120,120)"],[1,"rgb(120,120,120)"]],
            showscale=False,
            opacity=0.15,
            hoverinfo="skip",
            lighting=dict(ambient=1.0, diffuse=0.0, specular=0.0),
            showlegend=False,
        ))

    # cube edges
    verts = np.array(list(itertools.product([0.0,1.0],[0.0,1.0],[0.0,1.0])), dtype=float)
    edges = [(i,j) for i in range(8) for j in range(i+1,8)
             if np.sum(np.abs(verts[i]-verts[j])) == 1.0]
    t_edge = np.linspace(0.0,1.0,64)
    for i,j in edges:
        p0 = verts[i]; p1 = verts[j]
        srgb_line = p0 + (p1-p0)[None,:] * t_edge[:,None]
        xyz_line = srgb_to_xyz_np(srgb_line)
        jz_line = xyz_to_jzazbz_np(xyz_line)
        fig.add_trace(go.Scatter3d(
            x=jz_line[:,0], y=jz_line[:,1], z=jz_line[:,2],
            mode="lines",
            line=dict(width=3, color="rgb(128,128,128)"),
            hoverinfo="skip",
            showlegend=False,
        ))

    # Original Fourier-smoothed HSV hex
    fig.add_trace(go.Scatter3d(
        x=jz_fourier[:,0], y=jz_fourier[:,1], z=jz_fourier[:,2],
        mode="lines+markers",
        line=dict(width=5, color="#ffffff"),
        marker=dict(size=4, color=rgb_fourier_srgb),
        name="Fourier Jz",
    ))

    # Iso-Jz ring (constant Jz plane)
    fig.add_trace(go.Scatter3d(
        x=jz_iso[:,0], y=jz_iso[:,1], z=jz_iso[:,2],
        mode="lines+markers",
        line=dict(width=4, color="#00ffff"),
        marker=dict(size=4, color=rgb_iso_srgb),
        name="Iso-Jz ellipse",
    ))

    # Equal-arc HSV ticks as gray orthogonal ticks
    n = jz_arc.shape[0]
    tangents = np.zeros_like(jz_arc)
    for i in range(n):
        p_prev = jz_arc[(i-1) % n]
        p_next = jz_arc[(i+1) % n]
        tangents[i] = p_next - p_prev
    tangents /= np.linalg.norm(tangents, axis=1, keepdims=True) + 1e-12

    normals = np.zeros_like(jz_arc)
    for i in range(n):
        tvec = tangents[i]
        base = np.array([1.0,0.0,0.0])
        if abs(np.dot(base,tvec)) > 0.8:
            base = np.array([0.0,1.0,0.0])
        nrm = np.cross(tvec, base)
        nrm_n = np.linalg.norm(nrm)
        if nrm_n < 1e-12:
            base = np.array([0.0,0.0,1.0])
            nrm = np.cross(tvec, base)
            nrm_n = np.linalg.norm(nrm)
        normals[i] = nrm / (nrm_n + 1e-12)

    xs, ys, zs = [], [], []
    for p, nrm in zip(jz_arc, normals):
        p1 = p + tick_len * nrm
        p2 = p - tick_len * nrm
        xs.extend([p1[0], p2[0], None])
        ys.extend([p1[1], p2[1], None])
        zs.extend([p1[2], p2[2], None])

    fig.add_trace(go.Scatter3d(
        x=xs, y=ys, z=zs,
        mode="lines",
        line=dict(width=2, color="rgb(160,160,160)"),
        name="HSV hex ticks",
        hoverinfo="skip",
    ))

    # sinebow
    fig.add_trace(go.Scatter3d(
        x=jz_sine[:,0], y=jz_sine[:,1], z=jz_sine[:,2],
        mode="lines+markers",
        line=dict(width=2, color="#888888"),
        marker=dict(size=3, color=rgb_sine_srgb),
        name="sinebow",
    ))

    # HSV
    fig.add_trace(go.Scatter3d(
        x=jz_hsv[:,0], y=jz_hsv[:,1], z=jz_hsv[:,2],
        mode="lines+markers",
        line=dict(width=2, color="#cccccc"),
        marker=dict(size=3, color=rgb_hsv_srgb),
        name="HSV",
    ))

    gray = "rgb(200,200,200)"
    axis_style = dict(
        showbackground=False,
        gridcolor="rgba(200,200,200,0.25)",
        zerolinecolor="rgba(200,200,200,0.35)",
        tickfont=dict(color=gray),
    )

    fig.update_layout(
        width=1000, height=780,
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        scene=dict(
            bgcolor="rgba(0,0,0,0)",
            xaxis=dict(axis_style, title=dict(text="Jz", font=dict(color=gray))),
            yaxis=dict(axis_style, title=dict(text="az", font=dict(color=gray))),
            zaxis=dict(axis_style, title=dict(text="bz", font=dict(color=gray))),
            aspectmode="data",
        ),
        font=dict(color=gray),
        legend=dict(
            font=dict(color=gray, size=14),
            bgcolor="rgba(0,0,0,0)"
        ),
        title=dict(
            text="JzAzBz: gamut + Fourier Jz / Iso-Jz / ticks / sinebow / HSV",
            font=dict(color=gray, size=20),
        )
    )

    return HTML(pio.to_html(fig, include_plotlyjs="cdn", full_html=False))

# ============================================================
# Main visualization
# ============================================================

def visualize_fourier_hex_vs_sinebow_hsv_on_omnipose_jz(
    n_hues: int = 72,
    n_harmonics: int = 1,
    device_str: str = "cpu",
):
    dev = torch.device(device_str)

    # base HSV smooth ring
    t = np.linspace(0, 1, n_hues, endpoint=False)
    hsv_vals = np.stack([t, np.ones_like(t), np.ones_like(t)], axis=1)
    rgb_hsv_srgb_raw = hsv_to_rgb(hsv_vals)
    xyz_hsv_raw = srgb_to_xyz_np(rgb_hsv_srgb_raw)
    jz_hsv_raw = xyz_to_jzazbz_np(xyz_hsv_raw)

    # sinebow
    sine_r = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3))
    sine_g = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3))
    sine_b = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_srgb_raw = np.stack([sine_r, sine_g, sine_b], axis=1)
    xyz_sine_raw = srgb_to_xyz_np(rgb_sine_srgb_raw)
    jz_sine_raw = xyz_to_jzazbz_np(xyz_sine_raw)

    # Fourier-smoothed ring
    jz_fourier_raw, rgb_fourier_srgb_raw, jz_arc_raw, rgb_arc_srgb_raw = build_fourier_hsv_hex_ring_jz(
        n_hues=n_hues,
        n_harmonics=n_harmonics,
        oversample=8,
    )

    # align HSV to itself (red at 0, green forward)
    idx_red_h = find_reddest_index(rgb_hsv_srgb_raw)
    rgb_hsv_r0 = np.roll(rgb_hsv_srgb_raw, -idx_red_h, axis=0)
    g_h = find_greenest_index(rgb_hsv_r0)
    if g_h > n_hues // 2:
        rgb_hsv_r0 = rgb_hsv_r0[::-1].copy()
        g_h = find_greenest_index(rgb_hsv_r0)
    rgb_hsv_srgb = rgb_hsv_r0
    jz_hsv = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_hsv_srgb))

    # align arc-length samples
    rgb_arc_r0 = np.roll(rgb_arc_srgb_raw, -idx_red_h, axis=0)
    if g_h > n_hues // 2:
        rgb_arc_r0 = rgb_arc_r0[::-1].copy()
    rgb_arc_srgb = rgb_arc_r0
    jz_arc = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_arc_srgb))

    # align Fourier and sinebow to same phase
    rgb_fourier_srgb = align_ring_to_reference(rgb_fourier_srgb_raw, ref_green_idx=g_h)
    rgb_sine_srgb    = align_ring_to_reference(rgb_sine_srgb_raw,    ref_green_idx=g_h)

    jz_fourier = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_fourier_srgb))
    jz_sine    = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_sine_srgb))

    # iso-Jz ring from Fourier
    rgb_iso_srgb_raw = build_isoJ_from_fourier(jz_fourier, J0=0.5)
    rgb_iso_srgb = align_ring_to_reference(rgb_iso_srgb_raw, ref_green_idx=g_h)
    jz_iso = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_iso_srgb))

    # 3D JzAzBz trajectory
    traj_html = plot_jz_trajectory_hex_fourier_sine_hsv(
        jz_fourier, rgb_fourier_srgb,
        jz_iso,     rgb_iso_srgb,
        jz_arc,     rgb_arc_srgb,
        jz_sine,    rgb_sine_srgb,
        jz_hsv,     rgb_hsv_srgb,
    )
    display(traj_html)

    # convert aligned rings to linear RGB
    rgb_fourier_lin = srgb_to_linear_np(rgb_fourier_srgb)
    rgb_iso_lin     = srgb_to_linear_np(rgb_iso_srgb)
    rgb_sine_lin    = srgb_to_linear_np(rgb_sine_srgb)
    rgb_hsv_lin     = srgb_to_linear_np(rgb_hsv_srgb)

    center_srgb = np.array([0.5, 0.5, 0.5], dtype=float)
    center_lin = srgb_to_linear_np(center_srgb)

    # load Omnipose flows
    omnidir = Path(omnirefactor.__file__).parent.parent.parent
    basedir = os.path.join(omnidir, "docs", "_static")
    name = "ecoli"
    ext = ".tif"

    image = io.imread(os.path.join(basedir, name + ext))
    masks = io.imread(os.path.join(basedir, name + "_labels" + ext))
    slc = ocdkit.measure.crop_bbox(masks > 0, pad=0)[0]
    masks = fastremap.renumber(masks[slc])[0]
    image = image[slc]

    flows_all = omnirefactor.core.masks_to_flows(masks, device="cpu")[4].to(dev)
    flows = flows_all

    # phase 0 flow images
    disk_fourier, flow_fourier, flow_fourier_white, flow_fourier_black = make_flow_images_for_ring(
        rgb_fourier_lin, center_lin, flows
    )
    disk_iso,     flow_iso,     flow_iso_white,     flow_iso_black     = make_flow_images_for_ring(
        rgb_iso_lin,     center_lin, flows
    )
    disk_sine,    flow_sine,    flow_sine_white,    flow_sine_black    = make_flow_images_for_ring(
        rgb_sine_lin,    center_lin, flows
    )
    disk_hsv,     flow_hsv,     flow_hsv_white,     flow_hsv_black     = make_flow_images_for_ring(
        rgb_hsv_lin,     center_lin, flows
    )

    # rotated (+90°) rings and flows
    shift = n_hues // 4
    rgb_fourier_lin_rot = np.roll(rgb_fourier_lin, shift, axis=0)
    rgb_iso_lin_rot     = np.roll(rgb_iso_lin,     shift, axis=0)
    rgb_sine_lin_rot    = np.roll(rgb_sine_lin,    shift, axis=0)
    rgb_hsv_lin_rot     = np.roll(rgb_hsv_lin,     shift, axis=0)

    _, _, flow_fourier_white_rot, flow_fourier_black_rot = make_flow_images_for_ring(
        rgb_fourier_lin_rot, center_lin, flows
    )
    _, _, flow_iso_white_rot, flow_iso_black_rot         = make_flow_images_for_ring(
        rgb_iso_lin_rot,     center_lin, flows
    )
    _, _, flow_sine_white_rot, flow_sine_black_rot       = make_flow_images_for_ring(
        rgb_sine_lin_rot,    center_lin, flows
    )
    _, _, flow_hsv_white_rot, flow_hsv_black_rot         = make_flow_images_for_ring(
        rgb_hsv_lin_rot,     center_lin, flows
    )

    # final 4×6 grid
    fig, axes = plt.subplots(4, 6, figsize=(26, 10))
    row_titles = [
        f"Fourier Jz (M={n_harmonics})",
        "Iso-Jz (Jz≈0.5)",
        "sinebow",
        "HSV",
    ]

    # Fourier row
    axes[0,0].imshow(disk_fourier);           axes[0,0].axis("off")
    axes[0,1].imshow(flow_fourier);           axes[0,1].axis("off")
    axes[0,2].imshow(flow_fourier_white);     axes[0,2].axis("off")
    axes[0,3].imshow(flow_fourier_black);     axes[0,3].axis("off")
    axes[0,4].imshow(flow_fourier_white_rot); axes[0,4].axis("off")
    axes[0,5].imshow(flow_fourier_black_rot); axes[0,5].axis("off")

    # Iso-Jz row
    axes[1,0].imshow(disk_iso);               axes[1,0].axis("off")
    axes[1,1].imshow(flow_iso);               axes[1,1].axis("off")
    axes[1,2].imshow(flow_iso_white);         axes[1,2].axis("off")
    axes[1,3].imshow(flow_iso_black);         axes[1,3].axis("off")
    axes[1,4].imshow(flow_iso_white_rot);     axes[1,4].axis("off")
    axes[1,5].imshow(flow_iso_black_rot);     axes[1,5].axis("off")

    # sinebow row
    axes[2,0].imshow(disk_sine);              axes[2,0].axis("off")
    axes[2,1].imshow(flow_sine);              axes[2,1].axis("off")
    axes[2,2].imshow(flow_sine_white);        axes[2,2].axis("off")
    axes[2,3].imshow(flow_sine_black);        axes[2,3].axis("off")
    axes[2,4].imshow(flow_sine_white_rot);    axes[2,4].axis("off")
    axes[2,5].imshow(flow_sine_black_rot);    axes[2,5].axis("off")

    # HSV row
    axes[3,0].imshow(disk_hsv);               axes[3,0].axis("off")
    axes[3,1].imshow(flow_hsv);               axes[3,1].axis("off")
    axes[3,2].imshow(flow_hsv_white);         axes[3,2].axis("off")
    axes[3,3].imshow(flow_hsv_black);         axes[3,3].axis("off")
    axes[3,4].imshow(flow_hsv_white_rot);     axes[3,4].axis("off")
    axes[3,5].imshow(flow_hsv_black_rot);     axes[3,5].axis("off")

    for i,title in enumerate(row_titles):
        axes[i,0].set_ylabel(title, fontsize=12)

    axes[0,0].set_title("Hue disk (0°)")
    axes[0,1].set_title("Flow → chroma (0°)")
    axes[0,2].set_title("Alpha on white (0°)")
    axes[0,3].set_title("Alpha on black (0°)")
    axes[0,4].set_title("Alpha on white (+90°)")
    axes[0,5].set_title("Alpha on black (+90°)")

    plt.tight_layout()
    plt.show()

# ============================================================
# Run everything
# ============================================================

if __name__ == "__main__":
    N_HUES = 72
    N_HARMONICS = 1

    visualize_fourier_hex_vs_sinebow_hsv_on_omnipose_jz(
        n_hues=N_HUES,
        n_harmonics=N_HARMONICS,
        device_str="cpu",
    )

In [ ]:
import numpy as np
import torch
import os
from pathlib import Path
from skimage import io
import fastremap
import matplotlib.pyplot as plt

import plotly.graph_objects as go
from plotly.offline import plot
from IPython.display import HTML

import omnirefactor

dtype = torch.float64

# -------------------------------------------------------------------------
# Alignment helpers
# -------------------------------------------------------------------------

def find_reddest_index(rgb_ring_srgb: np.ndarray) -> int:
    """
    Return the index of the reddest color in an sRGB ring.
    Metric: R - 0.5*(G+B).
    """
    rgb = np.asarray(rgb_ring_srgb)
    r = rgb[:, 0]
    g = rgb[:, 1]
    b = rgb[:, 2]
    score = r - 0.5 * (g + b)
    return int(np.argmax(score))


def find_greenest_index(rgb_ring_srgb: np.ndarray) -> int:
    """
    Return the index of the greenest color in an sRGB ring.
    Metric: G - 0.5*(R+B).
    """
    rgb = np.asarray(rgb_ring_srgb)
    r = rgb[:, 0]
    g = rgb[:, 1]
    b = rgb[:, 2]
    score = g - 0.5 * (r + b)
    return int(np.argmax(score))


def align_ring_to_reference(rgb_ring_srgb: np.ndarray,
                            ref_green_idx: int) -> np.ndarray:
    """
    Align an sRGB ring to a canonical reference where:
      - reddest color is at index 0
      - greenest color lies in the same direction as ref_green_idx.
    """
    ring = np.asarray(rgb_ring_srgb)
    N = ring.shape[0]

    idx_red = find_reddest_index(ring)
    ring_r0 = np.roll(ring, -idx_red, axis=0)

    g1 = find_greenest_index(ring_r0)
    ring_rev = ring_r0[::-1].copy()
    g2 = find_greenest_index(ring_rev)

    def circ_dist(a, b):
        d = abs(a - b)
        return min(d, N - d)

    if circ_dist(g1, ref_green_idx) <= circ_dist(g2, ref_green_idx):
        return ring_r0
    else:
        return ring_rev


# -------------------------------------------------------------------------
# Iso-Jz ring (unchanged)
# -------------------------------------------------------------------------

def build_isoJ_from_fourier(jz_fourier: np.ndarray,
                            J0: float = 0.5) -> np.ndarray:
    """
    Build an iso-Jz ring from a JzAzBz Fourier ring.
    """
    jz_iso = jz_fourier.copy()
    jz_iso[:, 0] = J0

    jz_t = torch.from_numpy(jz_iso).to(dtype=dtype)
    center = jz_t.mean(dim=0)
    offsets = jz_t - center[None, :]

    def is_safe(scale: float) -> bool:
        jz_scaled = center[None, :] + scale * offsets
        jz_np = jz_scaled.cpu().numpy()
        xyz = jzazbz_to_xyz_np(jz_np)
        srgb = xyz_to_srgb_np(xyz)
        rgb_lin = srgb_to_linear_np(srgb)
        return np.all((rgb_lin >= -1e-6) & (rgb_lin <= 1.0 + 1e-6))

    s_lo, s_hi = 0.0, 1.0
    if not is_safe(1.0):
        for _ in range(40):
            mid = 0.5 * (s_lo + s_hi)
            if is_safe(float(mid)):
                s_lo = mid
            else:
                s_hi = mid
        s_opt = s_lo
    else:
        s_opt = 1.0

    jz_iso_safe_t = center[None, :] + s_opt * offsets
    jz_iso_safe = jz_iso_safe_t.cpu().numpy()
    xyz_iso_safe = jzazbz_to_xyz_np(jz_iso_safe)
    rgb_iso_srgb = np.clip(xyz_to_srgb_np(xyz_iso_safe), 0, 1)
    return rgb_iso_srgb


# -------------------------------------------------------------------------
# Salience vs black–white neutral line (global warp)
# -------------------------------------------------------------------------

def relative_luminance_from_linear(rgb_lin: np.ndarray) -> np.ndarray:
    """
    Compute relative luminance Y from linear sRGB.
    """
    rgb_lin = np.asarray(rgb_lin)
    r = rgb_lin[..., 0]
    g = rgb_lin[..., 1]
    b = rgb_lin[..., 2]
    return 0.2126 * r + 0.7152 * g + 0.0722 * b


def pc_contrast(Y_fg: float,
                Y_bg: float,
                exp: float = 0.4,
                eps: float = 1e-4) -> float:
    """
    Perceptual contrast-like metric between foreground and background
    luminances (Y_fg, Y_bg), using a symmetric power law.

        L = max(Y, eps)**exp
        PC = |L_fg - L_bg| / (L_fg + L_bg)
    """
    Y_fg = float(Y_fg)
    Y_bg = float(Y_bg)
    L_fg = max(Y_fg, eps) ** exp
    L_bg = max(Y_bg, eps) ** exp
    return abs(L_fg - L_bg) / (L_fg + L_bg)


def build_contrast_balanced_ring_from_fourier(
    jz_fourier: np.ndarray,
    C_min: float = 0.04,
    J_center_min: float = 0.45,
    J_center_max: float = 0.70,
    J_center_step: float = 0.02,
    C_scale_min: float = 0.6,
    C_scale_max: float = 1.35,
    C_scale_step: float = 0.05,
    w_mean: float = 1.0,
    w_min: float = 0.8,
    w_bg: float = 0.6,
    w_hue: float = 0.6,
    w_chr: float = 0.6,
    exp_pc: float = 0.4,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Build a contrast-balanced JzAzBz ring from a Fourier JzAzBz ring
    using a global warp:

        J'  = J_center + (J - J_mean)
        C'  = C_scale * C

    where C = sqrt(az^2 + bz^2), and az', bz' = C' * u, u being the
    chroma direction from the original Fourier ring.

    The objective aggregates over all hues and neutral backgrounds and
    enforces a chroma floor C_min so the ring never approaches the gray
    line. This avoids the per-hue spikes seen in the radial search.
    """
    jz_fourier = np.asarray(jz_fourier, dtype=np.float64)
    J_base = jz_fourier[:, 0]
    az_base = jz_fourier[:, 1]
    bz_base = jz_fourier[:, 2]

    C_base = np.hypot(az_base, bz_base)
    eps = 1e-6
    u_az = az_base / np.maximum(C_base, eps)
    u_bz = bz_base / np.maximum(C_base, eps)

    J_mean = float(J_base.mean())

    # backgrounds along the neutral line in luminance space
    Y_bgs = np.array([0.0, 0.25, 0.5, 0.75, 1.0], dtype=np.float64)

    best_score = -1e9
    best_jz = None
    best_rgb = None

    J_centers = np.arange(J_center_min, J_center_max + 1e-9, J_center_step)
    C_scales = np.arange(C_scale_min, C_scale_max + 1e-9, C_scale_step)

    for J_center in J_centers:
        J_new = J_center + (J_base - J_mean)
        if np.any(J_new < 0.0) or np.any(J_new > 1.0):
            continue

        for C_scale in C_scales:
            C_new = C_scale * C_base
            if np.min(C_new) < C_min:
                continue

            az_new = C_new * u_az
            bz_new = C_new * u_bz
            jz_candidate = np.stack([J_new, az_new, bz_new], axis=1)

            xyz = jzazbz_to_xyz_np(jz_candidate)
            srgb = xyz_to_srgb_np(xyz)
            rgb_clip = np.clip(srgb, 0.0, 1.0)
            rgb_lin = srgb_to_linear_np(rgb_clip)

            # reject out-of-gamut candidate
            if np.any(rgb_lin < -1e-4) or np.any(rgb_lin > 1.0 + 1e-4):
                continue

            Y_fg = relative_luminance_from_linear(rgb_lin)  # [N]

            # PC matrix [N, n_bg]
            pc_mat = np.empty((jz_candidate.shape[0], Y_bgs.size), dtype=np.float64)
            for j, Y_bg in enumerate(Y_bgs):
                for i in range(jz_candidate.shape[0]):
                    pc_mat[i, j] = pc_contrast(float(Y_fg[i]), float(Y_bg), exp=exp_pc)

            mean_pc = float(pc_mat.mean())
            min_pc = float(pc_mat.min())
            var_bg = float(pc_mat.var(axis=1).mean())
            var_hue = float(pc_mat.var(axis=0).mean())
            mean_C = float(C_new.mean())

            score = (
                w_mean * mean_pc +
                w_min * min_pc +
                w_chr * mean_C -
                w_bg * var_bg -
                w_hue * var_hue
            )

            if score > best_score:
                best_score = score
                best_jz = jz_candidate
                best_rgb = rgb_clip

    if best_jz is None:
        # fall back to the original Fourier ring if nothing better is found
        best_jz = jz_fourier
        xyz = jzazbz_to_xyz_np(best_jz)
        best_rgb = np.clip(xyz_to_srgb_np(xyz), 0.0, 1.0)

    return best_rgb, best_jz




def find_max_chroma_binary_search(J_target, az_dir, bz_dir, steps=10):
    """
    Finds the maximum Chroma (C) such that (J, C*az_dir, C*bz_dir) is in sRGB gamut.
    Assumes az_dir, bz_dir are normalized (or define the direction).
    Vectorized for speed.
    """
    # Start with a safe range [0, C_high]
    # C=0 is always gray (safe). C=1.0 is usually plenty for sRGB.
    low = np.zeros_like(J_target)
    high = np.ones_like(J_target) * 1.5 
    
    # Binary search
    for _ in range(steps):
        mid = (low + high) * 0.5
        
        # Reconstruct JzAzBz -> RGB
        jz_test = np.stack([J_target, mid * az_dir, mid * bz_dir], axis=1)
        xyz = jzazbz_to_xyz_np(jz_test)
        srgb = xyz_to_srgb_np(xyz)
        rgb_lin = srgb_to_linear_np(srgb)
        
        # Check strictly in gamut (with tiny tolerance)
        # We check linear RGB to be safe against gamma curve weirdness near 0
        is_valid = np.all((rgb_lin >= -1e-4) & (rgb_lin <= 1.0 + 1e-4), axis=1)
        
        # Update bounds
        low = np.where(is_valid, mid, low)
        high = np.where(is_valid, high, mid)
        
    return low

# def soft_min_chroma(c_target, c_limit, k=20.0):
#     """
#     Smoothly blends between c_target and c_limit using a SoftMin function.
#     k controls sharpness: higher k = closer to the wall but sharper corners.
#     k=20 is a good balance for color trajectories.
    
#     SoftMin(a, b) = -log(exp(-k*a) + exp(-k*b)) / k
#     """
#     # We use LogSumExp trick for numerical stability
#     # - (1/k) * log( exp(-k*a) + exp(-k*b) )
#     val = np.stack([c_target, c_limit], axis=0)
#     return -np.log(np.sum(np.exp(-k * val), axis=0)) / k

# def build_contrast_balanced_ring_from_fourier(
#     jz_fourier: np.ndarray,
#     C_min: float = 0.04,
#     J_center_min: float = 0.45,
#     J_center_max: float = 0.70,
#     J_center_step: float = 0.02,
#     C_scale_min: float = 0.6,
#     C_scale_max: float = 1.5,  # Allow checking larger scales
#     C_scale_step: float = 0.05,
#     w_mean: float = 1.0,
#     w_min: float = 0.8,
#     w_bg: float = 0.6,
#     w_hue: float = 0.6,
#     w_chr: float = 0.6,
#     exp_pc: float = 0.4,
# ) -> tuple[np.ndarray, np.ndarray]:
#     """
#     Build a contrast-balanced JzAzBz ring.
    
#     IMPROVEMENT:
#     Instead of hard-clipping RGB (which flips vectors), we calculate the 
#     max valid chroma for the gamut and apply a "Soft Minimum" between 
#     the target harmonic chroma and the gamut wall. This creates a 
#     smooth, expanded trajectory that 'hugs' the gamut boundary.
#     """
#     jz_fourier = np.asarray(jz_fourier, dtype=np.float64)
#     J_base = jz_fourier[:, 0]
#     az_base = jz_fourier[:, 1]
#     bz_base = jz_fourier[:, 2]

#     C_base = np.hypot(az_base, bz_base)
#     eps = 1e-6
#     u_az = az_base / np.maximum(C_base, eps)
#     u_bz = bz_base / np.maximum(C_base, eps)

#     J_mean = float(J_base.mean())
#     Y_bgs = np.array([0.0, 0.25, 0.5, 0.75, 1.0], dtype=np.float64)

#     best_score = -1e9
#     best_jz = None
#     best_rgb = None

#     J_centers = np.arange(J_center_min, J_center_max + 1e-9, J_center_step)
#     C_scales = np.arange(C_scale_min, C_scale_max + 1e-9, C_scale_step)

#     for J_center in J_centers:
#         J_new = J_center + (J_base - J_mean)
        
#         # Skip if Lightness itself is broken
#         if np.any(J_new < 0.0) or np.any(J_new > 1.0):
#             continue
            
#         # 1. Calculate the "Gamut Wall" (Max Chroma) for this specific Lightness ring
#         # We do this once per J_center to save time (since J dominates the gamut shape)
#         # Using a binary search to find the intersection of the hue vectors with the RGB cube
#         C_limits = find_max_chroma_binary_search(J_new, u_az, u_bz, steps=12)

#         for C_scale in C_scales:
#             C_target = C_scale * C_base
#             if np.min(C_target) < C_min:
#                 continue

#             # 2. Apply Soft Minimum
#             # This smoothly compresses the ellipse ONLY where it hits the wall.
#             # It preserves the Fourier shape where it is inside the gamut.
#             C_smooth = soft_min_chroma(C_target, C_limits, k=25.0)

#             # Reconstruct
#             az_final = C_smooth * u_az
#             bz_final = C_smooth * u_bz
#             jz_candidate = np.stack([J_new, az_final, bz_final], axis=1)

#             # Convert to RGB
#             # We can now safely clip for float errors because we are geometrically inside
#             xyz = jzazbz_to_xyz_np(jz_candidate)
#             srgb = xyz_to_srgb_np(xyz)
#             rgb_clip = np.clip(srgb, 0.0, 1.0)
#             rgb_lin = srgb_to_linear_np(rgb_clip)

#             # --- Scoring (unchanged) ---
#             Y_fg = relative_luminance_from_linear(rgb_lin)

#             pc_mat = np.empty((jz_candidate.shape[0], Y_bgs.size), dtype=np.float64)
#             for j, Y_bg in enumerate(Y_bgs):
#                 # Vectorized PC calc
#                 L_fg = np.maximum(Y_fg, 1e-4) ** exp_pc
#                 L_bg = max(Y_bg, 1e-4) ** exp_pc
#                 pc_mat[:, j] = np.abs(L_fg - L_bg) / (L_fg + L_bg)

#             mean_pc = float(pc_mat.mean())
#             min_pc = float(pc_mat.min())
#             var_bg = float(pc_mat.var(axis=1).mean())
#             var_hue = float(pc_mat.var(axis=0).mean())
#             mean_C = float(C_smooth.mean()) # Use the actual smooth chroma

#             score = (
#                 w_mean * mean_pc +
#                 w_min * min_pc +
#                 w_chr * mean_C -
#                 w_bg * var_bg -
#                 w_hue * var_hue
#             )

#             if score > best_score:
#                 best_score = score
#                 best_jz = jz_candidate
#                 best_rgb = rgb_clip

#     if best_jz is None:
#         best_jz = jz_fourier
#         xyz = jzazbz_to_xyz_np(best_jz)
#         best_rgb = np.clip(xyz_to_srgb_np(xyz), 0.0, 1.0)

#     return best_rgb, best_jz

# -------------------------------------------------------------------------
# Plotly 3D trajectory with warped sRGB cube surfaces
# -------------------------------------------------------------------------

def _srgb_face_to_jz_surface(n: int,
                             fixed_channel: int,
                             fixed_value: float) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Build a JzAzBz surface for one face of the sRGB cube.

    fixed_channel: 0=R, 1=G, 2=B
    fixed_value:   0.0 or 1.0
    """
    grid = np.linspace(0.0, 1.0, n)
    G, B = np.meshgrid(grid, grid)

    if fixed_channel == 0:
        R = np.full_like(G, fixed_value)
        srgb = np.stack([R, G, B], axis=-1)
    elif fixed_channel == 1:
        Gf = np.full_like(G, fixed_value)
        srgb = np.stack([G, Gf, B], axis=-1)
    else:
        Bf = np.full_like(G, fixed_value)
        srgb = np.stack([G, B, Bf], axis=-1)

    xyz = srgb_to_xyz_np(srgb.reshape(-1, 3))
    jz = xyz_to_jzazbz_np(xyz).reshape(srgb.shape)
    J = jz[..., 0]
    az = jz[..., 1]
    bz = jz[..., 2]
    return az, bz, J


def plot_jz_trajectory_hex_fourier_sine_hsv(
    jz_fourier: np.ndarray, rgb_fourier: np.ndarray,
    jz_arc: np.ndarray,      rgb_arc: np.ndarray,
    jz_sine: np.ndarray,     rgb_sine: np.ndarray,
    jz_hsv: np.ndarray,      rgb_hsv: np.ndarray,
    jz_iso: np.ndarray,      rgb_iso: np.ndarray,
    jz_bal: np.ndarray,      rgb_bal: np.ndarray,
):
    """
    Plot JzAzBz trajectories for several rings plus warped sRGB cube faces
    using Plotly in dark mode.

    Axes:
        x = az
        y = bz
        z = Jz
    """
    fig = go.Figure()

    # sRGB cube faces as translucent surfaces
    n_face = 17
    face_specs = [
        (0, 0.0, "R=0"),
        (0, 1.0, "R=1"),
        (1, 0.0, "G=0"),
        (1, 1.0, "G=1"),
        (2, 0.0, "B=0"),
        (2, 1.0, "B=1"),
    ]
    for fixed_channel, fixed_value, name in face_specs:
        az, bz, J = _srgb_face_to_jz_surface(n_face, fixed_channel, fixed_value)
        fig.add_trace(
            go.Surface(
                x=az,
                y=bz,
                z=J,
                opacity=0.18,
                showscale=False,
                colorscale="Viridis",
                name=name,
                hoverinfo="skip",
            )
        )

    def add_curve(jz, rgb, name, width=7):
        jz = np.asarray(jz)
        fig.add_trace(
            go.Scatter3d(
                x=jz[:, 1],
                y=jz[:, 2],
                z=jz[:, 0],
                mode="lines",
                name=name,
                line=dict(width=width),
            )
        )

    add_curve(jz_fourier, rgb_fourier, "Fourier", width=7)
    add_curve(jz_arc,      rgb_arc,      "Arc-length HSV", width=5)
    add_curve(jz_sine,     rgb_sine,     "sinebow", width=4)
    add_curve(jz_hsv,      rgb_hsv,      "HSV", width=4)
    add_curve(jz_iso,      rgb_iso,      "Iso-Jz", width=5)
    add_curve(jz_bal,      rgb_bal,      "Balanced", width=8)

    fig.update_layout(
        template="plotly_dark",
        paper_bgcolor="black",
        scene=dict(
            xaxis_title="az",
            yaxis_title="bz",
            zaxis_title="Jz",
            bgcolor="black",
            aspectmode="data",
        ),
        margin=dict(l=0, r=0, b=0, t=40),
        title="JzAzBz trajectories with warped sRGB cube and balanced ring",
        showlegend=True,
    )

    html = plot(fig, output_type="div", include_plotlyjs="cdn")
    return HTML(html)


# -------------------------------------------------------------------------
# Main visualization using the new balanced ring and Plotly figure
# -------------------------------------------------------------------------
def plot_jz_trajectory_hex_fourier_sine_hsv(
    jz_fourier: np.ndarray, rgb_fourier: np.ndarray,
    jz_arc: np.ndarray,      rgb_arc: np.ndarray,
    jz_sine: np.ndarray,     rgb_sine: np.ndarray,
    jz_hsv: np.ndarray,      rgb_hsv: np.ndarray,
    jz_iso: np.ndarray,      rgb_iso: np.ndarray,
    jz_bal: np.ndarray,      rgb_bal: np.ndarray,
):
    """
    Plot JzAzBz trajectories for several rings plus warped sRGB cube faces
    using Plotly in dark mode.

    Axes:
        x = az
        y = bz
        z = Jz
    """
    fig = go.Figure()

    # sRGB cube faces as translucent surfaces
    n_face = 17
    face_specs = [
        (0, 0.0, "R=0"),
        (0, 1.0, "R=1"),
        (1, 0.0, "G=0"),
        (1, 1.0, "G=1"),
        (2, 0.0, "B=0"),
        (2, 1.0, "B=1"),
    ]
    for fixed_channel, fixed_value, name in face_specs:
        az, bz, J = _srgb_face_to_jz_surface(n_face, fixed_channel, fixed_value)
        fig.add_trace(
            go.Surface(
                x=az,
                y=bz,
                z=J,
                opacity=0.18,
                showscale=False,
                colorscale="Viridis",
                name=name,
                hoverinfo="skip",
            )
        )

    def add_curve(jz, rgb, name, width=7):
        jz = np.asarray(jz)
        fig.add_trace(
            go.Scatter3d(
                x=jz[:, 1],
                y=jz[:, 2],
                z=jz[:, 0],
                mode="lines",
                name=name,
                line=dict(width=width),
            )
        )

    # Order here matches the row labels used below
    add_curve(jz_fourier, rgb_fourier, "Fourier",        width=7)
    add_curve(jz_arc,      rgb_arc,      "Arc-length HSV", width=5)
    add_curve(jz_iso,      rgb_iso,      "Iso-Jz",        width=5)
    add_curve(jz_bal,      rgb_bal,      "Balanced",      width=8)
    add_curve(jz_sine,     rgb_sine,     "sinebow",       width=4)
    add_curve(jz_hsv,      rgb_hsv,      "HSV",           width=4)

    fig.update_layout(
        template="plotly_dark",
        paper_bgcolor="black",
        scene=dict(
            xaxis_title="az",
            yaxis_title="bz",
            zaxis_title="Jz",
            bgcolor="black",
            aspectmode="data",
        ),
        margin=dict(l=0, r=0, b=0, t=40),
        title="JzAzBz trajectories with warped sRGB cube and balanced ring",
        showlegend=True,
    )

    html = plot(fig, output_type="div", include_plotlyjs="cdn")
    return HTML(html)


def visualize_fourier_hex_vs_sinebow_hsv_on_omnipose_jz(
    n_hues: int = 72,
    n_harmonics: int = 1,
    device_str: str = "cpu",
):
    dev = torch.device(device_str)

    # 1) base HSV wheel (smooth) in sRGB & JzAzBz
    t = np.linspace(0, 1, n_hues, endpoint=False)
    hsv_vals = np.stack([t, np.ones_like(t), np.ones_like(t)], axis=1)
    rgb_hsv_srgb_raw = hsv_to_rgb(hsv_vals)
    xyz_hsv_raw = srgb_to_xyz_np(rgb_hsv_srgb_raw)
    jz_hsv_raw = xyz_to_jzazbz_np(xyz_hsv_raw)

    # sinebow
    sine_r = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0 / 3))
    sine_g = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1 / 3))
    sine_b = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2 / 3))
    rgb_sine_srgb_raw = np.stack([sine_r, sine_g, sine_b], axis=1)
    xyz_sine_raw = srgb_to_xyz_np(rgb_sine_srgb_raw)
    jz_sine_raw = xyz_to_jzazbz_np(xyz_sine_raw)

    # 2) Fourier-smoothed HSV hex colormap in JzAzBz (raw)
    jz_fourier_raw, rgb_fourier_srgb_raw, jz_arc_raw, rgb_arc_srgb_raw = build_fourier_hsv_hex_ring_jz(
        n_hues=n_hues,
        n_harmonics=n_harmonics,
        oversample=8,
    )

    # Canonical alignment based on HSV
    idx_red_h = find_reddest_index(rgb_hsv_srgb_raw)
    rgb_hsv_r0 = np.roll(rgb_hsv_srgb_raw, -idx_red_h, axis=0)
    g_h = find_greenest_index(rgb_hsv_r0)
    if g_h > n_hues // 2:
        rgb_hsv_r0 = rgb_hsv_r0[::-1].copy()
        g_h = find_greenest_index(rgb_hsv_r0)
    rgb_hsv_srgb = rgb_hsv_r0
    jz_hsv = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_hsv_srgb))

    rgb_arc_r0 = np.roll(rgb_arc_srgb_raw, -idx_red_h, axis=0)
    if g_h > n_hues // 2:
        rgb_arc_r0 = rgb_arc_r0[::-1].copy()
    rgb_arc_srgb = rgb_arc_r0
    jz_arc = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_arc_srgb))

    rgb_fourier_srgb = align_ring_to_reference(rgb_fourier_srgb_raw, ref_green_idx=g_h)
    rgb_sine_srgb = align_ring_to_reference(rgb_sine_srgb_raw, ref_green_idx=g_h)

    jz_fourier = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_fourier_srgb))
    jz_sine = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_sine_srgb))

    # Iso-Jz ring
    rgb_iso_srgb_raw = build_isoJ_from_fourier(jz_fourier, J0=0.5)
    rgb_iso_srgb = align_ring_to_reference(rgb_iso_srgb_raw, ref_green_idx=g_h)
    jz_iso = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_iso_srgb))

    # Balanced ring via global warp
    rgb_bal_srgb_raw, jz_bal_raw = build_contrast_balanced_ring_from_fourier(
        jz_fourier,
        C_min=0.05,
        J_center_min=0.65,
        J_center_max=0.8,
        J_center_step=0.02,
        C_scale_min=0.7,
        C_scale_max=1.35,
        C_scale_step=0.05,
    )
    rgb_bal_srgb = align_ring_to_reference(rgb_bal_srgb_raw, ref_green_idx=g_h)
    jz_bal = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_bal_srgb))

    # Plotly JzAzBz figure with cube surfaces and all rings
    traj_html = plot_jz_trajectory_hex_fourier_sine_hsv(
        jz_fourier, rgb_fourier_srgb,
        jz_arc, rgb_arc_srgb,
        jz_sine, rgb_sine_srgb,
        jz_hsv, rgb_hsv_srgb,
        jz_iso, rgb_iso_srgb,
        jz_bal, rgb_bal_srgb,
    )
    display(traj_html)

    # Convert rings to linear RGB for disks/flows
    rgb_fourier_lin = srgb_to_linear_np(rgb_fourier_srgb)
    rgb_arc_lin = srgb_to_linear_np(rgb_arc_srgb)
    rgb_iso_lin = srgb_to_linear_np(rgb_iso_srgb)
    rgb_bal_lin = srgb_to_linear_np(rgb_bal_srgb)
    rgb_sine_lin = srgb_to_linear_np(rgb_sine_srgb)
    rgb_hsv_lin = srgb_to_linear_np(rgb_hsv_srgb)

    center_srgb = np.array([0.5, 0.5, 0.5], dtype=float)
    center_lin = srgb_to_linear_np(center_srgb)

    # Omnipose E. coli flows
    omnidir = Path(omnirefactor.__file__).parent.parent.parent
    basedir = os.path.join(omnidir, "docs", "_static")
    name = "ecoli"
    ext = ".tif"

    image = io.imread(os.path.join(basedir, name + ext))
    masks = io.imread(os.path.join(basedir, name + "_labels" + ext))
    slc = ocdkit.measure.crop_bbox(masks > 0, pad=0)[0]
    masks = fastremap.renumber(masks[slc])[0]
    image = image[slc]

    flows_all = omnirefactor.core.masks_to_flows(masks, device="cpu")[4].to(dev)
    flows = flows_all

    # Build images for each colormap (0° phase)
    disk_fourier, flow_fourier, flow_fourier_white, flow_fourier_black = make_flow_images_for_ring(
        rgb_fourier_lin, center_lin, flows
    )
    disk_arc, flow_arc, flow_arc_white, flow_arc_black = make_flow_images_for_ring(
        rgb_arc_lin, center_lin, flows
    )
    disk_iso, flow_iso, flow_iso_white, flow_iso_black = make_flow_images_for_ring(
        rgb_iso_lin, center_lin, flows
    )
    disk_bal, flow_bal, flow_bal_white, flow_bal_black = make_flow_images_for_ring(
        rgb_bal_lin, center_lin, flows
    )
    disk_sine, flow_sine, flow_sine_white, flow_sine_black = make_flow_images_for_ring(
        rgb_sine_lin, center_lin, flows
    )
    disk_hsv, flow_hsv, flow_hsv_white, flow_hsv_black = make_flow_images_for_ring(
        rgb_hsv_lin, center_lin, flows
    )

    # Rotated (+90°) colormaps
    shift = n_hues // 4

    rgb_fourier_lin_rot = np.roll(rgb_fourier_lin, shift, axis=0)
    rgb_arc_lin_rot = np.roll(rgb_arc_lin, shift, axis=0)
    rgb_iso_lin_rot = np.roll(rgb_iso_lin, shift, axis=0)
    rgb_bal_lin_rot = np.roll(rgb_bal_lin, shift, axis=0)
    rgb_sine_lin_rot = np.roll(rgb_sine_lin, shift, axis=0)
    rgb_hsv_lin_rot = np.roll(rgb_hsv_lin, shift, axis=0)

    _, _, flow_fourier_white_rot, flow_fourier_black_rot = make_flow_images_for_ring(
        rgb_fourier_lin_rot, center_lin, flows
    )
    _, _, flow_arc_white_rot, flow_arc_black_rot = make_flow_images_for_ring(
        rgb_arc_lin_rot, center_lin, flows
    )
    _, _, flow_iso_white_rot, flow_iso_black_rot = make_flow_images_for_ring(
        rgb_iso_lin_rot, center_lin, flows
    )
    _, _, flow_bal_white_rot, flow_bal_black_rot = make_flow_images_for_ring(
        rgb_bal_lin_rot, center_lin, flows
    )
    _, _, flow_sine_white_rot, flow_sine_black_rot = make_flow_images_for_ring(
        rgb_sine_lin_rot, center_lin, flows
    )
    _, _, flow_hsv_white_rot, flow_hsv_black_rot = make_flow_images_for_ring(
        rgb_hsv_lin_rot, center_lin, flows
    )

    # Unified figure: 6 rows × 6 columns
        # Unified figure: 6 rows × 6 columns
    fig, axes = plt.subplots(6, 6, figsize=(26, 14))
    row_titles = [
        "Fourier",
        "Arc-length HSV",
        "Iso-Jz",
        "Balanced",
        "sinebow",
        "HSV",
    ]

    # helper to hide ticks/spines but keep labels
    def _strip_axes(ax):
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)

    # Row 0: Fourier
    axes[0, 0].imshow(disk_fourier)
    _strip_axes(axes[0, 0])
    axes[0, 1].imshow(flow_fourier);           axes[0, 1].axis("off")
    axes[0, 2].imshow(flow_fourier_white);     axes[0, 2].axis("off")
    axes[0, 3].imshow(flow_fourier_black);     axes[0, 3].axis("off")
    axes[0, 4].imshow(flow_fourier_white_rot); axes[0, 4].axis("off")
    axes[0, 5].imshow(flow_fourier_black_rot); axes[0, 5].axis("off")

    # Row 1: Arc-length HSV
    axes[1, 0].imshow(disk_arc)
    _strip_axes(axes[1, 0])
    axes[1, 1].imshow(flow_arc);               axes[1, 1].axis("off")
    axes[1, 2].imshow(flow_arc_white);         axes[1, 2].axis("off")
    axes[1, 3].imshow(flow_arc_black);         axes[1, 3].axis("off")
    axes[1, 4].imshow(flow_arc_white_rot);     axes[1, 4].axis("off")
    axes[1, 5].imshow(flow_arc_black_rot);     axes[1, 5].axis("off")

    # Row 2: Iso-Jz
    axes[2, 0].imshow(disk_iso)
    _strip_axes(axes[2, 0])
    axes[2, 1].imshow(flow_iso);               axes[2, 1].axis("off")
    axes[2, 2].imshow(flow_iso_white);         axes[2, 2].axis("off")
    axes[2, 3].imshow(flow_iso_black);         axes[2, 3].axis("off")
    axes[2, 4].imshow(flow_iso_white_rot);     axes[2, 4].axis("off")
    axes[2, 5].imshow(flow_iso_black_rot);     axes[2, 5].axis("off")

    # Row 3: Balanced
    axes[3, 0].imshow(disk_bal)
    _strip_axes(axes[3, 0])
    axes[3, 1].imshow(flow_bal);               axes[3, 1].axis("off")
    axes[3, 2].imshow(flow_bal_white);         axes[3, 2].axis("off")
    axes[3, 3].imshow(flow_bal_black);         axes[3, 3].axis("off")
    axes[3, 4].imshow(flow_bal_white_rot);     axes[3, 4].axis("off")
    axes[3, 5].imshow(flow_bal_black_rot);     axes[3, 5].axis("off")

    # Row 4: sinebow
    axes[4, 0].imshow(disk_sine)
    _strip_axes(axes[4, 0])
    axes[4, 1].imshow(flow_sine);              axes[4, 1].axis("off")
    axes[4, 2].imshow(flow_sine_white);        axes[4, 2].axis("off")
    axes[4, 3].imshow(flow_sine_black);        axes[4, 3].axis("off")
    axes[4, 4].imshow(flow_sine_white_rot);    axes[4, 4].axis("off")
    axes[4, 5].imshow(flow_sine_black_rot);    axes[4, 5].axis("off")

    # Row 5: HSV
    axes[5, 0].imshow(disk_hsv)
    _strip_axes(axes[5, 0])
    axes[5, 1].imshow(flow_hsv);               axes[5, 1].axis("off")
    axes[5, 2].imshow(flow_hsv_white);         axes[5, 2].axis("off")
    axes[5, 3].imshow(flow_hsv_black);         axes[5, 3].axis("off")
    axes[5, 4].imshow(flow_hsv_white_rot);     axes[5, 4].axis("off")
    axes[5, 5].imshow(flow_hsv_black_rot);     axes[5, 5].axis("off")

    # y-axis labels on the disks, matching Plotly legend names
    for i, title in enumerate(row_titles):
        axes[i, 0].set_ylabel(title, fontsize=12)

    axes[0, 0].set_title("Hue disk (0°)")
    axes[0, 1].set_title("Flow → chroma (0°)")
    axes[0, 2].set_title("Alpha on white (0°)")
    axes[0, 3].set_title("Alpha on black (0°)")
    axes[0, 4].set_title("Alpha on white (+90°)")
    axes[0, 5].set_title("Alpha on black (+90°)")

    plt.tight_layout()
    plt.show()
if __name__ == "__main__":
    N_HUES = 72
    N_HARMONICS = 1

    visualize_fourier_hex_vs_sinebow_hsv_on_omnipose_jz(
        n_hues=N_HUES,
        n_harmonics=N_HARMONICS,
        device_str="cpu",
    )

In [ ]:
import numpy as np
import torch
import os
from pathlib import Path
from skimage import io
import fastremap
import matplotlib.pyplot as plt

import plotly.graph_objects as go
from plotly.offline import plot
from IPython.display import HTML

import omnirefactor

# Expect these helpers to exist in the environment:
#   hsv_to_rgb
#   srgb_to_xyz_np, xyz_to_srgb_np
#   jzazbz_to_xyz_np, xyz_to_jzazbz_np
#   srgb_to_linear_np
#   build_fourier_hsv_hex_ring_jz
#   make_flow_images_for_ring

dtype = torch.float64


# -------------------------------------------------------------------------
# Alignment helpers
# -------------------------------------------------------------------------

def find_reddest_index(rgb_ring_srgb: np.ndarray) -> int:
    """
    Return the index of the reddest color in an sRGB ring.
    Metric: R - 0.5*(G+B).
    """
    rgb = np.asarray(rgb_ring_srgb)
    r = rgb[:, 0]
    g = rgb[:, 1]
    b = rgb[:, 2]
    score = r - 0.5 * (g + b)
    return int(np.argmax(score))


def find_greenest_index(rgb_ring_srgb: np.ndarray) -> int:
    """
    Return the index of the greenest color in an sRGB ring.
    Metric: G - 0.5*(R+B).
    """
    rgb = np.asarray(rgb_ring_srgb)
    r = rgb[:, 0]
    g = rgb[:, 1]
    b = rgb[:, 2]
    score = g - 0.5 * (r + b)
    return int(np.argmax(score))


def align_ring_to_reference(rgb_ring_srgb: np.ndarray,
                            ref_green_idx: int) -> np.ndarray:
    """
    Align an sRGB ring to a canonical reference where:
      - reddest color is at index 0
      - greenest color lies in the same direction as ref_green_idx.
    """
    ring = np.asarray(rgb_ring_srgb)
    N = ring.shape[0]

    idx_red = find_reddest_index(ring)
    ring_r0 = np.roll(ring, -idx_red, axis=0)

    g1 = find_greenest_index(ring_r0)
    ring_rev = ring_r0[::-1].copy()
    g2 = find_greenest_index(ring_rev)

    def circ_dist(a, b):
        d = abs(a - b)
        return min(d, N - d)

    if circ_dist(g1, ref_green_idx) <= circ_dist(g2, ref_green_idx):
        return ring_r0
    else:
        return ring_rev


# -------------------------------------------------------------------------
# Iso-Jz ring
# -------------------------------------------------------------------------

def build_isoJ_from_fourier(jz_fourier: np.ndarray,
                            J0: float = 0.5) -> np.ndarray:
    """
    Build an iso-Jz ring from a JzAzBz Fourier ring.
    """
    jz_iso = jz_fourier.copy()
    jz_iso[:, 0] = J0

    jz_t = torch.from_numpy(jz_iso).to(dtype=dtype)
    center = jz_t.mean(dim=0)
    offsets = jz_t - center[None, :]

    def is_safe(scale: float) -> bool:
        jz_scaled = center[None, :] + scale * offsets
        jz_np = jz_scaled.cpu().numpy()
        xyz = jzazbz_to_xyz_np(jz_np)
        srgb = xyz_to_srgb_np(xyz)
        rgb_lin = srgb_to_linear_np(srgb)
        return np.all((rgb_lin >= -1e-6) & (rgb_lin <= 1.0 + 1e-6))

    s_lo, s_hi = 0.0, 1.0
    if not is_safe(1.0):
        for _ in range(40):
            mid = 0.5 * (s_lo + s_hi)
            if is_safe(float(mid)):
                s_lo = mid
            else:
                s_hi = mid
        s_opt = s_lo
    else:
        s_opt = 1.0

    jz_iso_safe_t = center[None, :] + s_opt * offsets
    jz_iso_safe = jz_iso_safe_t.cpu().numpy()
    xyz_iso_safe = jzazbz_to_xyz_np(jz_iso_safe)
    rgb_iso_srgb = np.clip(xyz_to_srgb_np(xyz_iso_safe), 0, 1)
    return rgb_iso_srgb


# -------------------------------------------------------------------------
# Salience versus gray backgrounds (global warp in J and C)
# -------------------------------------------------------------------------

def relative_luminance_from_linear(rgb_lin: np.ndarray) -> np.ndarray:
    """
    Compute relative luminance Y from linear sRGB.
    """
    rgb_lin = np.asarray(rgb_lin)
    r = rgb_lin[..., 0]
    g = rgb_lin[..., 1]
    b = rgb_lin[..., 2]
    return 0.2126 * r + 0.7152 * g + 0.0722 * b


def pc_contrast(Y_fg: float,
                Y_bg: float,
                exp: float = 0.4,
                eps: float = 1e-4) -> float:
    """
    Perceptual contrast-like metric between foreground and background
    luminances (Y_fg, Y_bg), using a symmetric power law.

        L = max(Y, eps)**exp
        PC = |L_fg - L_bg| / (L_fg + L_bg)
    """
    Y_fg = float(Y_fg)
    Y_bg = float(Y_bg)
    L_fg = max(Y_fg, eps) ** exp
    L_bg = max(Y_bg, eps) ** exp
    return abs(L_fg - L_bg) / (L_fg + L_bg)


def build_contrast_balanced_ring_from_fourier(
    jz_fourier: np.ndarray,
    C_min: float = 0.04,
    J_center_min: float = 0.45,
    J_center_max: float = 0.75,
    J_center_step: float = 0.02,
    C_scale_min: float = 0.6,
    C_scale_max: float = 1.35,
    C_scale_step: float = 0.05,
    w_mean: float = 1.0,
    w_varbg: float = 1.0,
    w_varhue: float = 0.3,
    w_chr: float = 0.6,
    w_cvar: float = 0.4,
    exp_pc: float = 0.4,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Build a contrast-balanced JzAzBz ring from a Fourier JzAzBz ring
    using a global warp:

        J'  = J_center + (J - J_mean)
        C'  = C_scale * C

    where C = sqrt(az^2 + bz^2), and az', bz' = C' * u, with u being the
    chroma direction from the original Fourier ring.

    Salience is integrated over a continuum of gray backgrounds, and
    variance across backgrounds and hues is penalized. Chroma is rewarded
    with a modest bonus. There is no explicit brightness preference.
    """
    jz_fourier = np.asarray(jz_fourier, dtype=np.float64)
    J_base = jz_fourier[:, 0]
    az_base = jz_fourier[:, 1]
    bz_base = jz_fourier[:, 2]

    C_base = np.hypot(az_base, bz_base)
    eps = 1e-6
    u_az = az_base / np.maximum(C_base, eps)
    u_bz = bz_base / np.maximum(C_base, eps)

    J_mean = float(J_base.mean())

    # Gray backgrounds for integration
    Y_bgs = np.linspace(0.0, 1.0, 13, dtype=np.float64)

    best_score = -1e9
    best_jz = None
    best_rgb = None

    J_centers = np.arange(J_center_min, J_center_max + 1e-9, J_center_step)
    C_scales = np.arange(C_scale_min, C_scale_max + 1e-9, C_scale_step)

    for J_center in J_centers:
        J_new = J_center + (J_base - J_mean)
        if np.any(J_new < 0.0) or np.any(J_new > 1.0):
            continue

        for C_scale in C_scales:
            C_new = C_scale * C_base
            if np.min(C_new) < C_min:
                continue

            az_new = C_new * u_az
            bz_new = C_new * u_bz
            jz_candidate = np.stack([J_new, az_new, bz_new], axis=1)

            xyz = jzazbz_to_xyz_np(jz_candidate)
            srgb = xyz_to_srgb_np(xyz)
            rgb_clip = np.clip(srgb, 0.0, 1.0)
            rgb_lin = srgb_to_linear_np(rgb_clip)

            # Reject candidate that leaves sRGB gamut
            if np.any(rgb_lin < -1e-4) or np.any(rgb_lin > 1.0 + 1e-4):
                continue

            Y_fg = relative_luminance_from_linear(rgb_lin)  # [N]

            # PC matrix [N_hues, N_bg]
            n_h = jz_candidate.shape[0]
            n_bg = Y_bgs.size
            pc_mat = np.empty((n_h, n_bg), dtype=np.float64)
            for j, Y_bg in enumerate(Y_bgs):
                for i in range(n_h):
                    pc_mat[i, j] = pc_contrast(float(Y_fg[i]), float(Y_bg), exp=exp_pc)

            mean_sal = float(pc_mat.mean())
            var_bg = float(pc_mat.var(axis=1).mean())   # per-hue variance over backgrounds
            var_hue = float(pc_mat.var(axis=0).mean())  # per-background variance over hues
            mean_C = float(C_new.mean())
            std_C = float(C_new.std())

            score = (
                w_mean * mean_sal
                - w_varbg * var_bg
                - w_varhue * var_hue
                + w_chr * mean_C
                - w_cvar * std_C
            )

            if score > best_score:
                best_score = score
                best_jz = jz_candidate
                best_rgb = rgb_clip

    if best_jz is None:
        # Fallback to the original Fourier ring
        best_jz = jz_fourier
        xyz = jzazbz_to_xyz_np(best_jz)
        best_rgb = np.clip(xyz_to_srgb_np(xyz), 0.0, 1.0)

    return best_rgb, best_jz
def build_contrast_balanced_ring_from_fourier(
    jz_fourier: np.ndarray,
    C_min: float = 0.04,
    J_center_min: float = 0.45,
    J_center_max: float = 0.75,
    J_center_step: float = 0.02,
    C_scale_min: float = 0.6,
    C_scale_max: float = 1.35,
    C_scale_step: float = 0.05,
    # salience weights
    w_mean: float = 1.0,
    w_varbg: float = 0.7,
    w_varhue: float = 0.3,
    w_bw: float = 1.0,
    # chroma priors
    w_chr: float = 0.4,
    w_cvar: float = 0.3,
    # HK brightness parameters
    k_HK: float = 0.30,
    gamma_HK: float = 0.80,
    exp_pc: float = 0.4,
    # backgrounds for salience (avoid true 0 / 1)
    bg_low: float = 0.12,
    bg_high: float = 0.88,
    n_bgs: int = 9,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Build a contrast-balanced JzAzBz ring from a Fourier JzAzBz ring
    using a global warp and an HK-aware salience metric.

    Parameterization (unchanged):

        J'(θ)  = J_center + (J_base(θ) - J_mean)
        C'(θ)  = C_scale * C_base(θ)
        az'(θ) = C'(θ) * u_az(θ)
        bz'(θ) = C'(θ) * u_bz(θ)

    where:
        C_base(θ) = sqrt(az_base^2 + bz_base^2),
        u_az, u_bz are the unit chroma direction in JzAzBz,
        J_center, C_scale are global scalars.

    HK-aware brightness model:

        B = Y + k_HK * C^gamma_HK

    where:
        Y  = relative luminance from linear sRGB,
        C  = sqrt(az'^2 + bz'^2) (opponent chroma in JzAzBz).

    Salience is computed on B against a band of gray backgrounds
    between `bg_low` and `bg_high` (e.g. 0.12–0.88), **not** vs pure
    black/white; this removes the strong downward bias.

    Objective (maximize):

        score =
            + w_mean   * mean_PC_B
            - w_varbg  * var_over_backgrounds_of_PC_B
            - w_varhue * var_over_hues_of_PC_B
            - w_bw     * mean(|PC_B_low - PC_B_high|)
            + w_chr    * mean_C
            - w_cvar   * std_C

    subject to:
        - candidate ring maps inside sRGB (no clipping),
        - minimum chroma C_min.
    """
    jz_fourier = np.asarray(jz_fourier, dtype=np.float64)
    J_base = jz_fourier[:, 0]
    az_base = jz_fourier[:, 1]
    bz_base = jz_fourier[:, 2]

    C_base = np.hypot(az_base, bz_base)
    eps = 1e-6
    u_az = az_base / np.maximum(C_base, eps)
    u_bz = bz_base / np.maximum(C_base, eps)

    J_mean = float(J_base.mean())

    # Gray backgrounds for integration in HK brightness space
    Y_bgs = np.linspace(bg_low, bg_high, n_bgs, dtype=np.float64)

    best_score = -1e9
    best_jz = None
    best_rgb = None

    J_centers = np.arange(J_center_min, J_center_max + 1e-9, J_center_step)
    C_scales = np.arange(C_scale_min, C_scale_max + 1e-9, C_scale_step)

    for J_center in J_centers:
        # global J warp
        J_new = J_center + (J_base - J_mean)
        if np.any(J_new < 0.0) or np.any(J_new > 1.0):
            continue

        for C_scale in C_scales:
            # global chroma warp
            C_new = C_scale * C_base
            if np.min(C_new) < C_min:
                continue

            az_new = C_new * u_az
            bz_new = C_new * u_bz
            jz_candidate = np.stack([J_new, az_new, bz_new], axis=1)

            # Convert to sRGB and linear RGB
            xyz = jzazbz_to_xyz_np(jz_candidate)
            srgb = xyz_to_srgb_np(xyz)

            # hard out-of-gamut reject (no clipping-based fix)
            if np.any(srgb < -1e-4) or np.any(srgb > 1.0 + 1e-4):
                continue

            srgb_clamped = np.clip(srgb, 0.0, 1.0)
            rgb_lin = srgb_to_linear_np(srgb_clamped)

            # relative luminance
            Y_fg = relative_luminance_from_linear(rgb_lin)  # [N]
            # opponent chroma in JzAzBz
            C_jz = np.hypot(az_new, bz_new)                # [N]

            # HK brightness correlate: B = Y + k_HK * C^gamma
            B_fg = Y_fg + k_HK * (C_jz ** gamma_HK)

            n_h = jz_candidate.shape[0]
            n_bg = Y_bgs.size
            pc_mat = np.empty((n_h, n_bg), dtype=np.float64)

            # salience across band of gray backgrounds (HK brightness)
            for j, Y_bg in enumerate(Y_bgs):
                B_bg = Y_bg  # gray backgrounds: C≈0 so B≈Y
                for i in range(n_h):
                    pc_mat[i, j] = pc_contrast(
                        float(B_fg[i]), float(B_bg), exp=exp_pc
                    )

            mean_sal = float(pc_mat.mean())
            var_bg = float(pc_mat.var(axis=1).mean())   # per-hue variance over bgs
            var_hue = float(pc_mat.var(axis=0).mean())  # per-bg variance over hues

            # salience vs "dark gray" and "light gray" (first/last backgrounds)
            pc_low = pc_mat[:, 0]       # vs bg_low
            pc_high = pc_mat[:, -1]     # vs bg_high
            bw_diff = float(np.mean(np.abs(pc_low - pc_high)))

            mean_C = float(C_new.mean())
            std_C = float(C_new.std())

            score = (
                w_mean   * mean_sal
                - w_varbg * var_bg
                - w_varhue * var_hue
                - w_bw    * bw_diff
                + w_chr   * mean_C
                - w_cvar  * std_C
            )

            if score > best_score:
                best_score = score
                best_jz = jz_candidate
                best_rgb = srgb_clamped

    if best_jz is None:
        # Fallback to the original Fourier ring
        best_jz = jz_fourier
        xyz = jzazbz_to_xyz_np(best_jz)
        best_rgb = np.clip(xyz_to_srgb_np(xyz), 0.0, 1.0)

    return best_rgb, best_jz


def build_contrast_balanced_ring_from_fourier(
    jz_fourier: np.ndarray,
    # J-search range
    J_center_min: float = 0.40,
    J_center_max: float = 0.90,
    J_center_step: float = 0.01,
    # Safety: 1.0 = touch the wall exactly. 0.98 = 2% buffer.
    safety_margin: float = 0.98, 
    # These args are kept for compatibility but IGNORED now
    C_min: float = 0.0, 
    w_mean: float = 0, w_min: float = 0, w_bg: float = 0, 
    w_hue: float = 0, w_chr: float = 0, exp_pc: float = 0,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Optimizes the Fourier ring by finding the Lightness (J) plane 
    that strictly MAXIMIZES the ring's scale (perimeter).
    
    It ignores contrast/background scoring entirely.
    """
    # 1. Analyze the input shape (direction vectors)
    jz_fourier = np.asarray(jz_fourier, dtype=np.float64)
    J_base = jz_fourier[:, 0]
    az_base = jz_fourier[:, 1]
    bz_base = jz_fourier[:, 2]
    J_mean_base = float(J_base.mean())

    C_base = np.hypot(az_base, bz_base)
    eps = 1e-9
    u_az = az_base / np.maximum(C_base, eps)
    u_bz = bz_base / np.maximum(C_base, eps)

    best_scale = -1.0
    best_jz = None
    best_rgb = None

    J_centers = np.arange(J_center_min, J_center_max + 1e-9, J_center_step)

    for J_center in J_centers:
        # Shift Lightness
        J_new = J_center + (J_base - J_mean_base)
        
        # Skip if any part of the ring leaves absolute J bounds [0,1]
        if np.any(J_new < 0.0) or np.any(J_new > 1.0):
            continue

        # Scan geometric distance to gamut wall for this J-ring
        C_limits = find_max_chroma_binary_search(J_new, u_az, u_bz, steps=15)
        
        # Find the global scale factor that fits the whole ring inside
        # S = min( Limit / Shape )
        ratios = C_limits / np.maximum(C_base, 1e-6)
        max_safe_scale = np.min(ratios)
        
        # Optimization Target: Pure Size
        if max_safe_scale > best_scale:
            best_scale = max_safe_scale
            
            # Construct the winner
            scale_final = max_safe_scale * safety_margin
            C_final = scale_final * C_base
            
            az_final = C_final * u_az
            bz_final = C_final * u_bz
            
            best_jz = np.stack([J_new, az_final, bz_final], axis=1)
            
            # Convert to RGB for output
            xyz = jzazbz_to_xyz_np(best_jz)
            srgb = xyz_to_srgb_np(xyz)
            best_rgb = np.clip(srgb, 0.0, 1.0)

    if best_jz is None:
        print("Warning: optimization failed. Returning base.")
        best_jz = jz_fourier
        xyz = jzazbz_to_xyz_np(best_jz)
        best_rgb = np.clip(xyz_to_srgb_np(xyz), 0.0, 1.0)

    return best_rgb, best_jz



# -------------------------------------------------------------------------
# Optimized "Largest Inscribed Ring" Solver
# -------------------------------------------------------------------------

def get_max_scale_for_center(J_val: float,
                             c_az: float,
                             c_bz: float,
                             u_az: np.ndarray,
                             u_bz: np.ndarray,
                             steps: int = 10) -> float:
    """
    Given a fixed center (J, c_az, c_bz) and a shape defined by unit vectors (u_az, u_bz),
    find the maximum scale factor S such that the shape stays in gamut.
    """
    # Broadcast scalars to array size matching u_az
    J_arr = np.full_like(u_az, J_val)
    c_az_arr = np.full_like(u_az, c_az)
    c_bz_arr = np.full_like(u_az, c_bz)
    
    low = np.zeros_like(u_az)
    high = np.ones_like(u_az) * 1.5 # Max reasonable scale
    
    # Quick check: is center itself valid?
    jz_center = np.stack([J_arr, c_az_arr, c_bz_arr], axis=1)
    xyz = jzazbz_to_xyz_np(jz_center)
    lin = srgb_to_linear_np(xyz_to_srgb_np(xyz))
    if np.any(lin < -1e-4) or np.any(lin > 1.0 + 1e-4):
        return 0.0

    # Binary search for the wall in every direction
    for _ in range(steps):
        mid = (low + high) * 0.5
        
        # Test point: Center + Scale * Direction
        az_test = c_az_arr + mid * u_az
        bz_test = c_bz_arr + mid * u_bz
        
        jz_test = np.stack([J_arr, az_test, bz_test], axis=1)
        
        xyz = jzazbz_to_xyz_np(jz_test)
        rgb_lin = srgb_to_linear_np(xyz_to_srgb_np(xyz))
        
        in_gamut = np.all((rgb_lin > -1e-4) & (rgb_lin < 1.0 + 1e-4), axis=1)
        
        low = np.where(in_gamut, mid, low)
        high = np.where(in_gamut, high, mid)

    # The max safe scale for the *whole* ring is the minimum of the individual ray limits
    return float(np.min(low))


def build_contrast_balanced_ring_from_fourier(
    jz_fourier: np.ndarray,
    J_center_min: float = 0.45,
    J_center_max: float = 0.85,
    J_center_step: float = 0.02,  # Coarser step for J is okay
    grid_search_range: float = 0.04, # Look +/- 0.04 in Az/Bz
    grid_search_res: int = 7,     # 7x7 grid = 49 checks per J
    safety_margin: float = 0.99,  # Very slight buffer
    # Ignored compatibility args
    C_min=None, w_mean=None, w_min=None, w_bg=None, w_hue=None, w_chr=None, exp_pc=None,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Finds the Largest Inscribed Fourier Ring by optimizing:
    1. Lightness (J)
    2. Center Position (Az, Bz)
    
    It performs a grid search at every J level to wiggle the center
    until the ring fits evenly against the walls.
    """
    # 1. Analyze Shape
    jz_fourier = np.asarray(jz_fourier, dtype=np.float64)
    az_base = jz_fourier[:, 1]
    bz_base = jz_fourier[:, 2]

    # Normalize input shape to unit vectors + base magnitude
    # We treat the input "shape" as the relative proportions of the harmonics.
    # C_base[i] tells us that direction i is "supposed" to be longer/shorter.
    C_base = np.hypot(az_base, bz_base)
    eps = 1e-9
    u_az = az_base / np.maximum(C_base, eps)
    u_bz = bz_base / np.maximum(C_base, eps)

    best_global_scale = -1.0
    best_jz = None
    best_rgb = None

    # 2. Optimization Loop
    J_centers = np.arange(J_center_min, J_center_max + 1e-9, J_center_step)
    
    # Create the grid of offsets once
    grid_1d = np.linspace(-grid_search_range, grid_search_range, grid_search_res)
    offset_az, offset_bz = np.meshgrid(grid_1d, grid_1d)
    offsets = np.stack([offset_az.ravel(), offset_bz.ravel()], axis=1)

    print(f"Optimizing Ring: Scanning {len(J_centers)} J-planes x {len(offsets)} positions...")

    for J_curr in J_centers:
        
        # For this Lightness level, check all grid positions
        for (c_az, c_bz) in offsets:
            
            # Find the "Bottleneck Scale" for this specific J and Center.
            # The function returns the scale applied to the UNIT vectors.
            # We need to normalize relative to C_base to keep the shape aspect ratio.
            
            # Ray cast to find absolute max distance in every direction
            # Note: We pass u_az, u_bz. The function returns the uniform scalar limit.
            # But our shape is NOT a circle. It varies by C_base.
            # So we need to know if (C_base * S) fits.
            
            # Let's perform the binary search directly on the "Shape Scale" here
            # to be precise.
            
            # --- Inner Binary Search for Scale 'S' ---
            low = 0.0
            high = 2.0 # Generous upper bound
            
            # Check validity of the center first
            jz_c = np.array([[J_curr, c_az, c_bz]])
            xyz_c = jzazbz_to_xyz_np(jz_c)
            lin_c = srgb_to_linear_np(xyz_to_srgb_np(xyz_c))
            if np.any(lin_c < 0) or np.any(lin_c > 1):
                continue

            # Quick 8-step binary search for the scalar S
            for _ in range(10):
                mid = (low + high) * 0.5
                
                # Apply Shape: Center + (S * Shape_Magnitude * Direction)
                # This preserves the harmonic shape!
                az_t = c_az + mid * C_base * u_az
                bz_t = c_bz + mid * C_base * u_bz
                
                jz_t = np.stack([np.full_like(az_t, J_curr), az_t, bz_t], axis=1)
                xyz_t = jzazbz_to_xyz_np(jz_t)
                lin_t = srgb_to_linear_np(xyz_to_srgb_np(xyz_t))
                
                if np.all((lin_t > -1e-4) & (lin_t < 1.0 + 1e-4)):
                    low = mid # This scale fits
                else:
                    high = mid # Too big
            
            current_scale = low
            
            # Optimization: Maximize Scale
            if current_scale > best_global_scale:
                best_global_scale = current_scale
                
                # Build the winner
                scale_final = current_scale * safety_margin
                az_final = c_az + scale_final * C_base * u_az
                bz_final = c_bz + scale_final * C_base * u_bz
                J_final = np.full_like(az_final, J_curr)
                
                best_jz = np.stack([J_final, az_final, bz_final], axis=1)

    if best_jz is None:
        print("Optimization failed (no valid center found).")
        return build_contrast_balanced_ring_from_fourier(jz_fourier, grid_search_range=0)

    # Final conversion
    xyz = jzazbz_to_xyz_np(best_jz)
    srgb = xyz_to_srgb_np(xyz)
    best_rgb = np.clip(srgb, 0.0, 1.0)
    
    print(f"Optimal Ring Found: J={best_jz[0,0]:.2f}, Scale={best_global_scale:.3f}")
    return best_rgb, best_jz


# -------------------------------------------------------------------------
# 3D Rigid Loop Optimizer
# -------------------------------------------------------------------------

def find_max_scale_for_3d_loop(
    J_current: np.ndarray,     # The loop's J coordinates (shifted)
    c_az: float,               # The loop's global center az
    c_bz: float,               # The loop's global center bz
    vec_az: np.ndarray,        # The shape vectors (from center to point)
    vec_bz: np.ndarray,
    steps: int = 12
) -> float:
    """
    Determines the maximum uniform Scale 'S' allowed for a 3D loop before
    any single point hits the sRGB gamut wall.
    
    Point[i] = (J[i], c_az + S*vec_az[i], c_bz + S*vec_bz[i])
    """
    # Bounds for Scale S
    low = 0.0
    high = 5.0  # Large upper bound to allow significant expansion
    
    # 1. Check if the "skeleton" (scale=0) is valid. 
    # If the centers/Js themselves are OOB, scale 0 is the limit.
    jz_center = np.stack([J_current, np.full_like(J_current, c_az), np.full_like(J_current, c_bz)], axis=1)
    xyz_c = jzazbz_to_xyz_np(jz_center)
    lin_c = srgb_to_linear_np(xyz_to_srgb_np(xyz_c))
    if np.any(lin_c < -1e-4) or np.any(lin_c > 1.0 + 1e-4):
        return 0.0

    # 2. Binary search for the global Scale limit
    for _ in range(steps):
        mid = (low + high) * 0.5
        
        # Apply scale to the vectors
        az_test = c_az + mid * vec_az
        bz_test = c_bz + mid * vec_bz
        
        jz_test = np.stack([J_current, az_test, bz_test], axis=1)
        
        xyz = jzazbz_to_xyz_np(jz_test)
        # We check Linear RGB for strict geometric containment
        srgb = xyz_to_srgb_np(xyz)
        lin = srgb_to_linear_np(srgb)
        
        # "Are ALL points in the loop valid at this scale?"
        is_valid = np.all((lin >= -1e-4) & (lin <= 1.0 + 1e-4))
        
        if is_valid:
            low = mid
        else:
            high = mid
            
    return low


# def build_contrast_balanced_ring_from_fourier(
#     jz_fourier: np.ndarray,
#     # Search Parameters
#     J_shift_range: float = 0.15,   # How far up/down to wiggle the loop
#     J_steps: int = 15,
#     C_shift_range: float = 0.05,   # How far sideways (az/bz) to wiggle center
#     C_steps: int = 7,              # Grid resolution
#     safety_margin: float = 0.99,   # Stay just inside the wall
#     # Ignored args
#     C_min=None, J_center_min=None, J_center_max=None, J_center_step=None,
#     w_mean=None, w_min=None, w_bg=None, w_hue=None, w_chr=None, exp_pc=None,
# ) -> tuple[np.ndarray, np.ndarray]:
#     """
#     Rigidly transforms the input Fourier loop to maximize its size inside sRGB.
    
#     1. Centers the input loop geometry.
#     2. Scans 3D space (dJ, dAz, dBz) for the optimal position.
#     3. Scales the loop uniformly until it touches the gamut boundary.
#     """
#     # 1. Decompose Input Shape relative to its own center
#     jz_fourier = np.asarray(jz_fourier, dtype=np.float64)
#     J_orig = jz_fourier[:, 0]
#     az_orig = jz_fourier[:, 1]
#     bz_orig = jz_fourier[:, 2]
    
#     # Remove offsets to get the "Wireframe Shape"
#     J_mean_orig = np.mean(J_orig)
#     az_mean_orig = np.mean(az_orig)
#     bz_mean_orig = np.mean(bz_orig)
    
#     # The relative shape vectors
#     J_rel = J_orig - J_mean_orig
#     vec_az = az_orig - az_mean_orig
#     vec_bz = bz_orig - bz_mean_orig
    
#     # Normalize shape vectors so Scale=1.0 means "Original Size"
#     # (Actually we don't strictly need to normalize, Scale will just adjust relative to input)

#     best_scale = -1.0
#     best_jz = None
#     best_rgb = None
    
#     # 2. Define Search Grid
#     # Search around the original lightness center (or you can hardcode a starting center)
#     J_shifts = np.linspace(-J_shift_range, J_shift_range, J_steps)
    
#     # Search for center offset in chroma plane
#     grid_c = np.linspace(-C_shift_range, C_shift_range, C_steps)
#     c_az_grid, c_bz_grid = np.meshgrid(grid_c, grid_c)
#     offsets = np.stack([c_az_grid.ravel(), c_bz_grid.ravel()], axis=1)
    
#     print(f"Optimizing 3D Loop: Scanning {len(J_shifts)} J-planes x {len(offsets)} centers...")

#     for dJ in J_shifts:
        
#         # Current J geometry
#         J_current = (J_mean_orig + dJ) + J_rel
        
#         # Immediate reject if the shape itself is vertically out of bounds
#         if np.any(J_current < 0.0) or np.any(J_current > 1.0):
#             continue

#         for (dAz, dBz) in offsets:
#             # Current Chroma Center
#             # Note: JzAzBz neutral axis is (0,0). We search around 0.
#             # We ignore the original az/bz mean because it might be biased. 
#             # We want the geometric sweet spot.
#             curr_c_az = dAz
#             curr_c_bz = dBz
            
#             # 3. Find Max Scale for this configuration
#             # This replaces the complex per-pixel raycast with a global shape check
#             max_scale = find_max_scale_for_3d_loop(
#                 J_current, curr_c_az, curr_c_bz, vec_az, vec_bz, steps=10
#             )
            
#             if max_scale > best_scale:
#                 best_scale = max_scale
                
#                 # Construct Winner
#                 final_scale = max_scale * safety_margin
                
#                 az_final = curr_c_az + final_scale * vec_az
#                 bz_final = curr_c_bz + final_scale * vec_bz
                
#                 # Store
#                 best_jz = np.stack([J_current, az_final, bz_final], axis=1)

#     # 4. Output
#     if best_jz is None:
#         print("Optimization failed (shape too large or OOB?). Returning original.")
#         best_jz = jz_fourier
#     else:
#         print(f"Optimal Loop: dJ={best_jz[:,0].mean()-J_mean_orig:.3f}, Scale={best_scale:.3f}")

#     xyz = jzazbz_to_xyz_np(best_jz)
#     srgb = xyz_to_srgb_np(xyz)
#     # Clip is safe here because we ensured validity geometrically
#     best_rgb = np.clip(srgb, 0.0, 1.0)

#     return best_rgb, best_jz




# -------------------------------------------------------------------------
# Plotly 3D trajectory with warped sRGB cube surfaces
# -------------------------------------------------------------------------

def _srgb_face_to_jz_surface(n: int,
                             fixed_channel: int,
                             fixed_value: float) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Build a JzAzBz surface for one face of the sRGB cube.

    fixed_channel: 0=R, 1=G, 2=B
    fixed_value:   0.0 or 1.0
    """
    grid = np.linspace(0.0, 1.0, n)
    G, B = np.meshgrid(grid, grid)

    if fixed_channel == 0:
        R = np.full_like(G, fixed_value)
        srgb = np.stack([R, G, B], axis=-1)
    elif fixed_channel == 1:
        Gf = np.full_like(G, fixed_value)
        srgb = np.stack([G, Gf, B], axis=-1)
    else:
        Bf = np.full_like(G, fixed_value)
        srgb = np.stack([G, B, Bf], axis=-1)

    xyz = srgb_to_xyz_np(srgb.reshape(-1, 3))
    jz = xyz_to_jzazbz_np(xyz).reshape(srgb.shape)
    J = jz[..., 0]
    az = jz[..., 1]
    bz = jz[..., 2]
    return az, bz, J


def plot_jz_trajectory_hex_fourier_sine_hsv(
    jz_fourier: np.ndarray, rgb_fourier: np.ndarray,
    jz_arc: np.ndarray,      rgb_arc: np.ndarray,
    jz_sine: np.ndarray,     rgb_sine: np.ndarray,
    jz_hsv: np.ndarray,      rgb_hsv: np.ndarray,
    jz_iso: np.ndarray,      rgb_iso: np.ndarray,
    jz_bal: np.ndarray,      rgb_bal: np.ndarray,
):
    """
    Plot JzAzBz trajectories for several rings plus warped sRGB cube faces
    using Plotly in dark mode.

    Axes:
        x = az
        y = bz
        z = Jz
    """
    fig = go.Figure()

    # sRGB cube faces as translucent surfaces
    n_face = 17
    face_specs = [
        (0, 0.0, "R=0"),
        (0, 1.0, "R=1"),
        (1, 0.0, "G=0"),
        (1, 1.0, "G=1"),
        (2, 0.0, "B=0"),
        (2, 1.0, "B=1"),
    ]
    for fixed_channel, fixed_value, name in face_specs:
        az, bz, J = _srgb_face_to_jz_surface(n_face, fixed_channel, fixed_value)
        fig.add_trace(
            go.Surface(
                x=az,
                y=bz,
                z=J,
                opacity=0.18,
                showscale=False,
                colorscale="Viridis",
                name=name,
                hoverinfo="skip",
            )
        )

    def add_curve(jz, name, width=7):
        jz = np.asarray(jz)
        fig.add_trace(
            go.Scatter3d(
                x=jz[:, 1],
                y=jz[:, 2],
                z=jz[:, 0],
                mode="lines",
                name=name,
                line=dict(width=width),
            )
        )

    # Order here matches row labels in the grid figure
    add_curve(jz_fourier, "Fourier",        width=7)
    add_curve(jz_arc,      "Arc-length HSV", width=5)
    add_curve(jz_iso,      "Iso-Jz",        width=5)
    add_curve(jz_bal,      "Balanced",      width=8)
    add_curve(jz_sine,     "sinebow",       width=4)
    add_curve(jz_hsv,      "HSV",           width=4)

    fig.update_layout(
        template="plotly_dark",
        paper_bgcolor="black",
        scene=dict(
            xaxis_title="az",
            yaxis_title="bz",
            zaxis_title="Jz",
            bgcolor="black",
            aspectmode="data",
        ),
        margin=dict(l=0, r=0, b=0, t=40),
        title="JzAzBz trajectories with warped sRGB cube and balanced ring",
        showlegend=True,
    )

    html = plot(fig, output_type="div", include_plotlyjs="cdn")
    return HTML(html)


# -------------------------------------------------------------------------
# Main visualization using the new balanced ring and plotting fixes
# -------------------------------------------------------------------------

def visualize_fourier_hex_vs_sinebow_hsv_on_omnipose_jz(
    n_hues: int = 72,
    n_harmonics: int = 1,
    device_str: str = "cpu",
):
    dev = torch.device(device_str)

    # 1) base HSV wheel (smooth) in sRGB & JzAzBz
    t = np.linspace(0, 1, n_hues, endpoint=False)
    hsv_vals = np.stack([t, np.ones_like(t), np.ones_like(t)], axis=1)
    rgb_hsv_srgb_raw = hsv_to_rgb(hsv_vals)
    xyz_hsv_raw = srgb_to_xyz_np(rgb_hsv_srgb_raw)
    jz_hsv_raw = xyz_to_jzazbz_np(xyz_hsv_raw)

    # sinebow
    sine_r = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0 / 3))
    sine_g = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1 / 3))
    sine_b = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2 / 3))
    rgb_sine_srgb_raw = np.stack([sine_r, sine_g, sine_b], axis=1)
    xyz_sine_raw = srgb_to_xyz_np(rgb_sine_srgb_raw)
    jz_sine_raw = xyz_to_jzazbz_np(xyz_sine_raw)

    # 2) Fourier-smoothed HSV hex colormap in JzAzBz (raw)
    jz_fourier_raw, rgb_fourier_srgb_raw, jz_arc_raw, rgb_arc_srgb_raw = build_fourier_hsv_hex_ring_jz(
        n_hues=n_hues,
        n_harmonics=n_harmonics,
        oversample=8,
    )

    # Canonical alignment based on HSV
    idx_red_h = find_reddest_index(rgb_hsv_srgb_raw)
    rgb_hsv_r0 = np.roll(rgb_hsv_srgb_raw, -idx_red_h, axis=0)
    g_h = find_greenest_index(rgb_hsv_r0)
    if g_h > n_hues // 2:
        rgb_hsv_r0 = rgb_hsv_r0[::-1].copy()
        g_h = find_greenest_index(rgb_hsv_r0)
    rgb_hsv_srgb = rgb_hsv_r0
    jz_hsv = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_hsv_srgb))

    rgb_arc_r0 = np.roll(rgb_arc_srgb_raw, -idx_red_h, axis=0)
    if g_h > n_hues // 2:
        rgb_arc_r0 = rgb_arc_r0[::-1].copy()
    rgb_arc_srgb = rgb_arc_r0
    jz_arc = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_arc_srgb))

    rgb_fourier_srgb = align_ring_to_reference(rgb_fourier_srgb_raw, ref_green_idx=g_h)
    rgb_sine_srgb = align_ring_to_reference(rgb_sine_srgb_raw, ref_green_idx=g_h)

    jz_fourier = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_fourier_srgb))
    jz_sine = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_sine_srgb))

    # Iso-Jz ring
    rgb_iso_srgb_raw = build_isoJ_from_fourier(jz_fourier, J0=0.5)
    rgb_iso_srgb = align_ring_to_reference(rgb_iso_srgb_raw, ref_green_idx=g_h)
    jz_iso = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_iso_srgb))

    # Contrast-balanced ring via global warp and integrated salience
    # rgb_bal_srgb_raw, jz_bal_raw = build_contrast_balanced_ring_from_fourier(
    #     jz_fourier,
    #     C_min=0.05,
    #     J_center_min=0.65,
    #     J_center_max=1,
    #     J_center_step=0.02,
    #     C_scale_min=0.6,
    #     C_scale_max=1.5,
    #     C_scale_step=0.05,
    # )

    rgb_bal_srgb_raw, jz_bal_raw = build_contrast_balanced_ring_from_fourier(
        jz_fourier,
        C_min=0.05,
        J_center_min=0.0,
        J_center_max=2,
        J_center_step=0.01,
        safety_margin=1,  # <--- Replaces C_scale params
    )
    
    rgb_bal_srgb = align_ring_to_reference(rgb_bal_srgb_raw, ref_green_idx=g_h)
    jz_bal = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_bal_srgb))

    # Plotly JzAzBz figure with cube surfaces and all rings
    traj_html = plot_jz_trajectory_hex_fourier_sine_hsv(
        jz_fourier, rgb_fourier_srgb,
        jz_arc, rgb_arc_srgb,
        jz_sine, rgb_sine_srgb,
        jz_hsv, rgb_hsv_srgb,
        jz_iso, rgb_iso_srgb,
        jz_bal, rgb_bal_srgb,
    )
    display(traj_html)

    # Convert rings to linear RGB for disks/flows
    rgb_fourier_lin = srgb_to_linear_np(rgb_fourier_srgb)
    rgb_arc_lin = srgb_to_linear_np(rgb_arc_srgb)
    rgb_iso_lin = srgb_to_linear_np(rgb_iso_srgb)
    rgb_bal_lin = srgb_to_linear_np(rgb_bal_srgb)
    rgb_sine_lin = srgb_to_linear_np(rgb_sine_srgb)
    rgb_hsv_lin = srgb_to_linear_np(rgb_hsv_srgb)

    center_srgb = np.array([0.5, 0.5, 0.5], dtype=float)
    center_lin = srgb_to_linear_np(center_srgb)

    # Omnipose E. coli flows
    omnidir = Path(omnirefactor.__file__).parent.parent.parent
    basedir = os.path.join(omnidir, "docs", "_static")
    name = "ecoli"
    ext = ".tif"

    image = io.imread(os.path.join(basedir, name + ext))
    masks = io.imread(os.path.join(basedir, name + "_labels" + ext))
    slc = ocdkit.measure.crop_bbox(masks > 0, pad=0)[0]
    masks = fastremap.renumber(masks[slc])[0]
    image = image[slc]

    flows_all = omnirefactor.core.masks_to_flows(masks, device="cpu")[4].to(dev)
    flows = flows_all

    # Build images for each colormap (0° phase)
    disk_fourier, flow_fourier, flow_fourier_white, flow_fourier_black = make_flow_images_for_ring(
        rgb_fourier_lin, center_lin, flows
    )
    disk_arc, flow_arc, flow_arc_white, flow_arc_black = make_flow_images_for_ring(
        rgb_arc_lin, center_lin, flows
    )
    disk_iso, flow_iso, flow_iso_white, flow_iso_black = make_flow_images_for_ring(
        rgb_iso_lin, center_lin, flows
    )
    disk_bal, flow_bal, flow_bal_white, flow_bal_black = make_flow_images_for_ring(
        rgb_bal_lin, center_lin, flows
    )
    disk_sine, flow_sine, flow_sine_white, flow_sine_black = make_flow_images_for_ring(
        rgb_sine_lin, center_lin, flows
    )
    disk_hsv, flow_hsv, flow_hsv_white, flow_hsv_black = make_flow_images_for_ring(
        rgb_hsv_lin, center_lin, flows
    )

    # Rotated (+90°) colormaps
    shift = n_hues // 4

    rgb_fourier_lin_rot = np.roll(rgb_fourier_lin, shift, axis=0)
    rgb_arc_lin_rot = np.roll(rgb_arc_lin, shift, axis=0)
    rgb_iso_lin_rot = np.roll(rgb_iso_lin, shift, axis=0)
    rgb_bal_lin_rot = np.roll(rgb_bal_lin, shift, axis=0)
    rgb_sine_lin_rot = np.roll(rgb_sine_lin, shift, axis=0)
    rgb_hsv_lin_rot = np.roll(rgb_hsv_lin, shift, axis=0)

    _, _, flow_fourier_white_rot, flow_fourier_black_rot = make_flow_images_for_ring(
        rgb_fourier_lin_rot, center_lin, flows
    )
    _, _, flow_arc_white_rot, flow_arc_black_rot = make_flow_images_for_ring(
        rgb_arc_lin_rot, center_lin, flows
    )
    _, _, flow_iso_white_rot, flow_iso_black_rot = make_flow_images_for_ring(
        rgb_iso_lin_rot, center_lin, flows
    )
    _, _, flow_bal_white_rot, flow_bal_black_rot = make_flow_images_for_ring(
        rgb_bal_lin_rot, center_lin, flows
    )
    _, _, flow_sine_white_rot, flow_sine_black_rot = make_flow_images_for_ring(
        rgb_sine_lin_rot, center_lin, flows
    )
    _, _, flow_hsv_white_rot, flow_hsv_black_rot = make_flow_images_for_ring(
        rgb_hsv_lin_rot, center_lin, flows
    )

    # Unified figure: 6 rows × 6 columns, with fixed row labels
    fig, axes = plt.subplots(6, 6, figsize=(26, 14))
    row_titles = [
        "Fourier",
        "Arc-length HSV",
        "Iso-Jz",
        "Balanced",
        "sinebow",
        "HSV",
    ]

    def _strip_axes(ax):
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)

    # Row 0: Fourier
    axes[0, 0].imshow(disk_fourier); _strip_axes(axes[0, 0])
    axes[0, 1].imshow(flow_fourier);           axes[0, 1].axis("off")
    axes[0, 2].imshow(flow_fourier_white);     axes[0, 2].axis("off")
    axes[0, 3].imshow(flow_fourier_black);     axes[0, 3].axis("off")
    axes[0, 4].imshow(flow_fourier_white_rot); axes[0, 4].axis("off")
    axes[0, 5].imshow(flow_fourier_black_rot); axes[0, 5].axis("off")

    # Row 1: Arc-length HSV
    axes[1, 0].imshow(disk_arc); _strip_axes(axes[1, 0])
    axes[1, 1].imshow(flow_arc);               axes[1, 1].axis("off")
    axes[1, 2].imshow(flow_arc_white);         axes[1, 2].axis("off")
    axes[1, 3].imshow(flow_arc_black);         axes[1, 3].axis("off")
    axes[1, 4].imshow(flow_arc_white_rot);     axes[1, 4].axis("off")
    axes[1, 5].imshow(flow_arc_black_rot);     axes[1, 5].axis("off")

    # Row 2: Iso-Jz
    axes[2, 0].imshow(disk_iso); _strip_axes(axes[2, 0])
    axes[2, 1].imshow(flow_iso);               axes[2, 1].axis("off")
    axes[2, 2].imshow(flow_iso_white);         axes[2, 2].axis("off")
    axes[2, 3].imshow(flow_iso_black);         axes[2, 3].axis("off")
    axes[2, 4].imshow(flow_iso_white_rot);     axes[2, 4].axis("off")
    axes[2, 5].imshow(flow_iso_black_rot);     axes[2, 5].axis("off")

    # Row 3: Balanced
    axes[3, 0].imshow(disk_bal); _strip_axes(axes[3, 0])
    axes[3, 1].imshow(flow_bal);               axes[3, 1].axis("off")
    axes[3, 2].imshow(flow_bal_white);         axes[3, 2].axis("off")
    axes[3, 3].imshow(flow_bal_black);         axes[3, 3].axis("off")
    axes[3, 4].imshow(flow_bal_white_rot);     axes[3, 4].axis("off")
    axes[3, 5].imshow(flow_bal_black_rot);     axes[3, 5].axis("off")

    # Row 4: sinebow
    axes[4, 0].imshow(disk_sine); _strip_axes(axes[4, 0])
    axes[4, 1].imshow(flow_sine);              axes[4, 1].axis("off")
    axes[4, 2].imshow(flow_sine_white);        axes[4, 2].axis("off")
    axes[4, 3].imshow(flow_sine_black);        axes[4, 3].axis("off")
    axes[4, 4].imshow(flow_sine_white_rot);    axes[4, 4].axis("off")
    axes[4, 5].imshow(flow_sine_black_rot);    axes[4, 5].axis("off")

    # Row 5: HSV
    axes[5, 0].imshow(disk_hsv); _strip_axes(axes[5, 0])
    axes[5, 1].imshow(flow_hsv);               axes[5, 1].axis("off")
    axes[5, 2].imshow(flow_hsv_white);         axes[5, 2].axis("off")
    axes[5, 3].imshow(flow_hsv_black);         axes[5, 3].axis("off")
    axes[5, 4].imshow(flow_hsv_white_rot);     axes[5, 4].axis("off")
    axes[5, 5].imshow(flow_hsv_black_rot);     axes[5, 5].axis("off")

    # y-axis labels on the disks, matching Plotly legend names
    for i, title in enumerate(row_titles):
        axes[i, 0].set_ylabel(title, fontsize=12)

    axes[0, 0].set_title("Hue disk (0°)")
    axes[0, 1].set_title("Flow → chroma (0°)")
    axes[0, 2].set_title("Alpha on white (0°)")
    axes[0, 3].set_title("Alpha on black (0°)")
    axes[0, 4].set_title("Alpha on white (+90°)")
    axes[0, 5].set_title("Alpha on black (+90°)")

    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    N_HUES = 72
    N_HARMONICS = 1

    visualize_fourier_hex_vs_sinebow_hsv_on_omnipose_jz(
        n_hues=N_HUES,
        n_harmonics=N_HARMONICS,
        device_str="cpu",
    )

In [ ]:
the above looks perfect at first, but notice that the 90 degree roated one has much more difference in blue over tis face than the 
prupele region, idication fdelta E is not very good here. need to ensure cosntant sampling 
and also mayeb the color place is the fault here
    shoudl try reparametizing sinebow

In [ ]:
import numpy as np
import torch
import os
from pathlib import Path
from skimage import io
import fastremap
import matplotlib.pyplot as plt

import plotly.graph_objects as go
from plotly.offline import plot
from IPython.display import HTML

import omnirefactor

dtype = torch.float64

# ----------------------------------------------------------------------
# Hellwig 2022 helpers using colour-science
# ----------------------------------------------------------------------
try:
    import colour
    from colour.appearance import (
        CAM_Specification_Hellwig2022,
        XYZ_to_Hellwig2022,
        Hellwig2022_to_XYZ,
    )
except ImportError as e:
    raise ImportError("colour-science package required: pip install colour-science") from e


def _hellwig_viewing_conditions():
    """
    Standard viewing conditions for Hellwig 2022 JMh.
    Matches D65 sRGB white, average surround.
    """
    XYZ_w = srgb_to_xyz_np(np.array([[1.0, 1.0, 1.0]]))[0]  # sRGB white
    Y_b = 20.0
    L_A = 64.0 / np.pi
    surround = colour.appearance.hellwig2022.InductionFactors_Hellwig2022(
        F=1.0, c=0.69, N_c=1.0
    )
    return XYZ_w, Y_b, L_A, surround


_XYZ_w_H, _Y_b_H, _L_A_H, _surround_H = _hellwig_viewing_conditions()


def srgb_to_JMh_hellwig_np(rgb_srgb: np.ndarray) -> np.ndarray:
    """
    Convert sRGB [N,3] to Hellwig 2022 JMh [N,3] under fixed conditions.
    """
    rgb_srgb = np.asarray(rgb_srgb, dtype=np.float64)
    XYZ = srgb_to_xyz_np(rgb_srgb)
    spec = XYZ_to_Hellwig2022(
        XYZ,
        _XYZ_w_H,
        _L_A_H,
        _Y_b_H,
        surround=_surround_H,
        discount_illuminant=False,
    )
    # spec.J, spec.M, spec.h are arrays
    J = np.asarray(spec.J, dtype=np.float64)
    M = np.asarray(spec.M, dtype=np.float64)
    h = np.asarray(spec.h, dtype=np.float64)
    return np.stack([J, M, h], axis=-1)


def JMh_hellwig_to_srgb_np(JMh: np.ndarray) -> np.ndarray:
    """
    Convert Hellwig 2022 JMh [N,3] to sRGB [N,3] under fixed conditions.
    """
    JMh = np.asarray(JMh, dtype=np.float64)
    J = JMh[:, 0]
    M = JMh[:, 1]
    h = JMh[:, 2]
    spec = CAM_Specification_Hellwig2022(J=J, M=M, h=h)
    XYZ = Hellwig2022_to_XYZ(
        spec,
        _XYZ_w_H,
        _L_A_H,
        _Y_b_H,
        surround=_surround_H,
        discount_illuminant=False,
    )
    rgb = xyz_to_srgb_np(XYZ)
    return rgb


# ----------------------------------------------------------------------
# Alignment helpers
# ----------------------------------------------------------------------

def find_reddest_index(rgb_ring_srgb: np.ndarray) -> int:
    rgb = np.asarray(rgb_ring_srgb)
    r = rgb[:, 0]
    g = rgb[:, 1]
    b = rgb[:, 2]
    score = r - 0.5 * (g + b)
    return int(np.argmax(score))


def find_greenest_index(rgb_ring_srgb: np.ndarray) -> int:
    rgb = np.asarray(rgb_ring_srgb)
    r = rgb[:, 0]
    g = rgb[:, 1]
    b = rgb[:, 2]
    score = g - 0.5 * (r + b)
    return int(np.argmax(score))


def align_ring_to_reference(rgb_ring_srgb: np.ndarray,
                            ref_green_idx: int) -> np.ndarray:
    ring = np.asarray(rgb_ring_srgb)
    N = ring.shape[0]

    idx_red = find_reddest_index(ring)
    ring_r0 = np.roll(ring, -idx_red, axis=0)

    g1 = find_greenest_index(ring_r0)
    ring_rev = ring_r0[::-1].copy()
    g2 = find_greenest_index(ring_rev)

    def circ_dist(a, b):
        d = abs(a - b)
        return min(d, N - d)

    if circ_dist(g1, ref_green_idx) <= circ_dist(g2, ref_green_idx):
        return ring_r0
    else:
        return ring_rev


# ----------------------------------------------------------------------
# Iso-Jz ring in JzAzBz (comparison row)
# ----------------------------------------------------------------------

def build_isoJ_from_fourier(jz_fourier: np.ndarray,
                            J0: float = 0.5) -> np.ndarray:
    jz_iso = jz_fourier.copy()
    jz_iso[:, 0] = J0

    jz_t = torch.from_numpy(jz_iso).to(dtype=dtype)
    center = jz_t.mean(dim=0)
    offsets = jz_t - center[None, :]

    def is_safe(scale: float) -> bool:
        pts = center[None, :] + scale * offsets
        xyz = jzazbz_to_xyz_np(pts.cpu().numpy())
        srgb = xyz_to_srgb_np(xyz)
        rgb_lin = srgb_to_linear_np(srgb)
        return np.all((rgb_lin >= -1e-6) & (rgb_lin <= 1.0 + 1e-6))

    s_lo, s_hi = 0.0, 1.0
    if not is_safe(1.0):
        for _ in range(40):
            mid = 0.5 * (s_lo + s_hi)
            if is_safe(float(mid)):
                s_lo = mid
            else:
                s_hi = mid
        s_opt = s_lo
    else:
        s_opt = 1.0

    jz_iso_safe = (center[None, :] + s_opt * offsets).cpu().numpy()
    xyz_iso_safe = jzazbz_to_xyz_np(jz_iso_safe)
    rgb_iso_srgb = np.clip(xyz_to_srgb_np(xyz_iso_safe), 0.0, 1.0)
    return rgb_iso_srgb


# ----------------------------------------------------------------------
# Luminance and PC on brightness channel
# ----------------------------------------------------------------------

def relative_luminance_from_linear(rgb_lin: np.ndarray) -> np.ndarray:
    rgb_lin = np.asarray(rgb_lin)
    r = rgb_lin[..., 0]
    g = rgb_lin[..., 1]
    b = rgb_lin[..., 2]
    return 0.2126 * r + 0.7152 * g + 0.0722 * b


def pc_contrast_B(B_fg: float,
                  B_bg: float,
                  exp: float = 0.4,
                  eps: float = 1e-4) -> float:
    B_fg = float(B_fg)
    B_bg = float(B_bg)
    L_fg = max(B_fg, eps) ** exp
    L_bg = max(B_bg, eps) ** exp
    return abs(L_fg - L_bg) / (L_fg + L_bg)


# ----------------------------------------------------------------------
# Hellwig 2022 JMh balancer with HK-aware salience
# ----------------------------------------------------------------------

def build_contrast_balanced_ring_from_fourier_hellwig_JMh(
    JMh_fourier: np.ndarray,
    M_min: float = 0.04,
    J_center_min: float = 0.45,
    J_center_max: float = 0.75,
    J_center_step: float = 0.02,
    M_scale_min: float = 0.6,
    M_scale_max: float = 1.8,
    M_scale_step: float = 0.05,
    # salience weights
    w_mean: float = 1.0,
    w_varbg: float = 0.7,
    w_varhue: float = 0.3,
    w_bw: float = 1.0,
    # chroma priors
    w_M: float = 0.4,
    w_Mvar: float = 0.3,
    # HK brightness parameters
    k_HK: float = 0.30,
    gamma_HK: float = 0.80,
    exp_pc: float = 0.4,
    # salience backgrounds (avoid true 0 / 1)
    bg_low: float = 0.12,
    bg_high: float = 0.88,
    n_bgs: int = 9,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Build a contrast-balanced Hellwig 2022 JMh ring from a Fourier JMh ring.

    Global warp in Hellwig opponent space:
        J'(θ)  = J_center + (J_base(θ) - J_mean)
        M'(θ)  = M_scale * M_base(θ)
        h'(θ)  = h_base(θ)

    HK-aware brightness model:
        B = Y + k_HK * M^gamma_HK

    Salience is computed on B against gray backgrounds between bg_low
    and bg_high, not pure black/white, to avoid the strong downward
    bias toward very dark colors.
    """
    JMh_fourier = np.asarray(JMh_fourier, dtype=np.float64)
    J_base = JMh_fourier[:, 0]
    M_base = JMh_fourier[:, 1]
    h_base = JMh_fourier[:, 2]

    J_mean = float(J_base.mean())

    Y_bgs = np.linspace(bg_low, bg_high, n_bgs, dtype=np.float64)

    best_score = -1e9
    best_JMh = None
    best_rgb = None

    J_centers = np.arange(J_center_min, J_center_max + 1e-9, J_center_step)
    M_scales = np.arange(M_scale_min, M_scale_max + 1e-9, M_scale_step)

    for J_center in J_centers:
        J_new = J_center + (J_base - J_mean)
        if np.any(J_new < 0.0) or np.any(J_new > 1.0):
            continue

        for M_scale in M_scales:
            M_new = M_scale * M_base
            if np.min(M_new) < M_min:
                continue

            JMh_candidate = np.stack([J_new, M_new, h_base], axis=1)

            srgb = JMh_hellwig_to_srgb_np(JMh_candidate)

            if np.any(srgb < -1e-4) or np.any(srgb > 1.0 + 1e-4):
                continue

            srgb_clamped = np.clip(srgb, 0.0, 1.0)
            rgb_lin = srgb_to_linear_np(srgb_clamped)

            Y_fg = relative_luminance_from_linear(rgb_lin)
            B_fg = Y_fg + k_HK * (M_new ** gamma_HK)

            n_h = JMh_candidate.shape[0]
            n_bg = Y_bgs.size
            pc_mat = np.empty((n_h, n_bg), dtype=np.float64)

            for j, Y_bg in enumerate(Y_bgs):
                B_bg = Y_bg
                for i in range(n_h):
                    pc_mat[i, j] = pc_contrast_B(
                        float(B_fg[i]), float(B_bg), exp=exp_pc
                    )

            mean_sal = float(pc_mat.mean())
            var_bg = float(pc_mat.var(axis=1).mean())
            var_hue = float(pc_mat.var(axis=0).mean())

            pc_low = pc_mat[:, 0]
            pc_high = pc_mat[:, -1]
            bw_diff = float(np.mean(np.abs(pc_low - pc_high)))

            mean_M = float(M_new.mean())
            std_M = float(M_new.std())

            score = (
                w_mean * mean_sal
                - w_varbg * var_bg
                - w_varhue * var_hue
                - w_bw * bw_diff
                + w_M * mean_M
                - w_Mvar * std_M
            )

            if score > best_score:
                best_score = score
                best_JMh = JMh_candidate
                best_rgb = srgb_clamped

    if best_JMh is None:
        best_JMh = JMh_fourier
        best_rgb = np.clip(JMh_hellwig_to_srgb_np(best_JMh), 0.0, 1.0)

    return best_rgb, best_JMh


# ----------------------------------------------------------------------
# Plotly 3D JzAzBz trajectories with sRGB cube
# ----------------------------------------------------------------------

def _srgb_face_to_jz_surface(n: int,
                             fixed_channel: int,
                             fixed_value: float) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    grid = np.linspace(0.0, 1.0, n)
    G, B = np.meshgrid(grid, grid)

    if fixed_channel == 0:
        R = np.full_like(G, fixed_value)
        srgb = np.stack([R, G, B], axis=-1)
    elif fixed_channel == 1:
        Gf = np.full_like(G, fixed_value)
        srgb = np.stack([G, Gf, B], axis=-1)
    else:
        Bf = np.full_like(G, fixed_value)
        srgb = np.stack([G, B, Bf], axis=-1)

    xyz = srgb_to_xyz_np(srgb.reshape(-1, 3))
    jz = xyz_to_jzazbz_np(xyz).reshape(srgb.shape)
    J = jz[..., 0]
    az = jz[..., 1]
    bz = jz[..., 2]
    return az, bz, J


def plot_jz_trajectory_hex_fourier_sine_hsv(
    jz_fourier: np.ndarray, rgb_fourier: np.ndarray,
    jz_arc: np.ndarray,      rgb_arc: np.ndarray,
    jz_sine: np.ndarray,     rgb_sine: np.ndarray,
    jz_hsv: np.ndarray,      rgb_hsv: np.ndarray,
    jz_iso: np.ndarray,      rgb_iso: np.ndarray,
    jz_bal: np.ndarray,      rgb_bal: np.ndarray,
):
    fig = go.Figure()

    n_face = 17
    for ch, val in [(0, 0.0), (0, 1.0), (1, 0.0), (1, 1.0), (2, 0.0), (2, 1.0)]:
        az, bz, J = _srgb_face_to_jz_surface(n_face, ch, val)
        fig.add_trace(
            go.Surface(
                x=az,
                y=bz,
                z=J,
                opacity=0.18,
                showscale=False,
                colorscale="Viridis",
                hoverinfo="skip",
            )
        )

    def add_curve(jz, name, width):
        jz = np.asarray(jz)
        fig.add_trace(
            go.Scatter3d(
                x=jz[:, 1],
                y=jz[:, 2],
                z=jz[:, 0],
                mode="lines",
                name=name,
                line=dict(width=width),
            )
        )

    add_curve(jz_fourier, "Fourier", 7)
    add_curve(jz_arc,      "Arc-length HSV", 5)
    add_curve(jz_iso,      "Iso-Jz", 5)
    add_curve(jz_bal,      "Balanced (Hellwig HK)", 8)
    add_curve(jz_sine,     "sinebow", 4)
    add_curve(jz_hsv,      "HSV", 4)

    fig.update_layout(
        template="plotly_dark",
        paper_bgcolor="black",
        scene=dict(
            xaxis_title="az",
            yaxis_title="bz",
            zaxis_title="Jz",
            bgcolor="black",
            aspectmode="data",
        ),
        margin=dict(l=0, r=0, b=0, t=40),
        title="JzAzBz trajectories with warped sRGB cube and Hellwig-balanced ring",
        showlegend=True,
    )

    html = plot(fig, output_type="div", include_plotlyjs="cdn")
    return HTML(html)


# ----------------------------------------------------------------------
# Main visualization using Hellwig JMh-balanced ring
# ----------------------------------------------------------------------

def visualize_fourier_hex_vs_sinebow_hsv_on_omnipose_hellwig(
    n_hues: int = 72,
    n_harmonics: int = 1,
    device_str: str = "cpu",
):
    dev = torch.device(device_str)

    # base HSV wheel
    t = np.linspace(0, 1, n_hues, endpoint=False)
    hsv_vals = np.stack([t, np.ones_like(t), np.ones_like(t)], axis=1)
    rgb_hsv_raw = hsv_to_rgb(hsv_vals)

    # sinebow
    sr = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0 / 3))
    sg = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1 / 3))
    sb = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2 / 3))
    rgb_sine_raw = np.stack([sr, sg, sb], axis=1)

    # Fourier + arc-length in JzAzBz
    jz_fourier_raw, rgb_fourier_raw, jz_arc_raw, rgb_arc_raw = build_fourier_hsv_hex_ring_jz(
        n_hues=n_hues,
        n_harmonics=n_harmonics,
        oversample=8,
    )

    # align using HSV during reference phase
    idx_red = find_reddest_index(rgb_hsv_raw)
    rgb_hsv_r0 = np.roll(rgb_hsv_raw, -idx_red, axis=0)
    g_h = find_greenest_index(rgb_hsv_r0)
    if g_h > n_hues // 2:
        rgb_hsv_r0 = rgb_hsv_r0[::-1].copy()
        g_h = find_greenest_index(rgb_hsv_r0)
    rgb_hsv = rgb_hsv_r0
    jz_hsv = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_hsv))

    rgb_fourier = align_ring_to_reference(rgb_fourier_raw, g_h)
    rgb_arc = align_ring_to_reference(rgb_arc_raw, g_h)
    rgb_sine = align_ring_to_reference(rgb_sine_raw, g_h)

    jz_fourier = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_fourier))
    jz_arc = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_arc))
    jz_sine = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_sine))

    # Iso-Jz ring
    rgb_isoJ = build_isoJ_from_fourier(jz_fourier)
    jz_isoJ = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_isoJ))

    # Hellwig Fourier JMh
    JMh_fourier = srgb_to_JMh_hellwig_np(rgb_fourier)

    # Hellwig HK-balanced ring
    rgb_bal_raw, JMh_bal_raw = build_contrast_balanced_ring_from_fourier_hellwig_JMh(
        JMh_fourier,
        M_min=0.05,
        J_center_min=0.45,
        J_center_max=0.80,
        J_center_step=0.02,
        M_scale_min=0.6,
        M_scale_max=1.8,
        M_scale_step=0.05,
    )
    rgb_bal = align_ring_to_reference(rgb_bal_raw, g_h)
    jz_bal = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_bal))

    # Plotly 3D
    traj_html = plot_jz_trajectory_hex_fourier_sine_hsv(
        jz_fourier, rgb_fourier,
        jz_arc, rgb_arc,
        jz_sine, rgb_sine,
        jz_hsv, rgb_hsv,
        jz_isoJ, rgb_isoJ,
        jz_bal, rgb_bal,
    )
    display(traj_html)

    # linear RGB for flows
    rgb_fourier_lin = srgb_to_linear_np(rgb_fourier)
    rgb_arc_lin = srgb_to_linear_np(rgb_arc)
    rgb_isoJ_lin = srgb_to_linear_np(rgb_isoJ)
    rgb_bal_lin = srgb_to_linear_np(rgb_bal)
    rgb_sine_lin = srgb_to_linear_np(rgb_sine)
    rgb_hsv_lin = srgb_to_linear_np(rgb_hsv)

    center_srgb = np.array([0.5, 0.5, 0.5], dtype=float)
    center_lin = srgb_to_linear_np(center_srgb)

    omnidir = Path(omnirefactor.__file__).resolve().parent.parent.parent
    basedir = omnidir / "docs" / "_static"
    name = "ecoli"
    ext = ".tif"

    image = io.imread(str(basedir / f"{name}{ext}"))
    masks = io.imread(str(basedir / f"{name}_labels{ext}"))
    slc = ocdkit.measure.crop_bbox(masks > 0, pad=0)[0]
    masks = fastremap.renumber(masks[slc])[0]
    image = image[slc]

    flows_all = omnirefactor.core.masks_to_flows(masks, device="cpu")[4].to(dev)
    flows = flows_all

    def build_images(rgb_lin_ring):
        return make_flow_images_for_ring(rgb_lin_ring, center_lin, flows)

    disk_fourier, fl_fourier, fl_fourier_w, fl_fourier_b = build_images(rgb_fourier_lin)
    disk_arc, fl_arc, fl_arc_w, fl_arc_b = build_images(rgb_arc_lin)
    disk_isoJ, fl_isoJ, fl_isoJ_w, fl_isoJ_b = build_images(rgb_isoJ_lin)
    disk_bal, fl_bal, fl_bal_w, fl_bal_b = build_images(rgb_bal_lin)
    disk_sine, fl_sine, fl_sine_w, fl_sine_b = build_images(rgb_sine_lin)
    disk_hsv, fl_hsv, fl_hsv_w, fl_hsv_b = build_images(rgb_hsv_lin)

    # rotated +90°
    shift = n_hues // 4

    def build_rot(rgb_lin_ring):
        rgb_rot = np.roll(rgb_lin_ring, shift, axis=0)
        _, _, fw, fb = make_flow_images_for_ring(rgb_rot, center_lin, flows)
        return fw, fb

    fl_fourier_w_r, fl_fourier_b_r = build_rot(rgb_fourier_lin)
    fl_arc_w_r, fl_arc_b_r = build_rot(rgb_arc_lin)
    fl_isoJ_w_r, fl_isoJ_b_r = build_rot(rgb_isoJ_lin)
    fl_bal_w_r, fl_bal_b_r = build_rot(rgb_bal_lin)
    fl_sine_w_r, fl_sine_b_r = build_rot(rgb_sine_lin)
    fl_hsv_w_r, fl_hsv_b_r = build_rot(rgb_hsv_lin)

    # grid 6×6
    fig, axes = plt.subplots(6, 6, figsize=(26, 14))
    row_titles = [
        "Fourier",
        "Arc-length HSV",
        "Iso-Jz",
        "Balanced (Hellwig HK)",
        "sinebow",
        "HSV",
    ]

    def strip_axes(ax):
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)

    rows = [
        (disk_fourier, fl_fourier, fl_fourier_w, fl_fourier_b, fl_fourier_w_r, fl_fourier_b_r),
        (disk_arc,     fl_arc,     fl_arc_w,     fl_arc_b,     fl_arc_w_r,     fl_arc_b_r),
        (disk_isoJ,    fl_isoJ,    fl_isoJ_w,    fl_isoJ_b,    fl_isoJ_w_r,    fl_isoJ_b_r),
        (disk_bal,     fl_bal,     fl_bal_w,     fl_bal_b,     fl_bal_w_r,     fl_bal_b_r),
        (disk_sine,    fl_sine,    fl_sine_w,    fl_sine_b,    fl_sine_w_r,    fl_sine_b_r),
        (disk_hsv,     fl_hsv,     fl_hsv_w,     fl_hsv_b,     fl_hsv_w_r,     fl_hsv_b_r),
    ]

    for r in range(6):
        imgs = rows[r]
        for c in range(6):
            axes[r, c].imshow(imgs[c])
            if c == 0:
                strip_axes(axes[r, c])
            else:
                axes[r, c].axis("off")
        axes[r, 0].set_ylabel(row_titles[r], fontsize=12)

    axes[0, 0].set_title("Hue disk (0°)")
    axes[0, 1].set_title("Flow → chroma (0°)")
    axes[0, 2].set_title("Alpha on white (0°)")
    axes[0, 3].set_title("Alpha on black (0°)")
    axes[0, 4].set_title("Alpha on white (+90°)")
    axes[0, 5].set_title("Alpha on black (+90°)")

    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    N_HUES = 72
    N_HARMONICS = 1

    visualize_fourier_hex_vs_sinebow_hsv_on_omnipose_hellwig(
        n_hues=N_HUES,
        n_harmonics=N_HARMONICS,
        device_str="cpu",
    )

In [ ]:
loopp in opponent color theory 

try merely reparametrizing sinebow - nopw that causes hotspot in green 

In [ ]:
import numpy as np
import torch
import os
from pathlib import Path
from skimage import io
import fastremap
import matplotlib.pyplot as plt

import plotly.graph_objects as go
from plotly.offline import plot
from IPython.display import HTML

import omnirefactor

import colour
from colour.appearance import (
    CAM_Specification_Hellwig2022,
    XYZ_to_Hellwig2022,
    Hellwig2022_to_XYZ,
)

dtype = torch.float64

# ----------------------------------------------------------------------
# sRGB <-> XYZ <-> linear  (you already had this)
# ----------------------------------------------------------------------

def srgb_to_linear_np(rgb_srgb: np.ndarray) -> np.ndarray:
    rgb_srgb = np.asarray(rgb_srgb, dtype=np.float64)
    a = 0.055
    rgb_lin = np.where(
        rgb_srgb <= 0.04045,
        rgb_srgb / 12.92,
        ((rgb_srgb + a) / (1 + a)) ** 2.4,
    )
    return rgb_lin


def linear_to_srgb_np(rgb_lin: np.ndarray) -> np.ndarray:
    rgb_lin = np.asarray(rgb_lin, dtype=np.float64)
    a = 0.055
    rgb_srgb = np.where(
        rgb_lin <= 0.0031308,
        12.92 * rgb_lin,
        (1 + a) * np.power(rgb_lin, 1/2.4) - a,
    )
    return rgb_srgb


_M_RGB_TO_XYZ = np.array([
    [0.4124564, 0.3575761, 0.1804375],
    [0.2126729, 0.7151522, 0.0721750],
    [0.0193339, 0.1191920, 0.9503041],
], dtype=np.float64)

_M_XYZ_TO_RGB = np.linalg.inv(_M_RGB_TO_XYZ)


def srgb_to_xyz_np(rgb_srgb: np.ndarray) -> np.ndarray:
    rgb_lin = srgb_to_linear_np(rgb_srgb)
    xyz = rgb_lin @ _M_RGB_TO_XYZ.T
    return xyz


def xyz_to_srgb_np(xyz: np.ndarray) -> np.ndarray:
    rgb_lin = xyz @ _M_XYZ_TO_RGB.T
    rgb_srgb = linear_to_srgb_np(rgb_lin)
    return rgb_srgb


# ----------------------------------------------------------------------
# Real Hellwig 2022 JMh conversions (colour-science)  (you already had)
# ----------------------------------------------------------------------

def _hellwig_viewing_conditions():
    XYZ_w = srgb_to_xyz_np(np.array([[1.0, 1.0, 1.0]]) )[0]  # sRGB D65 white
    Y_b = 20.0
    L_A = 64.0 / np.pi
    surround = colour.appearance.hellwig2022.InductionFactors_Hellwig2022(
        F=1.0, c=0.69, N_c=1.0
    )
    return XYZ_w, Y_b, L_A, surround


_XYZ_w_H, _Y_b_H, _L_A_H, _surround_H = _hellwig_viewing_conditions()


def srgb_to_JMh_hellwig_np(rgb_srgb: np.ndarray) -> np.ndarray:
    rgb_srgb = np.asarray(rgb_srgb, dtype=np.float64)
    xyz = srgb_to_xyz_np(rgb_srgb)
    spec = XYZ_to_Hellwig2022(
        xyz,
        _XYZ_w_H,
        _L_A_H,
        _Y_b_H,
        surround=_surround_H,
        discount_illuminant=False,
    )
    J = np.asarray(spec.J, dtype=np.float64)
    M = np.asarray(spec.M, dtype=np.float64)
    h = np.asarray(spec.h, dtype=np.float64)
    return np.stack([J, M, h], axis=-1)


def JMh_hellwig_to_srgb_np(JMh: np.ndarray) -> np.ndarray:
    JMh = np.asarray(JMh, dtype=np.float64)
    J = JMh[:, 0]
    M = JMh[:, 1]
    h = JMh[:, 2]
    spec = CAM_Specification_Hellwig2022(J=J, M=M, h=h)
    xyz = Hellwig2022_to_XYZ(
        spec,
        _XYZ_w_H,
        _L_A_H,
        _Y_b_H,
        surround=_surround_H,
        discount_illuminant=False,
    )
    rgb = xyz_to_srgb_np(xyz)
    return rgb


def is_in_srgb_gamut_JMh_hellwig(JMh: np.ndarray,
                                 eps: float = 1e-4) -> np.ndarray:
    JMh = np.asarray(JMh, dtype=np.float64)
    try:
        rgb = JMh_hellwig_to_srgb_np(JMh)
    except ValueError:
        return np.zeros(JMh.shape[0], dtype=bool)
    rgb_lin = srgb_to_linear_np(rgb)
    return np.all((rgb_lin >= -eps) & (rgb_lin <= 1.0 + eps), axis=-1)


# ----------------------------------------------------------------------
# Iso-Jz ring (unchanged from your code)
# ----------------------------------------------------------------------

def build_isoJ_from_fourier(jz_fourier: np.ndarray,
                            J0: float = 0.5) -> np.ndarray:
    jz_iso = jz_fourier.copy()
    jz_iso[:, 0] = J0

    jz_t = torch.from_numpy(jz_iso).to(dtype=dtype)
    center = jz_t.mean(dim=0)
    offsets = jz_t - center[None, :]

    def is_safe(scale: float) -> bool:
        pts = center[None, :] + scale * offsets
        xyz = jzazbz_to_xyz_np(pts.cpu().numpy())
        srgb = xyz_to_srgb_np(xyz)
        rgb_lin = srgb_to_linear_np(srgb)
        return np.all((rgb_lin >= -1e-6) & (rgb_lin <= 1.0 + 1e-6))

    s_lo, s_hi = 0.0, 1.0
    if not is_safe(1.0):
        for _ in range(40):
            mid = 0.5*(s_lo + s_hi)
            if is_safe(float(mid)):
                s_lo = mid
            else:
                s_hi = mid
        s_opt = s_lo
    else:
        s_opt = 1.0

    jz_iso_safe = (center[None, :] + s_opt*offsets).cpu().numpy()
    xyz_iso_safe = jzazbz_to_xyz_np(jz_iso_safe)
    rgb_iso_srgb = np.clip(xyz_to_srgb_np(xyz_iso_safe), 0.0, 1.0)
    return rgb_iso_srgb


# ----------------------------------------------------------------------
# NEW: global chroma scaling for a Hellwig JMh ring
# ----------------------------------------------------------------------

def max_global_chroma_scale_for_JMh_ring(
    JMh_ring: np.ndarray,
    s_hi_init: float = 1.5,
    s_cap: float = 4.0,
    n_bisect: int = 24,
    eps: float = 1e-4,
) -> tuple[float, np.ndarray, np.ndarray]:
    """
    Given a Hellwig JMh ring [N,3], find largest s such that M -> s*M
    stays in sRGB for all points.

    Returns:
        s_max       : scalar
        JMh_scaled  : [N,3]
        rgb_scaled  : [N,3]
    """
    JMh_ring = np.asarray(JMh_ring, dtype=np.float64)
    J = JMh_ring[:, 0]
    M = JMh_ring[:, 1]
    h = JMh_ring[:, 2]

    def ok(s: float) -> bool:
        M_s = M * s
        JMh_s = np.stack([J, M_s, h], axis=-1)
        mask = is_in_srgb_gamut_JMh_hellwig(JMh_s, eps=eps)
        return bool(mask.all())

    lo, hi = 0.0, s_hi_init
    if not ok(hi):
        hi = s_hi_init
    else:
        while hi < s_cap:
            s_try = hi * 2.0
            if not ok(s_try):
                break
            hi = s_try

    if not ok(hi):
        for _ in range(n_bisect):
            mid = 0.5*(lo + hi)
            if ok(mid):
                lo = mid
            else:
                hi = mid
        s_max = lo
    else:
        s_max = hi

    M_scaled = M * s_max
    JMh_scaled = np.stack([J, M_scaled, h], axis=-1)
    rgb_scaled = np.clip(JMh_hellwig_to_srgb_np(JMh_scaled), 0.0, 1.0)

    print(f"Hellwig chroma scale s_max={s_max:.3f}")
    return s_max, JMh_scaled, rgb_scaled


# ----------------------------------------------------------------------
# Plotly 3D JzAzBz trajectories with sRGB cube (you already had)
# ----------------------------------------------------------------------

def _srgb_face_to_jz_surface(n: int,
                             fixed_channel: int,
                             fixed_value: float) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    grid = np.linspace(0.0, 1.0, n)
    G, B = np.meshgrid(grid, grid)

    if fixed_channel == 0:
        R = np.full_like(G, fixed_value)
        srgb = np.stack([R, G, B], axis=-1)
    elif fixed_channel == 1:
        Gf = np.full_like(G, fixed_value)
        srgb = np.stack([G, Gf, B], axis=-1)
    else:
        Bf = np.full_like(G, fixed_value)
        srgb = np.stack([G, B, Bf], axis=-1)

    xyz = srgb_to_xyz_np(srgb.reshape(-1, 3))
    jz = xyz_to_jzazbz_np(xyz).reshape(srgb.shape)
    J = jz[..., 0]
    az = jz[..., 1]
    bz = jz[..., 2]
    return az, bz, J


def plot_jz_trajectory_hex_fourier_sine_hsv(
    jz_fourier: np.ndarray, rgb_fourier: np.ndarray,
    jz_arc: np.ndarray,      rgb_arc: np.ndarray,
    jz_sine: np.ndarray,     rgb_sine: np.ndarray,
    jz_hsv: np.ndarray,      rgb_hsv: np.ndarray,
    jz_iso: np.ndarray,      rgb_iso: np.ndarray,
    jz_hellwig: np.ndarray,  rgb_hellwig: np.ndarray,
):
    fig = go.Figure()

    n_face = 17
    for ch, val in [(0, 0.0), (0, 1.0), (1, 0.0), (1, 1.0), (2, 0.0), (2, 1.0)]:
        az, bz, J = _srgb_face_to_jz_surface(n_face, ch, val)
        fig.add_trace(
            go.Surface(
                x=az,
                y=bz,
                z=J,
                opacity=0.18,
                showscale=False,
                colorscale="Viridis",
                hoverinfo="skip",
            )
        )

    def add_curve(jz, name, width):
        jz = np.asarray(jz)
        fig.add_trace(
            go.Scatter3d(
                x=jz[:, 1],
                y=jz[:, 2],
                z=jz[:, 0],
                mode="lines",
                name=name,
                line=dict(width=width),
            )
        )

    add_curve(jz_fourier, "Fourier", 7)
    add_curve(jz_arc,      "Arc-length HSV", 5)
    add_curve(jz_iso,      "Iso-Jz", 5)
    add_curve(jz_hellwig,  "Hellwig J-fixed, free M", 8)
    add_curve(jz_sine,     "sinebow", 4)
    add_curve(jz_hsv,      "HSV", 4)

    fig.update_layout(
        template="plotly_dark",
        paper_bgcolor="black",
        scene=dict(
            xaxis_title="az",
            yaxis_title="bz",
            zaxis_title="Jz",
            bgcolor="black",
            aspectmode="data",
        ),
        margin=dict(l=0, r=0, b=0, t=40),
        title="JzAzBz trajectories with Hellwig J-fixed Fourier-shaped loop",
        showlegend=True,
    )

    html = plot(fig, output_type="div", include_plotlyjs="cdn")
    return HTML(html)


# ======================================================================
# Hellwig JMh <-> Cartesian (J, X, Y) and Fourier fit to HSV boundary
# ======================================================================

def JMh_to_JXY(JMh: np.ndarray) -> np.ndarray:
    """
    Hellwig JMh -> Cartesian (J, X, Y) for Fourier fitting:
        X = M*cos(h), Y = M*sin(h), J unchanged.
    """
    JMh = np.asarray(JMh, dtype=np.float64)
    J = JMh[:, 0]
    M = JMh[:, 1]
    h_rad = np.deg2rad(JMh[:, 2])
    X = M * np.cos(h_rad)
    Y = M * np.sin(h_rad)
    return np.stack([J, X, Y], axis=-1)


def JXY_to_JMh(JXY: np.ndarray) -> np.ndarray:
    """
    Cartesian (J, X, Y) -> Hellwig JMh.
    """
    JXY = np.asarray(JXY, dtype=np.float64)
    J = JXY[:, 0]
    X = JXY[:, 1]
    Y = JXY[:, 2]
    M = np.sqrt(X*X + Y*Y)
    h = np.rad2deg(np.arctan2(Y, X)) % 360.0
    return np.stack([J, M, h], axis=-1)


def build_fourier_hsv_hex_ring_hellwig(
    n_hues: int = 72,
    n_harmonics: int = 1,
    oversample: int = 8,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Hellwig-analogue of build_fourier_hsv_hex_ring_jz:

      1) Start from oversampled HSV hue ring in sRGB.
      2) Convert to Hellwig JMh, then to JXY = (J, X, Y).
      3) Reparameterize curve by arc length s in [0, 2π).
      4) Fit truncated Fourier series to J(s), X(s), Y(s) up to n_harmonics.
      5) Evaluate Fourier fit at n_hues equally spaced s to get smoothed
         Hellwig Fourier ring.
      6) Also downsample the arc-length boundary to n_hues points.

    Returns:
        JMh_fourier : (n_hues, 3) Hellwig JMh Fourier-smoothed ring
        rgb_fourier : (n_hues, 3) sRGB for that ring
        JMh_arc     : (n_hues, 3) arc-length reparam Hellwig boundary
        rgb_arc     : (n_hues, 3) sRGB for arc-length boundary
    """
    # ------------------------------------------------------------------
    # 1) Oversampled HSV ring in sRGB
    # ------------------------------------------------------------------
    N_overs = n_hues * oversample
    t = np.linspace(0.0, 1.0, N_overs, endpoint=False)
    hsv_vals = np.stack([t, np.ones_like(t), np.ones_like(t)], axis=-1)
    rgb_hsv = hsv_to_rgb(hsv_vals)

    # ------------------------------------------------------------------
    # 2) sRGB -> Hellwig JMh -> JXY
    # ------------------------------------------------------------------
    JMh_base = srgb_to_JMh_hellwig_np(rgb_hsv)      # (N_overs, 3)
    JXY_base = JMh_to_JXY(JMh_base)                 # (N_overs, 3)

    # ------------------------------------------------------------------
    # 3) Arc-length parameterization
    # ------------------------------------------------------------------
    diffs = np.diff(JXY_base, axis=0, prepend=JXY_base[0:1])
    seg_len = np.linalg.norm(diffs, axis=1)
    s = np.cumsum(seg_len)
    s -= s[0]
    if s[-1] <= 0:
        s[-1] = 1.0
    s /= s[-1]           # normalize to [0,1]
    s *= 2.0 * np.pi     # parameter in [0, 2π)

    # arc-length reparameterized boundary (downsample to n_hues)
    s_eval = np.linspace(0.0, 2.0*np.pi, n_hues, endpoint=False)
    JXY_arc = np.zeros((n_hues, 3), dtype=np.float64)
    for k in range(3):
        # simple linear interpolation over s for arc boundary
        JXY_arc[:, k] = np.interp(s_eval, s, JXY_base[:, k])
    JMh_arc = JXY_to_JMh(JXY_arc)
    rgb_arc = np.clip(JMh_hellwig_to_srgb_np(JMh_arc), 0.0, 1.0)

    # ------------------------------------------------------------------
    # 4) Fourier fit to J(s), X(s), Y(s)
    # ------------------------------------------------------------------
    H = n_harmonics
    n_coeff = 1 + 2*H  # [a0, a1c, a1s, ..., aHc, aHs]

    # design matrix A(s)
    A = np.zeros((N_overs, n_coeff), dtype=np.float64)
    A[:, 0] = 1.0
    for h_idx in range(1, H+1):
        A[:, 2*h_idx-1] = np.cos(h_idx * s)
        A[:, 2*h_idx]   = np.sin(h_idx * s)

    # least-squares fit for each coordinate
    coeffs = []
    for dim in range(3):
        y = JXY_base[:, dim]
        c, *_ = np.linalg.lstsq(A, y, rcond=None)
        coeffs.append(c)
    coeffs = np.stack(coeffs, axis=0)  # (3, n_coeff)

    # ------------------------------------------------------------------
    # 5) Evaluate Fourier fit at n_hues samples
    # ------------------------------------------------------------------
    A_eval = np.zeros((n_hues, n_coeff), dtype=np.float64)
    A_eval[:, 0] = 1.0
    for h_idx in range(1, H+1):
        A_eval[:, 2*h_idx-1] = np.cos(h_idx * s_eval)
        A_eval[:, 2*h_idx]   = np.sin(h_idx * s_eval)

    JXY_fourier = (coeffs @ A_eval.T).T  # (n_hues, 3)
    JMh_fourier = JXY_to_JMh(JXY_fourier)
    # clamp negative M if any numerical issues
    JMh_fourier[:, 1] = np.clip(JMh_fourier[:, 1], 0.0, None)
    rgb_fourier = np.clip(JMh_hellwig_to_srgb_np(JMh_fourier), 0.0, 1.0)

    return JMh_fourier, rgb_fourier, JMh_arc, rgb_arc

# ======================================================================
# Hellwig JMh <-> Cartesian (J, X, Y) and Fourier fit to HSV boundary
# ======================================================================

def JMh_to_JXY(JMh: np.ndarray) -> np.ndarray:
    """
    Hellwig JMh -> Cartesian (J, X, Y) for Fourier fitting:
        X = M*cos(h), Y = M*sin(h), J unchanged.
    """
    JMh = np.asarray(JMh, dtype=np.float64)
    J = JMh[:, 0]
    M = JMh[:, 1]
    h_rad = np.deg2rad(JMh[:, 2])
    X = M * np.cos(h_rad)
    Y = M * np.sin(h_rad)
    return np.stack([J, X, Y], axis=-1)


def JXY_to_JMh(JXY: np.ndarray) -> np.ndarray:
    """
    Cartesian (J, X, Y) -> Hellwig JMh.
    """
    JXY = np.asarray(JXY, dtype=np.float64)
    J = JXY[:, 0]
    X = JXY[:, 1]
    Y = JXY[:, 2]
    M = np.sqrt(X*X + Y*Y)
    h = np.rad2deg(np.arctan2(Y, X)) % 360.0
    return np.stack([J, M, h], axis=-1)


def _hellwig_ring_safe(JXY: np.ndarray, eps: float = 1e-4) -> bool:
    """
    Check if all points in JXY map to in-gamut sRGB for Hellwig JMh.
    """
    JMh = JXY_to_JMh(JXY)
    mask = is_in_srgb_gamut_JMh_hellwig(JMh, eps=eps)
    return bool(mask.all())


def build_fourier_hsv_hex_ring_hellwig(
    n_hues: int = 72,
    n_harmonics: int = 1,
    oversample: int = 8,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Hellwig-analogue of build_fourier_hsv_hex_ring_jz:

      1) Start from oversampled HSV hue ring in sRGB.
      2) Convert to Hellwig JMh, then to JXY = (J, X, Y).
      3) Reparameterize curve by arc length s in [0, 2π).
      4) Fit truncated Fourier series to J(s), X(s), Y(s) up to n_harmonics.
      5) Evaluate Fourier fit at n_hues equally spaced s to get smoothed
         Hellwig Fourier ring.
      6) Globally scale the Fourier JXY loop about its mean so the ring
         is entirely inside sRGB (no clipping).
      7) Downsample the arc-length boundary to n_hues points and apply
         the same global scale factor for a fair comparison.

    Returns:
        JMh_fourier : (n_hues, 3) Hellwig JMh Fourier-smoothed ring
        rgb_fourier : (n_hues, 3) sRGB for that ring
        JMh_arc     : (n_hues, 3) arc-length reparam Hellwig boundary
        rgb_arc     : (n_hues, 3) sRGB for arc-length boundary
    """
    # 1) Oversampled HSV ring in sRGB
    N_overs = n_hues * oversample
    t = np.linspace(0.0, 1.0, N_overs, endpoint=False)
    hsv_vals = np.stack([t, np.ones_like(t), np.ones_like(t)], axis=-1)
    rgb_hsv = hsv_to_rgb(hsv_vals)  # uses your hsv_to_rgb

    # 2) sRGB -> Hellwig JMh -> JXY
    JMh_base = srgb_to_JMh_hellwig_np(rgb_hsv)      # (N_overs, 3)
    JXY_base = JMh_to_JXY(JMh_base)                 # (N_overs, 3)

    # 3) Arc-length parameterization
    diffs = np.diff(JXY_base, axis=0, prepend=JXY_base[0:1])
    seg_len = np.linalg.norm(diffs, axis=1)
    s = np.cumsum(seg_len)
    s -= s[0]
    if s[-1] <= 0:
        s[-1] = 1.0
    s /= s[-1]
    s *= 2.0 * np.pi  # s in [0, 2π)

    # arc-length boundary at n_hues samples
    s_eval = np.linspace(0.0, 2.0*np.pi, n_hues, endpoint=False)
    JXY_arc = np.zeros((n_hues, 3), dtype=np.float64)
    for k in range(3):
        JXY_arc[:, k] = np.interp(s_eval, s, JXY_base[:, k])

    # 4) Fourier fit to J(s), X(s), Y(s)
    H = n_harmonics
    n_coeff = 1 + 2*H  # [a0, a1c, a1s, ..., aHc, aHs]

    A = np.zeros((N_overs, n_coeff), dtype=np.float64)
    A[:, 0] = 1.0
    for h_idx in range(1, H+1):
        A[:, 2*h_idx-1] = np.cos(h_idx * s)
        A[:, 2*h_idx]   = np.sin(h_idx * s)

    coeffs = []
    for dim in range(3):
        y = JXY_base[:, dim]
        c, *_ = np.linalg.lstsq(A, y, rcond=None)
        coeffs.append(c)
    coeffs = np.stack(coeffs, axis=0)  # (3, n_coeff)

    # evaluate Fourier fit at s_eval
    A_eval = np.zeros((n_hues, n_coeff), dtype=np.float64)
    A_eval[:, 0] = 1.0
    for h_idx in range(1, H+1):
        A_eval[:, 2*h_idx-1] = np.cos(h_idx * s_eval)
        A_eval[:, 2*h_idx]   = np.sin(h_idx * s_eval)

    JXY_fourier = (coeffs @ A_eval.T).T  # (n_hues, 3)

    # 5) global scaling of Fourier JXY about its mean to stay in gamut
    center = JXY_fourier.mean(axis=0)
    offsets = JXY_fourier - center[None, :]

    def scaled_ring_ok(scale: float) -> bool:
        JXY_scaled = center[None, :] + scale * offsets
        return _hellwig_ring_safe(JXY_scaled)

    s_lo, s_hi = 0.0, 1.0
    if not scaled_ring_ok(1.0):
        for _ in range(40):
            mid = 0.5*(s_lo + s_hi)
            if scaled_ring_ok(mid):
                s_lo = mid
            else:
                s_hi = mid
        s_opt = s_lo
    else:
        s_opt = 1.0

    JXY_fourier_scaled = center[None, :] + s_opt * offsets
    JMh_fourier = JXY_to_JMh(JXY_fourier_scaled)
    JMh_fourier[:, 1] = np.clip(JMh_fourier[:, 1], 0.0, None)  # safety
    rgb_fourier = np.clip(JMh_hellwig_to_srgb_np(JMh_fourier), 0.0, 1.0)

    # 6) apply same global scale to arc boundary for fairness
    center_arc = JXY_arc.mean(axis=0)
    offsets_arc = JXY_arc - center_arc[None, :]
    JXY_arc_scaled = center_arc[None, :] + s_opt * offsets_arc
    JMh_arc = JXY_to_JMh(JXY_arc_scaled)
    JMh_arc[:, 1] = np.clip(JMh_arc[:, 1], 0.0, None)
    rgb_arc = np.clip(JMh_hellwig_to_srgb_np(JMh_arc), 0.0, 1.0)

    return JMh_fourier, rgb_fourier, JMh_arc, rgb_arc

def visualize_fourier_hex_vs_sinebow_hsv_on_omnipose_hellwig_fourierJMh(
    n_hues: int = 72,
    n_harmonics: int = 1,
    device_str: str = "cpu",
):
    dev = torch.device(device_str)

    # 1) base HSV wheel
    t = np.linspace(0, 1, n_hues, endpoint=False)
    hsv_vals = np.stack([t, np.ones_like(t), np.ones_like(t)], axis=1)
    rgb_hsv_raw = hsv_to_rgb(hsv_vals)

    # 2) sinebow
    sr = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0 / 3))
    sg = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1 / 3))
    sb = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2 / 3))
    rgb_sine_raw = np.stack([sr, sg, sb], axis=1)

    # 3) Fourier + arc-length in JzAzBz (existing)
    jz_fourier_raw, rgb_fourier_raw, jz_arc_raw, rgb_arc_raw = build_fourier_hsv_hex_ring_jz(
        n_hues=n_hues,
        n_harmonics=n_harmonics,
        oversample=8,
    )

    # align based on HSV
    idx_red = find_reddest_index(rgb_hsv_raw)
    rgb_hsv_r0 = np.roll(rgb_hsv_raw, -idx_red, axis=0)
    g_h = find_greenest_index(rgb_hsv_r0)
    if g_h > n_hues // 2:
        rgb_hsv_r0 = rgb_hsv_r0[::-1].copy()
        g_h = find_greenest_index(rgb_hsv_r0)
    rgb_hsv = rgb_hsv_r0
    jz_hsv = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_hsv))

    rgb_fourier = align_ring_to_reference(rgb_fourier_raw, g_h)
    rgb_arc = align_ring_to_reference(rgb_arc_raw, g_h)
    rgb_sine = align_ring_to_reference(rgb_sine_raw, g_h)

    jz_fourier = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_fourier))
    jz_arc = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_arc))
    jz_sine = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_sine))

    # 4) Iso-Jz ring in JzAzBz
    rgb_isoJ_raw = build_isoJ_from_fourier(jz_fourier)
    rgb_isoJ = align_ring_to_reference(rgb_isoJ_raw, g_h)
    jz_isoJ = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_isoJ))

    # 5) Hellwig Fourier ring in JMh
    JMh_fourier_H, rgb_fourier_H_raw, JMh_arc_H, rgb_arc_H_raw = build_fourier_hsv_hex_ring_hellwig(
        n_hues=n_hues,
        n_harmonics=n_harmonics,
        oversample=8,
    )
    rgb_fourier_H = align_ring_to_reference(rgb_fourier_H_raw, g_h)
    jz_fourier_H = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_fourier_H))

    # Hellwig diagnostics
    J_H = JMh_fourier_H[:, 0]
    M_H = JMh_fourier_H[:, 1]
    h_H = JMh_fourier_H[:, 2]
    rgb_H = np.clip(JMh_hellwig_to_srgb_np(JMh_fourier_H), 0.0, 1.0)

    fig_h, ax_h = plt.subplots(1, 3, figsize=(16,4))
    ax_h[0].scatter(h_H, J_H, c=rgb_H, s=8)
    ax_h[0].set_xlabel("h (deg)")
    ax_h[0].set_ylabel("J (Hellwig)")
    ax_h[0].set_title("Hellwig Fourier: J vs h")

    ax_h[1].scatter(h_H, M_H, c=rgb_H, s=8)
    ax_h[1].set_xlabel("h (deg)")
    ax_h[1].set_ylabel("M (Hellwig)")
    ax_h[1].set_title("Hellwig Fourier: M vs h")

    strip = rgb_H[np.newaxis, ...]
    ax_h[2].imshow(strip, aspect="auto")
    ax_h[2].set_axis_off()
    ax_h[2].set_title("Hellwig Fourier sRGB strip")
    plt.suptitle("Hellwig JMh Fourier-smooth HSV boundary")
    plt.tight_layout()
    plt.show()

    # 6) 3D JzAzBz plot with Hellwig Fourier ring
    traj_html = plot_jz_trajectory_hex_fourier_sine_hsv(
        jz_fourier, rgb_fourier,
        jz_arc, rgb_arc,
        jz_sine, rgb_sine,
        jz_hsv, rgb_hsv,
        jz_isoJ, rgb_isoJ,
        jz_fourier_H, rgb_fourier_H,
    )
    display(traj_html)

    # 7) Omnipose flows (6×6 grid)
    rgb_fourier_lin = srgb_to_linear_np(rgb_fourier)
    rgb_arc_lin = srgb_to_linear_np(rgb_arc)
    rgb_isoJ_lin = srgb_to_linear_np(rgb_isoJ)
    rgb_fourier_H_lin = srgb_to_linear_np(rgb_fourier_H)
    rgb_sine_lin = srgb_to_linear_np(rgb_sine)
    rgb_hsv_lin = srgb_to_linear_np(rgb_hsv)

    center_srgb = np.array([0.5, 0.5, 0.5], dtype=float)
    center_lin = srgb_to_linear_np(center_srgb)

    omnidir = Path(omnirefactor.__file__).resolve().parent.parent.parent
    basedir = omnidir / "docs" / "_static"
    name = "ecoli"
    ext = ".tif"

    image = io.imread(str(basedir / f"{name}{ext}"))
    masks = io.imread(str(basedir / f"{name}_labels{ext}"))
    slc = ocdkit.measure.crop_bbox(masks > 0, pad=0)[0]
    masks = fastremap.renumber(masks[slc])[0]
    image = image[slc]

    flows_all = omnirefactor.core.masks_to_flows(masks, device="cpu")[4].to(dev)
    flows = flows_all

    def build_images(rgb_lin_ring):
        return make_flow_images_for_ring(rgb_lin_ring, center_lin, flows)

    disk_fourier,    fl_fourier,    fl_fourier_w,    fl_fourier_b = build_images(rgb_fourier_lin)
    disk_arc,        fl_arc,        fl_arc_w,        fl_arc_b     = build_images(rgb_arc_lin)
    disk_isoJ,       fl_isoJ,       fl_isoJ_w,       fl_isoJ_b    = build_images(rgb_isoJ_lin)
    disk_fourier_H,  fl_fourier_H,  fl_fourier_H_w,  fl_fourier_H_b = build_images(rgb_fourier_H_lin)
    disk_sine,       fl_sine,       fl_sine_w,       fl_sine_b    = build_images(rgb_sine_lin)
    disk_hsv,        fl_hsv,        fl_hsv_w,        fl_hsv_b     = build_images(rgb_hsv_lin)

    shift = n_hues // 4

    def build_rot(rgb_lin_ring):
        rgb_rot = np.roll(rgb_lin_ring, shift, axis=0)
        _, _, fw, fb = make_flow_images_for_ring(rgb_rot, center_lin, flows)
        return fw, fb

    fl_fourier_w_r,    fl_fourier_b_r    = build_rot(rgb_fourier_lin)
    fl_arc_w_r,        fl_arc_b_r        = build_rot(rgb_arc_lin)
    fl_isoJ_w_r,       fl_isoJ_b_r       = build_rot(rgb_isoJ_lin)
    fl_fourier_H_w_r,  fl_fourier_H_b_r  = build_rot(rgb_fourier_H_lin)
    fl_sine_w_r,       fl_sine_b_r       = build_rot(rgb_sine_lin)
    fl_hsv_w_r,        fl_hsv_b_r        = build_rot(rgb_hsv_lin)

    fig, axes = plt.subplots(6, 6, figsize=(26, 14))
    row_titles = [
        "Fourier (JzAzBz)",
        "Arc-length HSV (JzAzBz)",
        "Iso-Jz",
        "Fourier (Hellwig JMh)",
        "sinebow",
        "HSV",
    ]

    def strip_axes(ax):
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)

    rows = [
        (disk_fourier,    fl_fourier,    fl_fourier_w,    fl_fourier_b,    fl_fourier_w_r,    fl_fourier_b_r),
        (disk_arc,        fl_arc,        fl_arc_w,        fl_arc_b,        fl_arc_w_r,        fl_arc_b_r),
        (disk_isoJ,       fl_isoJ,       fl_isoJ_w,       fl_isoJ_b,       fl_isoJ_w_r,       fl_isoJ_b_r),
        (disk_fourier_H,  fl_fourier_H,  fl_fourier_H_w,  fl_fourier_H_b,  fl_fourier_H_w_r,  fl_fourier_H_b_r),
        (disk_sine,       fl_sine,       fl_sine_w,       fl_sine_b,       fl_sine_w_r,       fl_sine_b_r),
        (disk_hsv,        fl_hsv,        fl_hsv_w,        fl_hsv_b,        fl_hsv_w_r,        fl_hsv_b_r),
    ]

    for r in range(6):
        imgs = rows[r]
        for c in range(6):
            axes[r, c].imshow(imgs[c])
            if c == 0:
                strip_axes(axes[r, c])
            else:
                axes[r, c].axis("off")
        axes[r, 0].set_ylabel(row_titles[r], fontsize=12)

    axes[0, 0].set_title("Hue disk (0°)")
    axes[0, 1].set_title("Flow → chroma (0°)")
    axes[0, 2].set_title("Alpha on white (0°)")
    axes[0, 3].set_title("Alpha on black (0°)")
    axes[0, 4].set_title("Alpha on white (+90°)")
    axes[0, 5].set_title("Alpha on black (+90°)")

    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    N_HUES = 72
    N_HARMONICS = 1  # with 1 harmonic, the Hellwig ring should be ellipse-like

    visualize_fourier_hex_vs_sinebow_hsv_on_omnipose_hellwig_fourierJMh(
        n_hues=N_HUES,
        n_harmonics=N_HARMONICS,
        device_str="cpu",
    )

In [ ]:
import numpy as np
import plotly.graph_objects as go

# ----------------------------------------------------------------------
# Jzazbz implementation (Safdar et al. 2017) from sRGB
# ----------------------------------------------------------------------

# Jzazbz constants
b = 1.15
g = 0.66
c1 = 3424 / 2**12
c2 = 2413 / 2**7
c3 = 2392 / 2**7
n_ = 2610 / 2**14
p_ = 1.7 * 2523 / 2**5
d = -0.56
d0 = 1.6295499532821566e-11

M1 = np.array([
    [ 0.41478972,  0.579999,    0.0146480],
    [-0.2015100,   1.120649,    0.0531008],
    [-0.0166008,   0.264800,    0.6684799],
])

M2 = np.array([
    [ 0.5,        0.5,        0.0       ],
    [ 3.524000,  -4.066708,   0.542708  ],
    [ 0.199076,   1.096799,  -1.295875  ],
])

# sRGB to XYZ (D65, relative)
M_srgb_to_xyz = np.array([
    [0.4124564, 0.3575761, 0.1804375],
    [0.2126729, 0.7151522, 0.0721750],
    [0.0193339, 0.1191920, 0.9503041],
])

def srgb_to_linear(srgb):
    srgb = np.asarray(srgb, dtype=np.float64)
    a = 0.055
    threshold = 0.04045
    linear = np.where(
        srgb <= threshold,
        srgb / 12.92,
        ((srgb + a) / (1.0 + a)) ** 2.4,
    )
    return linear

def linear_rgb_to_xyz(rgb):
    rgb = np.asarray(rgb, dtype=np.float64)
    shape = rgb.shape
    rgb_flat = rgb.reshape(-1, 3)
    xyz_flat = rgb_flat @ M_srgb_to_xyz.T
    return xyz_flat.reshape(shape)

def xyz_to_jzazbz(xyz):
    xyz = np.asarray(xyz, dtype=np.float64)
    shape = xyz.shape
    xyz_flat = xyz.reshape(-1, 3)

    # Apply scaling for Jzazbz specification
    X = xyz_flat[:, 0]
    Y = xyz_flat[:, 1]
    Z = xyz_flat[:, 2]

    Xp = b * X - (b - 1.0) * Z
    Yp = g * Y - (g - 1.0) * X
    Zp = Z

    XYZp = np.stack([Xp, Yp, Zp], axis=-1)

    # Nonlinear response
    num = c1 + c2 * XYZp
    den = 1.0 + c3 * XYZp
    LMS_prime = (num / den) ** n_

    Izazbz = LMS_prime @ M2.T
    Iz = Izazbz[:, 0]
    az = Izazbz[:, 1]
    bz = Izazbz[:, 2]

    Jz = ((1.0 + d) * Iz) / (1.0 + d * Iz) - d0

    jzazbz_flat = np.stack([Jz, az, bz], axis=-1)
    return jzazbz_flat.reshape(shape)

def srgb_to_jzazbz(srgb):
    rgb_lin = srgb_to_linear(srgb)
    xyz = linear_rgb_to_xyz(rgb_lin)
    return xyz_to_jzazbz(xyz)

# ----------------------------------------------------------------------
# Gamut sampling and approximate inscribed ellipsoid
# ----------------------------------------------------------------------

def sample_srgb_grid(n_per_axis=20):
    grid = np.linspace(0.0, 1.0, n_per_axis)
    r, g_, b_ = np.meshgrid(grid, grid, grid, indexing="ij")
    srgb = np.stack([r, g_, b_], axis=-1)
    return srgb.reshape(-1, 3)

def fit_principal_ellipsoid(points, scale=0.9):
    """
    Approximate large ellipsoid inside the point cloud:
    center at mean, axes along principal components,
    axis lengths limited by extents along those axes and scaled down.
    """
    points = np.asarray(points, dtype=np.float64)
    mean = points.mean(axis=0)

    # Covariance and eigen-decomposition
    cov = np.cov(points.T)
    w, v = np.linalg.eigh(cov)
    order = np.argsort(w)[::-1]
    w = w[order]
    v = v[:, order]

    # Project points onto principal axes
    proj = (points - mean) @ v

    max_pos = proj.max(axis=0)
    max_neg = proj.min(axis=0)
    extents = np.minimum(max_pos, -max_neg)

    axes = scale * extents
    return mean, v, axes

def ellipsoid_mesh(mean, vecs, axes, nu=60, nv=30):
    u = np.linspace(0.0, 2.0 * np.pi, nu)
    v = np.linspace(0.0, np.pi, nv)
    uu, vv = np.meshgrid(u, v)

    x = axes[0] * np.sin(vv) * np.cos(uu)
    y = axes[1] * np.sin(vv) * np.sin(uu)
    z = axes[2] * np.cos(vv)

    ellipsoid_local = np.stack([x, y, z], axis=-1)
    ellipsoid_global = ellipsoid_local @ vecs.T + mean

    X = ellipsoid_global[..., 0]
    Y = ellipsoid_global[..., 1]
    Z = ellipsoid_global[..., 2]
    return X, Y, Z

def build_jzazbz_gamut_figure(n_per_axis=18, scale=0.9):
    srgb = sample_srgb_grid(n_per_axis=n_per_axis)
    jzazbz = srgb_to_jzazbz(srgb)

    # Fit approximate inscribed ellipsoid
    mean, vecs, axes = fit_principal_ellipsoid(jzazbz, scale=scale)
    Ex, Ey, Ez = ellipsoid_mesh(mean, vecs, axes)

    # Colors for scatter (convert sRGB to 0-255)
    rgb255 = np.clip((srgb * 255.0).round().astype(np.int16), 0, 255)
    colors = [
        f"rgb({r},{g},{b})"
        for r, g, b in rgb255
    ]

    scatter = go.Scatter3d(
        x=jzazbz[:, 0],
        y=jzazbz[:, 1],
        z=jzazbz[:, 2],
        mode="markers",
        marker=dict(
            size=2,
            color=colors,
            opacity=0.5,
        ),
        name="sRGB samples in Jzazbz",
    )

    ellipsoid_surface = go.Surface(
        x=Ex,
        y=Ey,
        z=Ez,
        opacity=0.5,
        showscale=False,
        name="Approx. inscribed ellipsoid",
    )

    fig = go.Figure(data=[scatter, ellipsoid_surface])
    fig.update_layout(
        title="sRGB Gamut in Jzazbz with Approximate Inscribed Ellipsoid",
        scene=dict(
            xaxis_title="Jz",
            yaxis_title="az",
            zaxis_title="bz",
            aspectmode="data",
        ),
        margin=dict(l=0, r=0, b=0, t=40),
    )
    return fig

if __name__ == "__main__":
    fig = build_jzazbz_gamut_figure(n_per_axis=18, scale=0.9)
    fig.show()